# Figure 2 — clean

Single filter throughout: the cells your collaborator (Stan) kept, via his barcode list. His lab's library-size gating, applied once, everywhere. No confidence cutoffs, no depth corrections, no per-panel exclusions.

Controls only. Embryo panels use Tyser's own annotations (no label transfer).

Decisions applied: eNasM+lNasM merged as NasM (his discriminators TBX6/MESP2 don't separate them in our 42-72h data); aPM-LC/LPM-LC/CS-LC/stressed grouped as "other" (predominantly 96h states, beyond our window); pPS-LC kept but flagged.

In [ ]:
import sys
from pathlib import Path as _P
sys.path.insert(0, str(_P.cwd().parent / "utils"))
from palettes import (TISSUE_COLORS, EMBRYO_COLORS, PROGRAM_COLORS,
                      TISSUE_MAP_MERGED as TISSUE_MAP,
                      TISSUE_ORDER_MERGED as TISSUE_ORDER,
                      SITE_COLORS, CONDITION_COLORS, TIMEPOINT_COLORS,
                      RED, BLUE, GREY_FAINT, check)
import matplotlib as mpl
mpl.rcParams["axes.grid"] = False      # no gridlines on any panel
print("palettes:", check(TISSUE_ORDER)["order"])

# Colours removed 2026-08-08 -- now from palettes.py (imported below).
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import scipy.sparse as sp
import anndata as ad
import scanpy as sc

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=120, facecolor="white", frameon=False)
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42

import sys
sys.path.insert(0, str(Path.cwd().parent / "utils"))   # run notebooks from their own folder
from config import ROOT, PROJECT_DIR, CELLRANGER_DIR

MASTER   = ROOT / "00_DATA_FROZEN" / "03_analysis_ready" / "adata_master_normalized_magic.h5ad"
TYSER    = ROOT / "00_DATA_FROZEN" / "03_analysis_ready" / "tyser_embryo_genes_for_fig2_magic.h5ad"
STAN_CSV = ROOT / "00_DATA_FROZEN" / "01_source" / "sample_barcodes.csv"
FIG = ROOT / "07_figures" / "FIGURE2_clean"; FIG.mkdir(parents=True, exist_ok=True)
TAB = ROOT / "06_tables"  / "FIGURE2_clean"; TAB.mkdir(parents=True, exist_ok=True)

GENES = ["APLNR", "APELA"]
TIMES = ["42h", "72h"]
SAVE_FORMATS = ["svg", "pdf"]
SHOW_TITLES  = False
LEGEND_FS    = 20
EMBRYO_FS    = 13

#other meso color
# ---- palette: warm = mesoderm/blood (matches rest of paper), cool = the rest ----
RED, BLUE   = "#991E03", "#6B8E9E"
GREY_ZERO   = "#D6D4D0"   # zero expression on a UMAP
GREY_CTX    = "#F5F3F0"   # background context cells
GREY_TEXT   = "#52514E"
DOT_CMAP    = LinearSegmentedColormap.from_list("dot", [GREY_FAINT, RED])
EXPR_CMAP   = LinearSegmentedColormap.from_list("expr", [GREY_ZERO, RED])

TYSER_FULL = ROOT / "01_references/embryo_tyser/embryo_full_annotated.h5ad"
tyser_full = ad.read_h5ad(TYSER_FULL)
print(tyser_full.shape)  # expect ~1195 × 57490

# confirm genes present
tsym2key_full = dict(zip(tyser_full.var["symbol"].astype(str), tyser_full.var.index)) \
                if "symbol" in tyser_full.var.columns else {g: g for g in tyser_full.var_names}
for g in ["APELA", "APLN", "APLNR"]:
    print(f"{g}: {'FOUND' if g in tsym2key_full else 'NOT FOUND'}")

# transfer cell type annotation from tyser (same cells, same barcodes)
tyser_full.obs["tissue_embryo"] = tyser.obs["tissue_embryo"].reindex(tyser_full.obs_names)

def side_legend(ax, labels, colors, fontsize=LEGEND_FS, marker=7):
    h = [plt.Line2D([0],[0], marker="o", ls="", markersize=marker*2.5,
                    markerfacecolor=colors[l], markeredgecolor="none", label=l)
         for l in labels]
    return ax.legend(handles=h, loc="center left", bbox_to_anchor=(1.02, 0.5),
                     frameon=False, fontsize=fontsize, labelspacing=0.7,
                     handletextpad=0.6, borderaxespad=0)

def thin_size_legend(dp, keep=3):
    try:
        sl = dp.get_axes().get("size_legend_ax")
        if sl is None or not sl.collections: return
        coll = sl.collections[0]
        sizes = np.asarray(coll.get_sizes())
        labels = [t.get_text() for t in sl.get_xticklabels()]
        colors = coll.get_facecolors()
        n = len(sizes)
        if n <= keep:
            sl.grid(False); return
        idx = np.linspace(0, n-1, keep).round().astype(int)
        color = colors[0] if len(colors) else "gray"
        sl.clear()
        sl.scatter(np.arange(len(idx)), np.zeros(len(idx)), s=sizes[idx],
                   color=color, edgecolor="black", linewidth=0.25, zorder=100)
        sl.set_xticks(np.arange(len(idx)))
        if len(labels) == n:
            sl.set_xticklabels([labels[i] for i in idx], fontsize="small")
        sl.set_yticks([]); sl.grid(False)
        sl.tick_params(axis="both", left=False, bottom=False, labelleft=False)
        for s in sl.spines.values(): s.set_visible(False)
        sl.set_xlim(-0.7, len(idx)-0.3); sl.set_ylim(-0.5, 0.5)
    except Exception as e:
        print(f"  (size legend skip: {e})")

for n, p in [("master", MASTER), ("tyser", TYSER), ("stan csv", STAN_CSV)]:
    print(f"{n:10s} {'OK ' if p.exists() else 'MISSING'} {p.name}")



## Load, label, and apply Stan's filter

In [ ]:
# TISSUE_MAP / TISSUE_ORDER removed 2026-08-08 -- imported from palettes.py
# (merged variant: one NasM category, as Figure 2 intends).
adata = ad.read_h5ad(MASTER)
tyser = ad.read_h5ad(TYSER)

adata.obs["tissue"] = pd.Categorical(
    adata.obs["predicted_leiden_sub_joint"].astype(str).map(TISSUE_MAP),
    categories=TISSUE_ORDER, ordered=True)

# Stan's barcodes = the single filter
def bc16(x):
    h = re.findall(r"[ACGT]{14,}", str(x)); return h[0] if h else None
stan = pd.read_csv(STAN_CSV, index_col=0)
STAN_BC = set(bc16(x) for x in stan.index)
adata.obs["_bc"] = [bc16(x) for x in adata.obs_names]

ctrl = adata[(adata.obs["condition"].astype(str) == "Ctrl") &
             (adata.obs["_bc"].isin(STAN_BC))].copy()
ctrl.obs["tissue"] = ctrl.obs["tissue"].cat.remove_unused_categories()
print(f"control cells after Stan filter: {ctrl.n_obs:,}")
print(ctrl.obs.groupby(["site","timepoint_true"], observed=True).size().to_string())

def sym_index(a, g):
    hit = a.var.index[a.var["symbol"]==g] if "symbol" in a.var else a.var.index[a.var_names==g]
    if len(hit)==0: raise KeyError(g)
    return hit[0]
GK  = {g: sym_index(adata, g) for g in GENES}
TGK = {g: sym_index(tyser, g) for g in GENES}


In [ ]:
print(f"ctrl: {ctrl.n_obs:,} cells")
print(ctrl.obs['condition'].value_counts())
print(ctrl.obs['timepoint_true'].value_counts())

In [ ]:
# ============================================================================
# ALL-TISSUE CTRL WGCNA — run after your existing data-loading + Stan filter
# Requires: ctrl, TAB already defined
# ============================================================================

# Run once if needed:
# %pip install PyWGCNA

import numpy as np
import pandas as pd
import scipy.sparse as sp
import anndata as ad
import scanpy as sc
from sklearn.cluster import KMeans
import PyWGCNA

# ----------------------------- SETTINGS -------------------------------------

STATE_COL = "predicted_leiden_sub_joint"
KEEP_TIMES = ["42h", "72h"]

CELLS_PER_META = 25       # target ~25 cells / pseudocell
MIN_META_CELLS = 10       # discard tiny pseudocells
MIN_GENE_CELLS = 20       # gene must occur in >=20 Ctrl cells
N_WGCNA_GENES = 3000      # keeps TOM manageable
RANDOM_STATE = 2026

#OUT = TAB / "WGCNA_allCtrl_allTissues"
OUT = TAB / "WGCNA_Ctrl_APELA_epiblastside"

OUT.mkdir(parents=True, exist_ok=True)

# -------------------------- 1. ALL CTRL CELLS -------------------------------

# net = ctrl[
#     ctrl.obs["timepoint_true"].astype(str).isin(KEEP_TIMES)
# ].copy()

# print(f"Cells: {net.n_obs:,}")
# print("\nCell states:")
# print(net.obs[STATE_COL].astype(str).value_counts())

# assert "counts" in net.layers, (
#     "Need net.layers['counts']; do NOT use MAGIC for WGCNA."
# )

# Xc = net.layers["counts"]
# Xc = sp.csr_matrix(Xc) if not sp.issparse(Xc) else Xc.tocsr()

# -------------------------- 1. MESODERM CTRL CELLS --------------------------

# MESO = [
#     "pPS-LC",
#     "aPS-LC",
#     "eNasM",
#     "lNasM",
#     "aPM-LC",
#     "LPM-LC",
# ]

# net = ctrl[
#     ctrl.obs["timepoint_true"].astype(str).isin(KEEP_TIMES)
#     &
#     ctrl.obs[STATE_COL].astype(str).isin(MESO)
# ].copy()

# print(f"Cells: {net.n_obs:,}")
# print("\nCell states:")
# print(net.obs[STATE_COL].astype(str).value_counts())

# assert "counts" in net.layers, (
#     "Need net.layers['counts']; do NOT use MAGIC for WGCNA."
# )

# Xc = net.layers["counts"]
# Xc = sp.csr_matrix(Xc) if not sp.issparse(Xc) else Xc.tocsr()

# -------------------------- 1. APELA CTRL CELLS -----------------------------

APELA_STATES = [
    "pluripotent",
    "AE-LC",
    "PGC-LC",
]

net = ctrl[
    ctrl.obs["timepoint_true"].astype(str).isin(KEEP_TIMES)
    &
    ctrl.obs[STATE_COL].astype(str).isin(APELA_STATES)
].copy()

print(f"Cells: {net.n_obs:,}")

print("\nCells by group:")
print(
    net.obs.groupby(
        ["site", "timepoint_true", STATE_COL],
        observed=True
    ).size()
)

# IMPORTANT: rebuild raw-count matrix AFTER subsetting net
assert "counts" in net.layers, (
    "Need net.layers['counts']; do NOT use MAGIC for WGCNA."
)

Xc = net.layers["counts"]
Xc = sp.csr_matrix(Xc) if not sp.issparse(Xc) else Xc.tocsr()

print("\nnet shape:", net.shape)
print("Xc shape:", Xc.shape)
# -------------------------- 2. RAW-COUNT PCA --------------------------------
# PCA is ONLY used to group similar cells into pseudocells.
# WGCNA itself will use aggregated raw counts.

tmp = ad.AnnData(
    X=Xc.copy(),
    obs=net.obs.copy(),
    var=net.var.copy()
)

sc.pp.filter_genes(tmp, min_cells=MIN_GENE_CELLS)
sc.pp.normalize_total(tmp, target_sum=1e4)
sc.pp.log1p(tmp)

sc.pp.highly_variable_genes(
    tmp,
    n_top_genes=min(2000, tmp.n_vars),
    flavor="seurat"
)

sc.tl.pca(
    tmp,
    n_comps=30,
    use_highly_variable=True
)

Z = tmp.obsm["X_pca"]

# ----------------------- 3. MAKE PSEUDOCELLS --------------------------------
# Never mix site/time/state inside one pseudocell.

groups = ["site", "timepoint_true", STATE_COL]

meta_members = []
meta_rows = []

for key, pos in net.obs.groupby(groups, observed=True).indices.items():

    pos = np.asarray(pos)
    n = len(pos)

    if n < MIN_META_CELLS:
        continue

    n_meta = max(1, round(n / CELLS_PER_META))
    n_meta = min(n_meta, max(1, n // MIN_META_CELLS))

    if n_meta == 1:
        labels = np.zeros(n, dtype=int)
    else:
        labels = KMeans(
            n_clusters=n_meta,
            random_state=RANDOM_STATE,
            n_init=20
        ).fit_predict(Z[pos])

    for lab in np.unique(labels):

        members = pos[labels == lab]

        if len(members) < MIN_META_CELLS:
            continue

        site, tp, state = key

        meta_members.append(members)

        meta_rows.append({
            "site": str(site),
            "timepoint": str(tp),
            "state": str(state),
            "n_cells": len(members)
        })

meta_obs = pd.DataFrame(meta_rows)
meta_obs.index = [f"MC_{i:04d}" for i in range(len(meta_obs))]

print(f"\nPseudocells: {len(meta_obs)}")
print(meta_obs["n_cells"].describe())

print("\nPseudocells by group:")
print(
    meta_obs.groupby(
        ["site", "timepoint", "state"],
        observed=True
    ).size()
)

# -------------------------- 4. GENE NAMES -----------------------------------

if "symbol" in net.var.columns:
    symbols = net.var["symbol"].astype(str).values
else:
    symbols = net.var_names.astype(str).values

det_n = np.asarray((Xc > 0).sum(axis=0)).ravel()
total = np.asarray(Xc.sum(axis=0)).ravel()

gene_info = pd.DataFrame({
    "idx": np.arange(net.n_vars),
    "gene": symbols,
    "det_n": det_n,
    "total": total
})

# remove junk / technical genes
valid = (
    gene_info["gene"].notna()
    & ~gene_info["gene"].isin(["", "nan", "None"])
    & ~gene_info["gene"].str.startswith("ENSG")
    & ~gene_info["gene"].str.match(r"^(MT-|RPL|RPS)", case=False, na=False)
)

# keep reasonably detected genes; force seeds in
valid &= (
    (gene_info["det_n"] >= MIN_GENE_CELLS)
    | gene_info["gene"].isin(["APELA"])
)

gene_info = (
    gene_info.loc[valid]
    .sort_values("total", ascending=False)
    .drop_duplicates("gene")
)

gene_idx = gene_info["idx"].to_numpy()
gene_names = gene_info["gene"].to_numpy()

print(f"\nCandidate genes: {len(gene_names):,}")

# -------------------------- 5. AGGREGATE COUNTS -----------------------------

Xm = sp.vstack(
    [
        sp.csr_matrix(Xc[members][:, gene_idx].sum(axis=0))
        for members in meta_members
    ],
    format="csr"
)

# library-size normalize + log1p
lib = np.asarray(Xm.sum(axis=1)).ravel()

scale = np.divide(
    1e4,
    lib,
    out=np.zeros_like(lib, dtype=float),
    where=lib > 0
)

Xm = sp.diags(scale) @ Xm
Xm = Xm.astype(float)
Xm.data = np.log1p(Xm.data)

# ------------------------ 6. SELECT VARIABLE GENES --------------------------

mu = np.asarray(Xm.mean(axis=0)).ravel()
mu2 = np.asarray(Xm.power(2).mean(axis=0)).ravel()
var = np.maximum(mu2 - mu**2, 0)

n_keep = min(N_WGCNA_GENES, len(var))
selected = np.argsort(var)[::-1][:n_keep]

# force APLNR/APELA into network if available
name_to_idx = {g: i for i, g in enumerate(gene_names)}

forced = [
    name_to_idx[g]
    for g in ["APELA"]
    if g in name_to_idx
]

selected = np.unique(
    np.concatenate([selected, forced])
)

W = Xm[:, selected].toarray().astype(np.float32)
W_genes = gene_names[selected]

print(f"WGCNA matrix: {W.shape[0]} pseudocells × {W.shape[1]} genes")
#print("APLNR included:", "APLNR" in W_genes)
print("APELA included:", "APELA" in W_genes)

# ----------------------------- 7. AnnData -----------------------------------

wdata = ad.AnnData(
    X=W,
    obs=meta_obs.copy(),
    var=pd.DataFrame(index=W_genes)
)

wdata.var["gene_name"] = wdata.var_names

# numeric metadata can be useful later
wdata.obs["is_72h"] = (wdata.obs["timepoint"] == "72h").astype(int)

# ----------------------------- 8. WGCNA -------------------------------------

wgcna = PyWGCNA.WGCNA(
    name="Ctrl_APELA_epiblastside",
    species="human",
    anndata=wdata,

    TPMcutoff=0,

    networkType="signed",
    TOMType="signed",

    powers=list(range(1, 11)) + list(range(12, 21, 2)),
    RsquaredCut=0.80,

    minModuleSize=30,
    MEDissThres=0.20,

    save=True,
    outputPath=str(OUT) + "/",
    figureType="pdf"
)

wgcna.runWGCNA()
wgcna.CalculateSignedKME()
wgcna.saveWGCNA()

print("\nChosen soft-threshold power:", wgcna.power)

# --------------------------- 9. MODULE RESULTS ------------------------------

modules = wgcna.datExpr.var[
    ["moduleColors", "moduleLabels"]
].copy()

modules.to_csv(
    OUT / "WGCNA_gene_module_assignments.csv"
)

print("\nModule sizes:")
print(
    modules["moduleColors"]
    .value_counts()
    .to_string()
)

# ------------------------ 10. APLNR / APELA ---------------------------------

for seed in ["APLNR", "APELA"]:

    print("\n" + "=" * 70)
    print(seed)
    print("=" * 70)

    if seed not in modules.index:
        print(f"{seed} was not retained in the network.")
        continue

    module = modules.loc[seed, "moduleColors"]

    members = modules.index[
        modules["moduleColors"] == module
    ].tolist()

    print(f"Module: {module}")
    print(f"Module size: {len(members)} genes")

    hubs = wgcna.top_n_hub_genes(
        moduleName=module,
        n=30
    )

    display(hubs)

    hubs.to_csv(
        OUT / f"{seed}_{module}_top30_hubs.csv"
    )

    pd.Series(
        members,
        name="gene"
    ).to_csv(
        OUT / f"{seed}_{module}_all_genes.csv",
        index=False
    )

print("\nDone.")
print("Results saved to:")
print(OUT)

In [ ]:
# ============================================================================
# QC 1 — WHERE IS THE APLNR BLACK MODULE ACTIVE?
# Run after the WGCNA block
# Requires: wgcna, FIG, TAB
# ============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

MODULE = "black"

# ---------------------------------------------------------------------------
# module membership
# ---------------------------------------------------------------------------

modules = wgcna.datExpr.var[
    ["moduleColors", "moduleLabels"]
].copy()

black_genes = modules.index[
    modules["moduleColors"] == MODULE
].tolist()

print(f"{MODULE} module: {len(black_genes)} genes")
print("APLNR in module:", "APLNR" in black_genes)

# ---------------------------------------------------------------------------
# get module eigengene
# ---------------------------------------------------------------------------

ME = wgcna.datME.copy()

print("\nEigengene columns:")
print(ME.columns.tolist())

# PyWGCNA normally calls this MEblack
candidates = [
    c for c in ME.columns
    if str(c).lower() in ["meblack", "black"]
    or "black" in str(c).lower()
]

assert len(candidates) > 0, "Could not find black module eigengene."

ME_COL = candidates[0]
print("Using:", ME_COL)

# make sure sample order matches WGCNA pseudocells
if len(ME) == wgcna.datExpr.n_obs:
    ME.index = wgcna.datExpr.obs_names

qc = wgcna.datExpr.obs.copy()
qc["black_ME"] = np.asarray(ME.loc[qc.index, ME_COL]).ravel()

# ---------------------------------------------------------------------------
# orient score so HIGH = high expression of the black module
# ---------------------------------------------------------------------------

# WGCNA usually already aligns the eigengene this way, but verify.
module_idx = [
    wgcna.datExpr.var_names.get_loc(g)
    for g in black_genes
]

module_mean = np.asarray(
    wgcna.datExpr.X[:, module_idx].mean(axis=1)
).ravel()

r = np.corrcoef(qc["black_ME"], module_mean)[0, 1]

if r < 0:
    qc["black_ME"] *= -1
    print("Eigengene flipped so HIGH = high black-module expression.")

print(f"Correlation with mean module expression: {abs(r):.3f}")

# ---------------------------------------------------------------------------
# site-level summaries
# ---------------------------------------------------------------------------

site_me = (
    qc.groupby(
        ["site", "timepoint", "state"],
        observed=True
    )["black_ME"]
    .mean()
    .reset_index()
)

display(site_me)

site_me.to_csv(
    TAB / "APLNR_black_module_eigengene_by_site_state.csv",
    index=False
)

# ---------------------------------------------------------------------------
# order states biologically where possible
# ---------------------------------------------------------------------------

preferred = [
    "pluripotent",
    "AE-LC",
    "pPS-LC",
    "aPS-LC",
    "eNasM",
    "lNasM",
]

observed_states = site_me["state"].astype(str).unique().tolist()

STATE_ORDER = [
    x for x in preferred
    if x in observed_states
]

STATE_ORDER += [
    x for x in observed_states
    if x not in STATE_ORDER
]

# columns = site/time combinations
site_order = sorted(site_me["site"].astype(str).unique())

COL_ORDER = [
    (tp, site)
    for tp in ["42h", "72h"]
    for site in site_order
]

site_me["column"] = (
    site_me["timepoint"].astype(str)
    + " | "
    + site_me["site"].astype(str)
)

col_names = [
    f"{tp} | {site}"
    for tp, site in COL_ORDER
]

mat = (
    site_me
    .pivot(
        index="state",
        columns="column",
        values="black_ME"
    )
    .reindex(
        index=STATE_ORDER,
        columns=col_names
    )
)

# ---------------------------------------------------------------------------
# heatmap
# ---------------------------------------------------------------------------

arr = mat.to_numpy(dtype=float)

vmax = np.nanmax(np.abs(arr))
norm = TwoSlopeNorm(
    vmin=-vmax,
    vcenter=0,
    vmax=vmax
)

cmap = plt.cm.RdBu_r.copy()
cmap.set_bad("#E5E5E5")

fig, ax = plt.subplots(
    figsize=(1.5 * mat.shape[1] + 2.5,
             0.7 * mat.shape[0] + 2)
)

im = ax.imshow(
    np.ma.masked_invalid(arr),
    cmap=cmap,
    norm=norm,
    aspect="auto"
)

ax.set_xticks(range(mat.shape[1]))
ax.set_xticklabels(
    mat.columns,
    rotation=45,
    ha="right",
    fontsize=11
)

ax.set_yticks(range(mat.shape[0]))
ax.set_yticklabels(
    mat.index,
    fontsize=12
)

# white borders
ax.set_xticks(
    np.arange(-0.5, mat.shape[1], 1),
    minor=True
)
ax.set_yticks(
    np.arange(-0.5, mat.shape[0], 1),
    minor=True
)

ax.grid(
    which="minor",
    color="white",
    linewidth=2
)

ax.tick_params(
    which="minor",
    bottom=False,
    left=False
)

ax.tick_params(length=0)

for spine in ax.spines.values():
    spine.set_visible(False)

cbar = fig.colorbar(
    im,
    ax=ax,
    fraction=0.04,
    pad=0.03
)

cbar.set_label(
    "APLNR black-module eigengene",
    fontsize=11
)

ax.set_title(
    "APLNR-containing WGCNA module",
    fontsize=14
)

plt.tight_layout()

fig.savefig(
    FIG / "APLNR_black_module_eigengene_QC.svg",
    bbox_inches="tight"
)

fig.savefig(
    FIG / "APLNR_black_module_eigengene_QC.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================================
# QC 2 — SCORE THE CTRL-DERIVED BLACK MODULE IN CTRL vs APLNR KD
# Requires: adata, STAN_BC, FIG, TAB, black_genes
# ============================================================================

import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

STATE_COL = "predicted_leiden_sub_joint"
KEEP_TIMES = ["42h", "72h"]

# ---------------------------------------------------------------------------
# identify KD label
# ---------------------------------------------------------------------------

print("Conditions in dataset:")
print(adata.obs["condition"].astype(str).value_counts())

conditions = adata.obs["condition"].astype(str).unique().tolist()

if "DOX" in conditions:
    KD_LABEL = "DOX"
else:
    other = [
        x for x in conditions
        if x != "Ctrl"
    ]

    assert len(other) == 1, (
        "More than one non-Ctrl condition found. "
        "Set KD_LABEL manually."
    )

    KD_LABEL = other[0]

print("\nUsing KD condition:", KD_LABEL)

# ---------------------------------------------------------------------------
# Ctrl gets your Stan barcode filter;
# KD is retained normally
# ---------------------------------------------------------------------------

cond = adata.obs["condition"].astype(str)

keep_condition = cond.isin(
    ["Ctrl", KD_LABEL]
)

keep_time = (
    adata.obs["timepoint_true"]
    .astype(str)
    .isin(KEEP_TIMES)
)

keep_stan = (
    (cond != "Ctrl")
    | adata.obs["_bc"].isin(STAN_BC)
)

cmp = adata[
    keep_condition
    & keep_time
    & keep_stan
].copy()

cmp.obs["state"] = (
    cmp.obs[STATE_COL]
    .astype(str)
)

print(f"\nCells used: {cmp.n_obs:,}")

print(
    cmp.obs.groupby(
        ["condition", "site", "timepoint_true", "state"],
        observed=True
    )
    .size()
    .to_string()
)

# ---------------------------------------------------------------------------
# raw counts
# ---------------------------------------------------------------------------

assert "counts" in cmp.layers

X = cmp.layers["counts"]

if not sp.issparse(X):
    X = sp.csr_matrix(X)
else:
    X = X.tocsr()

# whole-cell library size
library_size = np.asarray(
    X.sum(axis=1)
).ravel()

# ---------------------------------------------------------------------------
# gene symbol -> columns
#
# If a symbol appears more than once, SUM those raw-count columns rather
# than arbitrarily throwing one away.
# ---------------------------------------------------------------------------

if "symbol" in cmp.var.columns:
    symbols = cmp.var["symbol"].astype(str).values
else:
    symbols = cmp.var_names.astype(str).values

symbol_to_cols = {}

for j, g in enumerate(symbols):
    symbol_to_cols.setdefault(g, []).append(j)

genes_present = [
    g for g in black_genes
    if g in symbol_to_cols
]

genes_missing = [
    g for g in black_genes
    if g not in symbol_to_cols
]

print(
    f"\nBlack-module genes found: "
    f"{len(genes_present)}/{len(black_genes)}"
)

if genes_missing:
    print("Missing:", genes_missing)

# ---------------------------------------------------------------------------
# build cell × module-gene RAW count matrix
# ---------------------------------------------------------------------------

module_columns = []

for g in genes_present:

    cols = symbol_to_cols[g]

    if len(cols) == 1:
        v = X[:, cols[0]]
    else:
        v = X[:, cols].sum(axis=1)

    module_columns.append(
        sp.csr_matrix(v)
    )

Xmod = sp.hstack(
    module_columns,
    format="csr"
)

# ---------------------------------------------------------------------------
# normalize to CP10K and log1p
# ---------------------------------------------------------------------------

scale = np.divide(
    1e4,
    library_size,
    out=np.zeros_like(
        library_size,
        dtype=float
    ),
    where=library_size > 0
)

Xmod = sp.diags(scale) @ Xmod
Xmod = Xmod.toarray().astype(np.float32)

Xmod = np.log1p(Xmod)

# ---------------------------------------------------------------------------
# module score:
#
# standardize each gene relative to CTRL,
# then average the z-scores across all black-module genes
# ---------------------------------------------------------------------------

ctrl_mask = (
    cmp.obs["condition"]
    .astype(str)
    .values
    == "Ctrl"
)

ctrl_mu = Xmod[ctrl_mask].mean(axis=0)
ctrl_sd = Xmod[ctrl_mask].std(axis=0)

# avoid divide-by-zero
usable = ctrl_sd > 1e-6

print(
    f"Genes contributing to score: "
    f"{usable.sum()}/{len(genes_present)}"
)

Z = (
    Xmod[:, usable]
    - ctrl_mu[usable]
) / ctrl_sd[usable]

cmp.obs["APLNR_black_module_score"] = (
    Z.mean(axis=1)
)

# ---------------------------------------------------------------------------
# CELL-level values are descriptive;
# summarize by SITE for the actual Ctrl/KD comparison.
# ---------------------------------------------------------------------------

site_score = (
    cmp.obs.groupby(
        [
            "site",
            "timepoint_true",
            "state",
            "condition"
        ],
        observed=True
    )["APLNR_black_module_score"]
    .mean()
    .reset_index()
)

site_score.to_csv(
    TAB / "APLNR_black_module_score_site_means.csv",
    index=False
)

display(site_score)

# ---------------------------------------------------------------------------
# calculate KD - Ctrl independently within each site/state/time
# ---------------------------------------------------------------------------

wide = (
    site_score
    .pivot_table(
        index=[
            "site",
            "timepoint_true",
            "state"
        ],
        columns="condition",
        values="APLNR_black_module_score"
    )
)

assert "Ctrl" in wide.columns
assert KD_LABEL in wide.columns

wide["KD_minus_Ctrl"] = (
    wide[KD_LABEL]
    - wide["Ctrl"]
)

delta = (
    wide["KD_minus_Ctrl"]
    .reset_index()
)

display(delta)

delta.to_csv(
    TAB / "APLNR_black_module_KD_minus_Ctrl_by_site.csv",
    index=False
)

# ---------------------------------------------------------------------------
# summary across Chicago + Michigan
# ---------------------------------------------------------------------------

summary = (
    delta.groupby(
        ["timepoint_true", "state"],
        observed=True
    )["KD_minus_Ctrl"]
    .agg(
        mean_delta="mean",
        min_delta="min",
        max_delta="max",
        n_sites="count"
    )
    .reset_index()
)

# both sites same direction?
site_direction = (
    delta.assign(
        sign=np.sign(delta["KD_minus_Ctrl"])
    )
    .groupby(
        ["timepoint_true", "state"],
        observed=True
    )["sign"]
    .agg(
        lambda x:
        len(set(x[x != 0])) <= 1
    )
    .rename("sites_same_direction")
    .reset_index()
)

summary = summary.merge(
    site_direction,
    on=["timepoint_true", "state"],
    how="left"
)

display(summary)

summary.to_csv(
    TAB / "APLNR_black_module_KD_summary.csv",
    index=False
)

# ---------------------------------------------------------------------------
# HEATMAP:
# KD - Ctrl for every state / site / time
# ---------------------------------------------------------------------------

preferred = [
    "pluripotent",
    "AE-LC",
    "pPS-LC",
    "aPS-LC",
    "eNasM",
    "lNasM",
]

observed_states = (
    delta["state"]
    .astype(str)
    .unique()
    .tolist()
)

STATE_ORDER = [
    x for x in preferred
    if x in observed_states
]

STATE_ORDER += [
    x for x in observed_states
    if x not in STATE_ORDER
]

delta["column"] = (
    delta["timepoint_true"].astype(str)
    + " | "
    + delta["site"].astype(str)
)

site_order = sorted(
    delta["site"]
    .astype(str)
    .unique()
)

COLS = [
    f"{tp} | {site}"
    for tp in ["42h", "72h"]
    for site in site_order
]

mat = (
    delta
    .pivot(
        index="state",
        columns="column",
        values="KD_minus_Ctrl"
    )
    .reindex(
        index=STATE_ORDER,
        columns=COLS
    )
)

arr = mat.to_numpy(dtype=float)

vmax = np.nanmax(np.abs(arr))

norm = TwoSlopeNorm(
    vmin=-vmax,
    vcenter=0,
    vmax=vmax
)

cmap = plt.cm.RdBu_r.copy()
cmap.set_bad("#E5E5E5")

fig, ax = plt.subplots(
    figsize=(
        1.5 * mat.shape[1] + 2.8,
        0.72 * mat.shape[0] + 2
    )
)

im = ax.imshow(
    np.ma.masked_invalid(arr),
    cmap=cmap,
    norm=norm,
    aspect="auto"
)

ax.set_xticks(
    range(mat.shape[1])
)

ax.set_xticklabels(
    mat.columns,
    rotation=45,
    ha="right",
    fontsize=11
)

ax.set_yticks(
    range(mat.shape[0])
)

ax.set_yticklabels(
    mat.index,
    fontsize=12
)

ax.set_xticks(
    np.arange(-0.5, mat.shape[1], 1),
    minor=True
)

ax.set_yticks(
    np.arange(-0.5, mat.shape[0], 1),
    minor=True
)

ax.grid(
    which="minor",
    color="white",
    linewidth=2
)

ax.tick_params(
    which="minor",
    bottom=False,
    left=False
)

ax.tick_params(length=0)

for spine in ax.spines.values():
    spine.set_visible(False)

cbar = fig.colorbar(
    im,
    ax=ax,
    fraction=0.04,
    pad=0.03
)

cbar.set_label(
    f"Module score: {KD_LABEL} − Ctrl",
    fontsize=11
)

ax.set_title(
    "Effect of APLNR KD on the Ctrl-derived APLNR module",
    fontsize=14
)

plt.tight_layout()

fig.savefig(
    FIG / "APLNR_black_module_KD_minus_Ctrl.svg",
    bbox_inches="tight"
)

fig.savefig(
    FIG / "APLNR_black_module_KD_minus_Ctrl.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================================
# APLNR MODULE ROBUSTNESS QC
# 1) hub rank
# 2) overlap with original all-tissue APLNR module
# 3) relationship to cell cycle
# ============================================================================

import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr

MODULE = "black"

# ============================================================================
# 1. APLNR HUB RANK IN MESODERM-ONLY NETWORK
# ============================================================================

modules = wgcna.datExpr.var[
    ["moduleColors", "moduleLabels"]
].copy()

meso_genes = modules.index[
    modules["moduleColors"] == MODULE
].tolist()

print("=" * 70)
print("MESODERM-ONLY APLNR MODULE")
print("=" * 70)

print("Module:", MODULE)
print("Genes:", len(meso_genes))
print("APLNR present:", "APLNR" in meso_genes)

# signed kME should already exist if CalculateSignedKME() was run
if not hasattr(wgcna, "signedKME") or wgcna.signedKME is None:
    wgcna.CalculateSignedKME()

print("\nkME columns:")
print(wgcna.signedKME.columns.tolist())

# identify black-module kME
kme_cols = [
    c for c in wgcna.signedKME.columns
    if "black" in str(c).lower()
]

assert len(kme_cols) > 0, "Could not find black-module kME column."

KME_COL = kme_cols[0]

kme = (
    wgcna.signedKME[KME_COL]
    .reindex(meso_genes)
    .dropna()
    .sort_values(ascending=False)
)

# rank
kme_table = pd.DataFrame({
    "gene": kme.index,
    "kME": kme.values
})

kme_table["hub_rank"] = np.arange(1, len(kme_table) + 1)

print("\nTOP 30 HUBS:")
display(kme_table.head(30))

if "APLNR" in set(kme_table["gene"]):
    aplnr_row = kme_table[
        kme_table["gene"] == "APLNR"
    ]

    print("\nAPLNR:")
    display(aplnr_row)

kme_table.to_csv(
    TAB / "WGCNA_Ctrl_mesoderm_only" /
    "APLNR_black_all_hub_ranks.csv",
    index=False
)


# ============================================================================
# 2. COMPARE TO ORIGINAL ALL-TISSUE BLACK MODULE
# ============================================================================

OLD_FILE = (
    TAB /
    "WGCNA_allCtrl_allTissues" /
    "APLNR_black_all_genes.csv"
)

print("\n" + "=" * 70)
print("ALL-TISSUE vs MESODERM-ONLY MODULE")
print("=" * 70)

if OLD_FILE.exists():

    old = pd.read_csv(OLD_FILE)

    # handle whichever column name the earlier export used
    if "gene" in old.columns:
        old_genes = set(old["gene"].dropna().astype(str))
    else:
        old_genes = set(
            old.iloc[:, -1]
            .dropna()
            .astype(str)
        )

    new_genes = set(meso_genes)

    overlap = old_genes & new_genes
    union = old_genes | new_genes

    print("All-tissue module:", len(old_genes))
    print("Mesoderm-only module:", len(new_genes))
    print("Shared genes:", len(overlap))

    print(
        "Fraction of old module retained:",
        f"{len(overlap) / len(old_genes):.1%}"
    )

    print(
        "Fraction of new module shared:",
        f"{len(overlap) / len(new_genes):.1%}"
    )

    print(
        "Jaccard:",
        f"{len(overlap) / len(union):.3f}"
    )

    overlap_df = pd.DataFrame({
        "gene": sorted(overlap)
    })

    overlap_df.to_csv(
        TAB / "WGCNA_Ctrl_mesoderm_only" /
        "APLNR_module_overlap_with_all_tissue.csv",
        index=False
    )

    print("\nShared genes:")
    print(", ".join(sorted(overlap)))

else:

    print("Could not find:")
    print(OLD_FILE)
    print(
        "\nIf the old folder has a different name, "
        "just point OLD_FILE to the original "
        "APLNR_black_all_genes.csv."
    )


# ============================================================================
# 3. CHECK SPECIFIC BIOLOGICAL GENES
# ============================================================================

CORE = [
    "APLNR",
    "MESP1",
    "MESP2",
    "MIXL1",
    "TBXT",
    "EOMES",
    "DLL3",
    "LHX1",
    "HAS2",
    "SERPINE2",
    "RSPO3",
    "WNT5A",
    "WNT8A",
    "CDH2",
    "CDH11",
    "FERMT2",
    "BMP2",
    "GATA6",
    "ZEB2",
    "ROR2",
    "PDGFRA",
    "CYP26A1",
]

core_table = pd.DataFrame({
    "gene": CORE,
    "in_meso_APLNR_module": [
        g in set(meso_genes)
        for g in CORE
    ]
})

if OLD_FILE.exists():
    core_table["in_all_tissue_APLNR_module"] = [
        g in old_genes
        for g in CORE
    ]

display(core_table)


# ============================================================================
# 4. IS THIS BASICALLY A CELL-CYCLE MODULE?
#
# Use an INDEPENDENT canonical cell-cycle gene set and REMOVE any genes that
# are themselves members of the black module before computing the score.
# This makes the test substantially less circular.
# ============================================================================

S_GENES = [
    "MCM5","PCNA","TYMS","FEN1","MCM2","MCM4","RRM1","UNG",
    "GINS2","MCM6","CDCA7","DTL","PRIM1","UHRF1","HELLS",
    "RFC2","RPA2","NASP","RAD51AP1","GMNN","WDR76","SLBP",
    "CCNE2","UBR7","POLD3","MSH2","ATAD2","RAD51","RRM2",
    "CDC45","CDC6","EXO1","TIPIN","DSCC1","BLM","CASP8AP2",
    "USP1","CLSPN","POLA1","CHAF1B","BRIP1","E2F8"
]

G2M_GENES = [
    "HMGB2","CDK1","NUSAP1","UBE2C","BIRC5","TPX2","TOP2A",
    "NDC80","CKS2","NUF2","CKS1B","MKI67","TMPO","CENPF",
    "TACC3","FAM64A","SMC4","CCNB2","CKAP2L","CKAP2","AURKB",
    "BUB1","KIF11","ANP32E","TUBB4B","GTSE1","KIF20B","HJURP",
    "CDCA3","HN1","CDC20","TTK","CDC25C","KIF2C","RANGAP1",
    "NCAPD2","DLGAP5","CDCA2","CDCA8","ECT2","KIF23","HMMR",
    "AURKA","PSRC1","ANLN","LBR","CKAP5","CENPE","CTCF",
    "NEK2","G2E3","GAS2L3","CBX5","CENPA"
]

cycle_genes = set(S_GENES + G2M_GENES)

available = set(wgcna.datExpr.var_names)

# critical: exclude black-module genes themselves
cycle_independent = sorted(
    (cycle_genes & available)
    - set(meso_genes)
)

print("\n" + "=" * 70)
print("CELL-CYCLE QC")
print("=" * 70)

print(
    "Independent cell-cycle genes available:",
    len(cycle_independent)
)

print(", ".join(cycle_independent))

# ---------------------------------------------------------------------------
# Black module eigengene
# ---------------------------------------------------------------------------

ME = wgcna.datME.copy()

ME_COL = [
    c for c in ME.columns
    if "black" in str(c).lower()
][0]

ME.index = wgcna.datExpr.obs_names

black_ME = np.asarray(
    ME.loc[wgcna.datExpr.obs_names, ME_COL]
).ravel()

# Orient high = high module expression
module_idx = [
    wgcna.datExpr.var_names.get_loc(g)
    for g in meso_genes
]

module_mean = np.asarray(
    wgcna.datExpr.X[:, module_idx].mean(axis=1)
).ravel()

if np.corrcoef(black_ME, module_mean)[0,1] < 0:
    black_ME *= -1

# ---------------------------------------------------------------------------
# Independent cell-cycle score
# ---------------------------------------------------------------------------

if len(cycle_independent) >= 5:

    cyc_idx = [
        wgcna.datExpr.var_names.get_loc(g)
        for g in cycle_independent
    ]

    C = np.asarray(
        wgcna.datExpr.X[:, cyc_idx]
    )

    # z-score each cell-cycle gene across pseudocells
    mu = C.mean(axis=0)
    sd = C.std(axis=0)

    good = sd > 1e-6

    Cz = (
        C[:, good]
        - mu[good]
    ) / sd[good]

    cycle_score = Cz.mean(axis=1)

    # -----------------------------------------------------------------------
    # raw association
    # -----------------------------------------------------------------------

    rp, pp = pearsonr(
        black_ME,
        cycle_score
    )

    rs, ps = spearmanr(
        black_ME,
        cycle_score
    )

    print(
        f"\nRaw Pearson black ME vs cell cycle: "
        f"r = {rp:.3f}, p = {pp:.3g}"
    )

    print(
        f"Raw Spearman: "
        f"rho = {rs:.3f}, p = {ps:.3g}"
    )

    # -----------------------------------------------------------------------
    # STRONGER TEST:
    # remove site × time × state differences first
    # -----------------------------------------------------------------------

    obs = wgcna.datExpr.obs.copy()

    design = pd.get_dummies(
        obs[
            ["site", "timepoint", "state"]
        ].astype(str),
        drop_first=True,
        dtype=float
    )

    # intercept
    Xdesign = np.column_stack([
        np.ones(len(design)),
        design.to_numpy()
    ])

    def residualize(y, X):
        beta, *_ = np.linalg.lstsq(
            X,
            y,
            rcond=None
        )
        return y - X @ beta

    black_resid = residualize(
        black_ME,
        Xdesign
    )

    cycle_resid = residualize(
        cycle_score,
        Xdesign
    )

    rp2, pp2 = pearsonr(
        black_resid,
        cycle_resid
    )

    rs2, ps2 = spearmanr(
        black_resid,
        cycle_resid
    )

    print(
        "\nAfter controlling for site + time + state:"
    )

    print(
        f"Pearson r = {rp2:.3f}, "
        f"p = {pp2:.3g}"
    )

    print(
        f"Spearman rho = {rs2:.3f}, "
        f"p = {ps2:.3g}"
    )

    cycle_qc = pd.DataFrame({
        "black_ME": black_ME,
        "cell_cycle_score": cycle_score,
        "black_ME_residual": black_resid,
        "cell_cycle_residual": cycle_resid,
    }, index=wgcna.datExpr.obs_names)

    cycle_qc = pd.concat(
        [
            wgcna.datExpr.obs[
                ["site", "timepoint", "state"]
            ],
            cycle_qc
        ],
        axis=1
    )

    cycle_qc.to_csv(
        TAB / "WGCNA_Ctrl_mesoderm_only" /
        "APLNR_black_module_cellcycle_QC.csv"
    )

else:

    print(
        "Too few independent cell-cycle genes remained "
        "after excluding black-module genes."
    )

In [ ]:
# ============================================================================
# MATCHED APLNR WGCNA NETWORK PLOTS
#
# Plot 1 = ALL Ctrl tissues
# Plot 2 = MESODERM-only Ctrl
#
# Node size  = kME (module membership)
# Edge       = TOM relationship
# Edge width = TOM strength
# APLNR      = thick black outline
#
# Selection is OBJECTIVE:
# top N genes by kME, with APLNR forced in if needed
# ============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import networkx as nx

from pathlib import Path
from PyWGCNA.utils import readWGCNA


# ---------------------------------------------------------------------------
# 1. LOAD THE TWO SAVED NETWORKS
# ---------------------------------------------------------------------------
# ---------------------------------------------------------------------------
# LOAD THE TWO SAVED WGCNA OBJECTS
# ---------------------------------------------------------------------------

from pathlib import Path
from PyWGCNA.utils import readWGCNA

ALL_DIR = (
    TAB
    / "WGCNA_allCtrl_allTissues"
)

MESO_DIR = (
    TAB
    / "WGCNA_Ctrl_mesoderm_only"
)

print("All-tissue folder:", ALL_DIR)
print("Mesoderm folder:  ", MESO_DIR)

assert ALL_DIR.exists(), f"Missing folder: {ALL_DIR}"
assert MESO_DIR.exists(), f"Missing folder: {MESO_DIR}"


def find_wgcna_object(folder):
    # PyWGCNA normally saves a .p object
    hits = sorted(folder.glob("*.p"))

    if len(hits) == 0:
        # show us what actually got saved if no .p exists
        print(f"\nNo .p file found in {folder}")
        print("Files present:")
        for x in folder.iterdir():
            print(" ", x.name)
        raise FileNotFoundError(folder)

    if len(hits) > 1:
        print(f"\nMultiple .p files found in {folder}:")
        for x in hits:
            print(" ", x.name)
        print("Using:", hits[0].name)

    return hits[0]


ALL_P = find_wgcna_object(ALL_DIR)
MESO_P = find_wgcna_object(MESO_DIR)

print("\nLoading:")
print(" all-tissue:", ALL_P.name)
print(" mesoderm:  ", MESO_P.name)

all_wgcna = readWGCNA(str(ALL_P))
meso_wgcna = readWGCNA(str(MESO_P))

print("\nLoaded both WGCNA objects.")

# ---------------------------------------------------------------------------
# 2. BROAD FUNCTIONAL CLASSES FOR NODE COLOR
#
# These colors are annotation ONLY.
# Network construction / node selection does not use these categories.
# ---------------------------------------------------------------------------

SPECIFICATION = {
    "MESP1", "MESP2", "MIXL1", "TBXT", "EOMES", "TBX6",
    "LHX1", "GATA6", "MSX1", "IRX3", "ZIC3", "CYP26A1",
    "FOXC1"
}

SIGNALING = {
    "APLNR", "RSPO3", "WNT5A", "WNT8A", "BMP2", "CER1",
    "FGF17", "CNTFR", "SP5"
}

MIGRATION_ECM = {
    "HAS2", "VIM", "FERMT2", "SERPINE2", "RND3",
    "FSCN1", "GJA1", "TFPI2", "MFAP4", "IGFBP4",
    "ANXA6", "CDH2", "CDH11", "ROR2", "PDGFRA"
}

PROLIFERATION = {
    "TPX2", "DLGAP5", "TOP2A", "AURKB", "BIRC5",
    "CCNB1", "CCNB2", "CDC20", "CENPF", "NUSAP1",
    "PTTG1", "SGO2", "HMMR", "PRR11", "UBE2S"
}

CLASS_COLORS = {
    "Mesoderm specification": "#A33A22",
    "Signaling / patterning":  "#D88A26",
    "Migration / ECM":         "#557F7A",
    "Proliferation":           "#7774A5",
    "Other":                   "#B9B9B9",
}


def gene_class(g):
    if g in SPECIFICATION:
        return "Mesoderm specification"
    if g in SIGNALING:
        return "Signaling / patterning"
    if g in MIGRATION_ECM:
        return "Migration / ECM"
    if g in PROLIFERATION:
        return "Proliferation"
    return "Other"


# ---------------------------------------------------------------------------
# 3. CIRCULAR NETWORK FUNCTION
# ---------------------------------------------------------------------------

def plot_aplnr_module_network(
    obj,
    module="black",
    n_nodes=25,
    n_edges=60,
    inner_n=9,
    title="",
    stem="APLNR_network",
):

    # -----------------------------------------------------------------------
    # module genes
    # -----------------------------------------------------------------------

    modules = obj.datExpr.var["moduleColors"].astype(str)

    module_genes = modules.index[
        modules == module
    ].tolist()

    assert "APLNR" in module_genes, (
        f"APLNR is not in the {module} module."
    )

    # -----------------------------------------------------------------------
    # kME
    # -----------------------------------------------------------------------

    if (
        not hasattr(obj, "signedKME")
        or obj.signedKME is None
        or len(obj.signedKME) == 0
    ):
        obj.CalculateSignedKME()

    candidates = [
        c for c in obj.signedKME.columns
        if str(c).lower() == f"kme{module}".lower()
    ]

    assert len(candidates) == 1, (
        f"Could not uniquely find kME column for {module}: "
        f"{obj.signedKME.columns.tolist()}"
    )

    kme_col = candidates[0]

    kme = (
        obj.signedKME[kme_col]
        .reindex(module_genes)
        .dropna()
        .sort_values(ascending=False)
    )

    # -----------------------------------------------------------------------
    # OBJECTIVE node selection:
    # top genes by kME
    # -----------------------------------------------------------------------

    chosen = kme.head(n_nodes).index.tolist()

    # This is an APLNR panel, so make sure APLNR appears.
    # If it falls below N, replace the last displayed hub.
    if "APLNR" not in chosen:
        chosen = chosen[:-1] + ["APLNR"]

    # retain kME ordering
    chosen = sorted(
        chosen,
        key=lambda g: kme.get(g, -np.inf),
        reverse=True
    )

    aplnr_rank = int(
        np.where(kme.index.to_numpy() == "APLNR")[0][0] + 1
    )

    # -----------------------------------------------------------------------
    # TOM submatrix
    #
    # PyWGCNA TOM rows follow datExpr.var_names.
    # -----------------------------------------------------------------------

    all_genes = pd.Index(obj.datExpr.var_names)

    idx = [
        all_genes.get_loc(g)
        for g in chosen
    ]

    TOM = np.asarray(obj.TOM)

    T = TOM[np.ix_(idx, idx)]

    # -----------------------------------------------------------------------
    # Build complete weighted graph among displayed genes
    # -----------------------------------------------------------------------

    full = nx.Graph()

    for g in chosen:
        full.add_node(g)

    for i in range(len(chosen)):
        for j in range(i + 1, len(chosen)):

            w = float(T[i, j])

            if np.isfinite(w):
                full.add_edge(
                    chosen[i],
                    chosen[j],
                    weight=w
                )

    # -----------------------------------------------------------------------
    # Keep enough edges to show structure without creating spaghetti.
    #
    # First use maximum-spanning tree so every displayed node has a path.
    # Then add strongest remaining TOM edges up to n_edges total.
    # -----------------------------------------------------------------------

    tree = nx.maximum_spanning_tree(
        full,
        weight="weight"
    )

    G = nx.Graph()

    G.add_nodes_from(chosen)
    G.add_edges_from(tree.edges(data=True))

    remaining = sorted(
        [
            (u, v, d)
            for u, v, d in full.edges(data=True)
            if not G.has_edge(u, v)
        ],
        key=lambda x: x[2]["weight"],
        reverse=True
    )

    for u, v, d in remaining:

        if G.number_of_edges() >= n_edges:
            break

        G.add_edge(
            u,
            v,
            **d
        )

    # -----------------------------------------------------------------------
    # Circular positions
    #
    # Highest-kME hubs = inner ring
    # remainder = outer ring
    #
    # Distance is therefore NOT an inferred biological metric;
    # ring assignment simply reflects module membership rank.
    # -----------------------------------------------------------------------

    inner = chosen[:inner_n]
    outer = chosen[inner_n:]

    pos = {}

    # inner ring
    angles = np.linspace(
        np.pi / 2,
        np.pi / 2 + 2*np.pi,
        len(inner),
        endpoint=False
    )

    for g, a in zip(inner, angles):
        pos[g] = (
            1.00 * np.cos(a),
            1.00 * np.sin(a)
        )

    # outer ring
    angles = np.linspace(
        np.pi / 2,
        np.pi / 2 + 2*np.pi,
        len(outer),
        endpoint=False
    )

    for g, a in zip(outer, angles):
        pos[g] = (
            1.90 * np.cos(a),
            1.90 * np.sin(a)
        )

    # -----------------------------------------------------------------------
    # plotting values
    # -----------------------------------------------------------------------

    kvals = np.array([
        float(kme[g])
        for g in chosen
    ])

    # size = kME
    kmin = np.nanmin(kvals)
    kmax = np.nanmax(kvals)

    sizes = {}

    for g in chosen:

        v = float(kme[g])

        scaled = (
            (v - kmin)
            / (kmax - kmin + 1e-12)
        )

        sizes[g] = 700 + 1300 * scaled

    edge_vals = np.array([
        G[u][v]["weight"]
        for u, v in G.edges()
    ])

    emin = edge_vals.min()
    emax = edge_vals.max()

    widths = []

    for u, v in G.edges():

        w = G[u][v]["weight"]

        scaled = (
            (w - emin)
            / (emax - emin + 1e-12)
        )

        widths.append(
            0.4 + 3.5 * scaled
        )

    # -----------------------------------------------------------------------
    # FIGURE
    # -----------------------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=(10, 10)
    )

    # edges
    nx.draw_networkx_edges(
        G,
        pos,
        width=widths,
        edge_color="#AAA7A2",
        alpha=0.42,
        ax=ax
    )

    # draw each functional group separately
    for cls, color in CLASS_COLORS.items():

        genes = [
            g for g in chosen
            if gene_class(g) == cls
            and g != "APLNR"
        ]

        if not genes:
            continue

        nx.draw_networkx_nodes(
            G,
            pos,
            nodelist=genes,
            node_size=[sizes[g] for g in genes],
            node_color=color,
            edgecolors="white",
            linewidths=1.6,
            ax=ax
        )

    # APLNR separately
    nx.draw_networkx_nodes(
        G,
        pos,
        nodelist=["APLNR"],
        node_size=[sizes["APLNR"]],
        node_color=CLASS_COLORS[
            gene_class("APLNR")
        ],
        edgecolors="black",
        linewidths=3.2,
        ax=ax
    )

    # -----------------------------------------------------------------------
    # labels slightly outside the nodes
    # -----------------------------------------------------------------------

    for g in chosen:

        x, y = pos[g]

        r = np.sqrt(x*x + y*y)

        if r == 0:
            dx, dy = 0, 0
        else:
            dx = 0.13 * x / r
            dy = 0.13 * y / r

        ax.text(
            x + dx,
            y + dy,
            g,
            ha="center",
            va="center",
            fontsize=10.5,
            fontweight=(
                "bold"
                if g == "APLNR"
                else "normal"
            ),
            color="#222222"
        )

    # -----------------------------------------------------------------------
    # title + useful quantitative subtitle
    # -----------------------------------------------------------------------

    ax.set_title(
        title,
        fontsize=17,
        pad=18
    )

    ax.text(
        0.5,
        0.965,
        (
            f"{len(module_genes)}-gene module  •  "
            f"APLNR kME={kme['APLNR']:.2f}  •  "
            f"APLNR rank={aplnr_rank}/{len(kme)}"
        ),
        transform=ax.transAxes,
        ha="center",
        va="top",
        fontsize=10.5,
        color="#555555"
    )

    # -----------------------------------------------------------------------
    # legends
    # -----------------------------------------------------------------------

    class_handles = [
        plt.Line2D(
            [0], [0],
            marker="o",
            linestyle="",
            markersize=9,
            markerfacecolor=color,
            markeredgecolor="none",
            label=cls
        )
        for cls, color in CLASS_COLORS.items()
    ]

    class_handles.append(
        plt.Line2D(
            [0], [0],
            color="#AAA7A2",
            linewidth=2,
            label="TOM co-expression"
        )
    )

    ax.legend(
        handles=class_handles,
        loc="lower center",
        bbox_to_anchor=(0.5, -0.08),
        ncol=3,
        frameon=False,
        fontsize=9
    )

    ax.set_aspect("equal")
    ax.axis("off")

    plt.tight_layout()

    # -----------------------------------------------------------------------
    # save plot + exact displayed network
    # -----------------------------------------------------------------------

    for ext in ["svg", "png"]:
        fig.savefig(
            FIG / f"{stem}.{ext}",
            dpi=300 if ext == "png" else None,
            bbox_inches="tight"
        )

    # save displayed nodes
    shown_nodes = pd.DataFrame({
        "gene": chosen,
        "kME": [kme[g] for g in chosen],
        "kME_rank": [
            int(
                np.where(
                    kme.index.to_numpy() == g
                )[0][0] + 1
            )
            for g in chosen
        ],
        "functional_class": [
            gene_class(g)
            for g in chosen
        ],
    })

    shown_nodes.to_csv(
        TAB / f"{stem}_nodes.csv",
        index=False
    )

    # save displayed edges
    shown_edges = pd.DataFrame([
        {
            "gene1": u,
            "gene2": v,
            "TOM": d["weight"]
        }
        for u, v, d in G.edges(data=True)
    ]).sort_values(
        "TOM",
        ascending=False
    )

    shown_edges.to_csv(
        TAB / f"{stem}_edges.csv",
        index=False
    )

    plt.show()

    return shown_nodes, shown_edges


# ============================================================================
# 4. PLOT A — ALL-TISSUE DISCOVERY NETWORK
# ============================================================================

all_nodes, all_edges = plot_aplnr_module_network(
    all_wgcna,
    module="black",
    n_nodes=25,
    n_edges=60,
    inner_n=9,
    title="APLNR co-expression module — all control cell states",
    stem="APLNR_WGCNA_all_tissues_network"
)


# ============================================================================
# 5. PLOT B — MESODERM-ONLY ROBUSTNESS NETWORK
# ============================================================================

meso_nodes, meso_edges = plot_aplnr_module_network(
    meso_wgcna,
    module="black",
    n_nodes=25,
    n_edges=60,
    inner_n=9,
    title="APLNR co-expression module — mesoderm lineage",
    stem="APLNR_WGCNA_mesoderm_network"
)

In [ ]:
# ============================================================================
# LOAD CTRL MESODERM-ONLY WGCNA
# EXPORT APLNR BLACK MODULE + kME
#
# Requires: TAB already defined
# ============================================================================

import pandas as pd
from PyWGCNA import readWGCNA


# ---------------------------------------------------------------------------
# 1. LOAD THE EXISTING CTRL MESODERM-ONLY WGCNA
# ---------------------------------------------------------------------------

WGCNA_FILE = (
    TAB
    / "WGCNA_Ctrl_mesoderm_only"
    / "Ctrl_mesoderm_only.p"
)

print("Loading:")
print(WGCNA_FILE)

ctrl_wgcna = readWGCNA(
    str(WGCNA_FILE)
)


# ---------------------------------------------------------------------------
# 2. GET MODULE ASSIGNMENTS
# ---------------------------------------------------------------------------

modules = ctrl_wgcna.datExpr.var[
    ["moduleColors", "moduleLabels"]
].copy()

print("\nModule sizes:")
print(
    modules["moduleColors"]
    .value_counts()
    .to_string()
)


# ---------------------------------------------------------------------------
# 3. GET APLNR MODULE
# ---------------------------------------------------------------------------

assert "APLNR" in modules.index, (
    "APLNR not found in the saved WGCNA object."
)

APLNR_MODULE = str(
    modules.loc["APLNR", "moduleColors"]
)

print("\nAPLNR module:", APLNR_MODULE)


# ---------------------------------------------------------------------------
# 4. GET ALL GENES IN THE APLNR MODULE
# ---------------------------------------------------------------------------

module_genes = (
    modules.index[
        modules["moduleColors"].astype(str) == APLNR_MODULE
    ]
    .astype(str)
    .tolist()
)

print("APLNR module size:", len(module_genes))


# ---------------------------------------------------------------------------
# 5. EXTRACT kME FOR EACH GENE
# ---------------------------------------------------------------------------

KME_COL = f"kME{APLNR_MODULE}"

assert KME_COL in ctrl_wgcna.signedKME.columns, (
    f"Could not find {KME_COL} in signedKME."
)

meso_kme = pd.DataFrame({
    "gene": module_genes,

    "kME": ctrl_wgcna.signedKME.loc[
        module_genes,
        KME_COL
    ].to_numpy()
})


# highest module-membership genes first
meso_kme = (
    meso_kme
    .sort_values(
        "kME",
        ascending=False
    )
    .reset_index(drop=True)
)

# add rank
meso_kme["kME_rank"] = (
    meso_kme.index + 1
)


# ---------------------------------------------------------------------------
# 6. QUICK SANITY CHECK
# ---------------------------------------------------------------------------

print("\nOutput shape:")
print(meso_kme.shape)

print("\nTop module genes:")
display(
    meso_kme.head(20)
)

print("\nAPLNR:")
display(
    meso_kme[
        meso_kme["gene"] == "APLNR"
    ]
)


# ---------------------------------------------------------------------------
# 7. SAVE
# ---------------------------------------------------------------------------

OUT = (
    TAB
    / "APLNR_Ctrl_mesoOnly_WGCNA_black_module_with_kME.csv"
)

meso_kme.to_csv(
    OUT,
    index=False
)

print("\nSaved to:")
print(OUT.resolve())

In [ ]:
# =============================================================================
# APLNR Ctrl-defined module preservation / rewiring in shAPLNR
#
# INPUT:
#   - existing `adata`
#   - Ctrl mesoderm WGCNA 218-gene module table
#
# QUESTION:
#   Do the 218 genes that form the normal Ctrl APLNR-containing module
#   remain coordinated after APLNR KD?
#
# IMPORTANT:
#   This does NOT discover a new KD module.
#   The Ctrl-defined 218 genes are frozen before looking at KD.
# =============================================================================

from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# ---------------------------------------------------------------------------
# paths
# ---------------------------------------------------------------------------
MODULE_FILE = TAB / "APLNR_Ctrl_mesoOnly_WGCNA_black_module_with_kME.csv"

OUT = TAB / "WGCNA_Ctrl_mesoderm_only" / "KD_preservation"
OUT.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------------
# AnnData columns
# ---------------------------------------------------------------------------
COND_COL  = "condition"
SITE_COL  = "site"
TIME_COL  = "timepoint_true"
STATE_COL = "predicted_leiden_sub_joint"
COUNTS_LAYER = "counts"

# Main well-powered developmental comparisons
KEEP_STATE_TIME = {
    ("pPS-LC", "42"),
    ("aPS-LC", "42"),
    ("eNasM",  "72"),
    ("lNasM",  "72"),
}

# pseudocells
TARGET_CELLS = 25
MIN_CELLS = 10
MIN_PSEUDOCELLS_PER_COND = 2
SEED = 17
rng = np.random.default_rng(SEED)

# =============================================================================
# 1. LOAD THE FROZEN CTRL MODULE
# =============================================================================

module = pd.read_csv(MODULE_FILE)

module["gene"] = module["gene"].astype(str)
module_genes = list(dict.fromkeys(module["gene"]))

module_genes = [g for g in module_genes if g in adata.var_names]

print(f"Ctrl WGCNA module genes in table: {len(module)}")
print(f"Found in adata: {len(module_genes)}")

if len(module_genes) < 150:
    raise ValueError(
        f"Only {len(module_genes)} module genes found in adata. "
        "Check whether adata.var_names are gene symbols."
    )

# =============================================================================
# 2. IDENTIFY CTRL AND KD LABELS
# =============================================================================

conditions = sorted(adata.obs[COND_COL].astype(str).unique())
print("\nConditions:", conditions)

CTRL_LABEL = next(
    (x for x in conditions if x.lower() in {"ctrl", "control"}),
    None
)

if CTRL_LABEL is None:
    raise ValueError("Could not identify Ctrl condition.")

kd_candidates = [
    x for x in conditions
    if x != CTRL_LABEL
    and any(k in x.lower() for k in ["dox", "kd", "shaplnr", "aplnr"])
]

# if dataset has exactly Ctrl + one other condition
if len(kd_candidates) == 0 and len(conditions) == 2:
    kd_candidates = [x for x in conditions if x != CTRL_LABEL]

if len(kd_candidates) != 1:
    raise ValueError(
        f"Could not uniquely identify KD. Candidates: {kd_candidates}\n"
        f"All conditions: {conditions}"
    )

KD_LABEL = kd_candidates[0]

print("CTRL =", CTRL_LABEL)
print("KD   =", KD_LABEL)

# =============================================================================
# 3. SELECT MATCHED MESODERM CELLS
# =============================================================================

obs = adata.obs.copy()

# normalize time labels -> "42" / "72"
obs["_time_clean"] = (
    obs[TIME_COL]
    .astype(str)
    .str.extract(r"(42|72)", expand=False)
)

state = obs[STATE_COL].astype(str)
cond  = obs[COND_COL].astype(str)

state_time_ok = np.array([
    (s, t) in KEEP_STATE_TIME
    for s, t in zip(state, obs["_time_clean"])
])

mask = (
    cond.isin([CTRL_LABEL, KD_LABEL]).to_numpy()
    & state_time_ok
)

# Apply the same Stan barcode QC to Ctrl, if that variable exists
if "_bc" in obs.columns and "STAN_BC" in globals():
    good_ctrl_bc = obs["_bc"].isin(STAN_BC).to_numpy()

    mask &= (
        (cond.to_numpy() != CTRL_LABEL)
        | good_ctrl_bc
    )

cell_pos = np.flatnonzero(mask)

obs_sub = obs.iloc[cell_pos][
    [COND_COL, SITE_COL, STATE_COL, "_time_clean"]
].copy()

obs_sub = obs_sub.reset_index(drop=True)

print("\nCells entering preservation analysis:")
print(
    obs_sub.groupby(
        [COND_COL, SITE_COL, STATE_COL, "_time_clean"]
    ).size()
)

# =============================================================================
# 4. GET RAW COUNTS
#
# We only pull the 218 module genes for expression,
# BUT library size comes from ALL genes.
# =============================================================================

adata_sub = adata[cell_pos]

X_module = adata_sub[:, module_genes].layers[COUNTS_LAYER]

# total raw library size across ALL genes
X_all = adata_sub.layers[COUNTS_LAYER]
library_size = np.asarray(X_all.sum(axis=1)).ravel()

# =============================================================================
# 5. MAKE PSEUDOCELLS
#
# Random grouping is intentional here:
# pseudocell membership is independent of the module expression we're testing.
#
# Within:
#   condition x site x time x state
# =============================================================================

pc_expr = []
pc_meta = []

group_cols = [
    COND_COL,
    SITE_COL,
    STATE_COL,
    "_time_clean",
]

pc_number = 0

for keys, idx in obs_sub.groupby(group_cols).groups.items():

    idx = np.asarray(list(idx), dtype=int)
    n = len(idx)

    if n < MIN_CELLS:
        continue

    idx = rng.permutation(idx)

    # approximately 25 cells / pseudocell
    n_groups = max(1, n // TARGET_CELLS)

    remainder = n - n_groups * TARGET_CELLS

    # make another pseudocell if enough cells remain
    if remainder >= MIN_CELLS:
        n_groups += 1

    chunks = np.array_split(idx, n_groups)

    for chunk in chunks:

        if len(chunk) < MIN_CELLS:
            continue

        # summed module counts
        counts = np.asarray(
            X_module[chunk].sum(axis=0)
        ).ravel()

        # summed TOTAL library size
        lib = library_size[chunk].sum()

        # CP10K -> log1p
        expr = np.log1p(
            counts / max(lib, 1) * 1e4
        )

        condition, site, state_name, time = keys

        pc_expr.append(expr)

        pc_meta.append({
            "pseudocell": f"PC{pc_number:04d}",
            "condition": condition,
            "site": site,
            "state": state_name,
            "time": time,
            "n_cells": len(chunk),
            "stratum": f"{site}|{state_name}|{time}",
        })

        pc_number += 1

pc_expr = pd.DataFrame(
    pc_expr,
    columns=module_genes
)

pc_meta = pd.DataFrame(pc_meta)

print("\nInitial pseudocells:")
print(
    pc_meta.groupby(["condition", "stratum"]).size()
)

# =============================================================================
# 6. BALANCE CTRL AND KD WITHIN site x state x time
#
# If Chicago pPS42 has 6 Ctrl pseudocells and 4 KD,
# retain 4 + 4.
# =============================================================================

keep_rows = []

for stratum, d in pc_meta.groupby("stratum"):

    ctrl_ids = d.index[
        d["condition"] == CTRL_LABEL
    ].to_numpy()

    kd_ids = d.index[
        d["condition"] == KD_LABEL
    ].to_numpy()

    n = min(len(ctrl_ids), len(kd_ids))

    if n < MIN_PSEUDOCELLS_PER_COND:
        continue

    keep_rows.extend(
        rng.choice(ctrl_ids, size=n, replace=False)
    )

    keep_rows.extend(
        rng.choice(kd_ids, size=n, replace=False)
    )

keep_rows = np.asarray(keep_rows)

pc_meta = pc_meta.loc[keep_rows].copy()
pc_expr = pc_expr.loc[keep_rows].copy()

# reset together
order = (
    pc_meta
    .sort_values(["stratum", "condition", "pseudocell"])
    .index
)

pc_meta = pc_meta.loc[order].reset_index(drop=True)
pc_expr = pc_expr.loc[order].reset_index(drop=True)

print("\nBALANCED pseudocells:")
print(
    pc_meta.groupby(["condition", "stratum"]).size()
)

print(
    "\nTotal:",
    pc_meta["condition"].value_counts().to_dict()
)

# save pseudocell matrix so we NEVER have to rebuild it
pc_out = pd.concat(
    [pc_meta, pc_expr],
    axis=1
)

pc_out.to_csv(
    OUT / "APLNR_module_balanced_pseudocells.csv",
    index=False
)

# =============================================================================
# 7. NETWORK METRICS
# =============================================================================

def get_condition_matrix(condition):

    idx = pc_meta.index[
        pc_meta["condition"] == condition
    ].to_numpy()

    X = pc_expr.iloc[idx].to_numpy(dtype=float)
    meta = pc_meta.iloc[idx].reset_index(drop=True)

    return X, meta


def center_within_stratum(X, meta):
    """
    Conservative second version:
    remove the average state/site/time expression before calculating
    correlations.

    This asks whether genes remain coordinated BEYOND simple composition.
    """
    Xr = X.copy()

    strata = meta["stratum"].to_numpy()

    for s in np.unique(strata):
        idx = np.where(strata == s)[0]

        if len(idx) >= 2:
            Xr[idx, :] -= Xr[idx, :].mean(axis=0)

    return Xr


def network_metrics(X_ctrl, X_kd, genes, mode):

    # remove genes with zero variance in either condition
    keep = (
        (np.nanstd(X_ctrl, axis=0) > 1e-8)
        & (np.nanstd(X_kd, axis=0) > 1e-8)
    )

    Xc = X_ctrl[:, keep]
    Xk = X_kd[:, keep]

    genes_use = np.asarray(genes)[keep]

    # gene-gene correlation networks
    Rc = np.corrcoef(Xc, rowvar=False)
    Rk = np.corrcoef(Xk, rowvar=False)

    n_gene = len(genes_use)
    tri = np.triu_indices(n_gene, 1)

    ctrl_edges = Rc[tri]
    kd_edges   = Rk[tri]

    valid_edges = (
        np.isfinite(ctrl_edges)
        & np.isfinite(kd_edges)
    )

    ctrl_edges_v = ctrl_edges[valid_edges]
    kd_edges_v   = kd_edges[valid_edges]

    # ---------------------------------------------------------------
    # Module eigengene + kME in each condition
    # ---------------------------------------------------------------
    def calc_kme(X):

        Z = StandardScaler().fit_transform(X)

        pca = PCA(n_components=1)
        score = pca.fit_transform(Z).ravel()

        kme = np.array([
            np.corrcoef(Z[:, j], score)[0, 1]
            for j in range(Z.shape[1])
        ])

        # PC sign is arbitrary; orient so module genes are positive
        if np.nanmean(kme) < 0:
            score *= -1
            kme *= -1

        return (
            kme,
            pca.explained_variance_ratio_[0]
        )

    kme_ctrl, pc1_ctrl = calc_kme(Xc)
    kme_kd,   pc1_kd   = calc_kme(Xk)

    # ---------------------------------------------------------------
    # average connection of each gene to rest of module
    # ---------------------------------------------------------------
    np.fill_diagonal(Rc, np.nan)
    np.fill_diagonal(Rk, np.nan)

    mean_r_ctrl = np.nanmean(Rc, axis=1)
    mean_r_kd   = np.nanmean(Rk, axis=1)

    # ---------------------------------------------------------------
    # summary preservation metrics
    # ---------------------------------------------------------------
    pairwise_pearson = pearsonr(
        ctrl_edges_v,
        kd_edges_v
    ).statistic

    pairwise_spearman = spearmanr(
        ctrl_edges_v,
        kd_edges_v
    ).statistic

    kme_preservation = pearsonr(
        kme_ctrl,
        kme_kd
    ).statistic

    summary = {
        "mode": mode,
        "n_genes": n_gene,
        "n_ctrl_pseudocells": Xc.shape[0],
        "n_kd_pseudocells": Xk.shape[0],

        # overall module coherence
        "mean_pairwise_r_ctrl":
            np.nanmean(ctrl_edges_v),

        "mean_pairwise_r_kd":
            np.nanmean(kd_edges_v),

        "delta_mean_pairwise_r":
            np.nanmean(kd_edges_v)
            - np.nanmean(ctrl_edges_v),

        # does the entire network retain the same structure?
        "pairwise_network_Pearson_r":
            pairwise_pearson,

        "pairwise_network_Spearman_r":
            pairwise_spearman,

        # are the same genes still central?
        "gene_kME_preservation_r":
            kme_preservation,

        # how much of module variation is captured by one common program?
        "PC1_variance_ctrl":
            pc1_ctrl,

        "PC1_variance_kd":
            pc1_kd,

        "delta_PC1_variance":
            pc1_kd - pc1_ctrl,

        "fraction_edges_lower_in_KD":
            np.mean(kd_edges_v < ctrl_edges_v),

        "median_abs_edge_change":
            np.nanmedian(
                np.abs(kd_edges_v - ctrl_edges_v)
            ),
    }

    # ---------------------------------------------------------------
    # gene-level table
    # ---------------------------------------------------------------
    gene_df = pd.DataFrame({
        "mode": mode,
        "gene": genes_use,

        "kME_ctrl": kme_ctrl,
        "kME_kd": kme_kd,
        "delta_kME": kme_kd - kme_ctrl,

        "mean_module_r_ctrl": mean_r_ctrl,
        "mean_module_r_kd": mean_r_kd,
        "delta_mean_module_r":
            mean_r_kd - mean_r_ctrl,
    })

    # attach original Ctrl WGCNA numbers
    ctrl_info = (
        module
        .drop_duplicates("gene")
        .set_index("gene")
    )

    if "kME" in ctrl_info.columns:
        gene_df["original_Ctrl_WGCNA_kME"] = (
            gene_df["gene"].map(ctrl_info["kME"])
        )

    if "kME_rank" in ctrl_info.columns:
        gene_df["original_Ctrl_WGCNA_rank"] = (
            gene_df["gene"].map(ctrl_info["kME_rank"])
        )

    # ---------------------------------------------------------------
    # edge-level table
    # ---------------------------------------------------------------
    g1 = genes_use[tri[0]][valid_edges]
    g2 = genes_use[tri[1]][valid_edges]

    edge_df = pd.DataFrame({
        "mode": mode,
        "gene1": g1,
        "gene2": g2,
        "r_ctrl": ctrl_edges_v,
        "r_kd": kd_edges_v,
        "delta_r": kd_edges_v - ctrl_edges_v,
    })

    edge_df["abs_delta_r"] = np.abs(
        edge_df["delta_r"]
    )

    edge_df = edge_df.sort_values(
        "abs_delta_r",
        ascending=False
    )

    return summary, gene_df, edge_df


# =============================================================================
# 8. RUN TWO VERSIONS
#
# A) balanced:
#    same site/state/time composition in Ctrl and KD
#
# B) within-stratum centered:
#    additionally removes mean site/state/time effects
# =============================================================================

Xc, mc = get_condition_matrix(CTRL_LABEL)
Xk, mk = get_condition_matrix(KD_LABEL)

results_summary = []
results_gene = []
results_edge = []

# A. balanced network
s, g, e = network_metrics(
    Xc,
    Xk,
    module_genes,
    mode="balanced"
)

results_summary.append(s)
results_gene.append(g)
results_edge.append(e)

# B. remove site/time/state mean effects
Xc_resid = center_within_stratum(Xc, mc)
Xk_resid = center_within_stratum(Xk, mk)

s, g, e = network_metrics(
    Xc_resid,
    Xk_resid,
    module_genes,
    mode="within_stratum_centered"
)

results_summary.append(s)
results_gene.append(g)
results_edge.append(e)

summary_df = pd.DataFrame(results_summary)
gene_df = pd.concat(results_gene, ignore_index=True)
edge_df = pd.concat(results_edge, ignore_index=True)

# =============================================================================
# 9. SAVE
# =============================================================================

summary_df.to_csv(
    OUT / "APLNR_module_preservation_summary.csv",
    index=False
)

gene_df.to_csv(
    OUT / "APLNR_module_gene_preservation.csv",
    index=False
)

edge_df.to_csv(
    OUT / "APLNR_module_edge_changes.csv",
    index=False
)

module[["gene", "kME", "kME_rank"]].to_csv(
    OUT / "APLNR_module_genes_used.csv",
    index=False
)

print("\n" + "="*80)
print("MODULE PRESERVATION SUMMARY")
print("="*80)

display(summary_df.round(3))

print("\nMost decreased connectivity / kME in KD:")
display(
    gene_df[
        gene_df["mode"] == "balanced"
    ]
    .sort_values("delta_kME")
    .head(20)
    .round(3)
)

print("\nSaved to:")
print(OUT)

for p in sorted(OUT.glob("*.csv")):
    print("  ", p.name)

In [ ]:
# =============================================================================
# PREPARE FORMAL WGCNA MODULE-PRESERVATION INPUT
# Ctrl reference -> APLNR KD test
#
# Uses ALL genes from the original Ctrl mesoderm WGCNA,
# not only the 218-gene black module.
# =============================================================================

from pathlib import Path
import numpy as np
import pandas as pd
import PyWGCNA

SEED = 17
rng = np.random.default_rng(SEED)

# ------------------------------------------------------------------
# paths
# ------------------------------------------------------------------
WGCNA_FILE = (
    TAB / "WGCNA_Ctrl_mesoderm_only" / "Ctrl_mesoderm_only.p"
)

OUT = (
    TAB / "WGCNA_Ctrl_mesoderm_only" /
    "KD_formal_module_preservation"
)
OUT.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# load original CTRL WGCNA
# ------------------------------------------------------------------
wgcna = PyWGCNA.readWGCNA(str(WGCNA_FILE))

gene_info = wgcna.datExpr.var.copy()

if "moduleColors" not in gene_info.columns:
    raise ValueError("moduleColors not found in saved WGCNA object")

gene_info = (
    gene_info[["moduleColors"]]
    .rename(columns={"moduleColors": "module"})
    .copy()
)

gene_info["gene"] = gene_info.index.astype(str)

# only genes also present in adata
gene_info = gene_info[
    gene_info["gene"].isin(adata.var_names)
].copy()

WGCNA_GENES = gene_info["gene"].tolist()

print("WGCNA genes found in adata:", len(WGCNA_GENES))
print("\nModules:")
print(gene_info["module"].value_counts())

print("\nBlack module size:",
      (gene_info["module"] == "black").sum())

# save Ctrl reference module assignments
gene_info[["gene", "module"]].to_csv(
    OUT / "Ctrl_WGCNA_gene_module_colors.csv",
    index=False
)

# ------------------------------------------------------------------
# columns
# ------------------------------------------------------------------
COND_COL  = "condition"
SITE_COL  = "site"
TIME_COL  = "timepoint_true"
STATE_COL = "predicted_leiden_sub_joint"
COUNTS_LAYER = "counts"

KEEP_STATE_TIME = {
    ("pPS-LC", "42"),
    ("aPS-LC", "42"),
    ("eNasM",  "72"),
    ("lNasM",  "72"),
}

TARGET_CELLS = 25
MIN_CELLS = 10
MIN_PCS_PER_COND = 2

# ------------------------------------------------------------------
# conditions
# ------------------------------------------------------------------
conditions = sorted(
    adata.obs[COND_COL].astype(str).unique()
)

CTRL_LABEL = next(
    x for x in conditions
    if x.lower() in {"ctrl", "control"}
)

kd_candidates = [
    x for x in conditions
    if x != CTRL_LABEL and
    any(k in x.lower()
        for k in ["dox", "kd", "shaplnr", "aplnr"])
]

if len(kd_candidates) == 0 and len(conditions) == 2:
    kd_candidates = [
        x for x in conditions if x != CTRL_LABEL
    ]

if len(kd_candidates) != 1:
    raise ValueError(
        f"KD identification failed: {kd_candidates}"
    )

KD_LABEL = kd_candidates[0]

print("\nCTRL:", CTRL_LABEL)
print("KD:", KD_LABEL)

# ------------------------------------------------------------------
# select matched mesoderm cells
# ------------------------------------------------------------------
obs = adata.obs.copy()

obs["_time_clean"] = (
    obs[TIME_COL]
    .astype(str)
    .str.extract(r"(42|72)", expand=False)
)

state_time_ok = np.array([
    (s, t) in KEEP_STATE_TIME
    for s, t in zip(
        obs[STATE_COL].astype(str),
        obs["_time_clean"]
    )
])

cond = obs[COND_COL].astype(str)

mask = (
    cond.isin([CTRL_LABEL, KD_LABEL]).to_numpy()
    & state_time_ok
)

# same Ctrl Stan filter as before
if "_bc" in obs.columns and "STAN_BC" in globals():
    mask &= (
        (cond.to_numpy() != CTRL_LABEL)
        | obs["_bc"].isin(STAN_BC).to_numpy()
    )

cell_pos = np.flatnonzero(mask)

adata_sub = adata[cell_pos]

meta = obs.iloc[cell_pos][
    [COND_COL, SITE_COL, STATE_COL, "_time_clean"]
].copy().reset_index(drop=True)

# ------------------------------------------------------------------
# raw counts:
# all original WGCNA genes for expression
# all genes for library size
# ------------------------------------------------------------------
X_wgcna = adata_sub[:, WGCNA_GENES].layers[COUNTS_LAYER]
X_all = adata_sub.layers[COUNTS_LAYER]

library_size = np.asarray(
    X_all.sum(axis=1)
).ravel()

# ------------------------------------------------------------------
# pseudocells
# random aggregation within:
# condition x site x state x time
# ------------------------------------------------------------------
pc_expr = []
pc_meta = []

group_cols = [
    COND_COL,
    SITE_COL,
    STATE_COL,
    "_time_clean"
]

pc_number = 0

for keys, idx in meta.groupby(group_cols).groups.items():

    idx = np.asarray(list(idx), dtype=int)

    if len(idx) < MIN_CELLS:
        continue

    idx = rng.permutation(idx)

    n_groups = max(
        1,
        len(idx) // TARGET_CELLS
    )

    remainder = len(idx) - n_groups * TARGET_CELLS

    if remainder >= MIN_CELLS:
        n_groups += 1

    chunks = np.array_split(idx, n_groups)

    for chunk in chunks:

        if len(chunk) < MIN_CELLS:
            continue

        counts = np.asarray(
            X_wgcna[chunk].sum(axis=0)
        ).ravel()

        lib = library_size[chunk].sum()

        # same CP10K + log1p representation
        expr = np.log1p(
            counts / max(lib, 1) * 1e4
        )

        condition, site, state, time = keys

        pc_expr.append(expr)

        pc_meta.append({
            "pseudocell": f"PC{pc_number:04d}",
            "condition": condition,
            "site": site,
            "state": state,
            "time": time,
            "n_cells": len(chunk),
            "stratum": f"{site}|{state}|{time}",
        })

        pc_number += 1

pc_expr = pd.DataFrame(
    pc_expr,
    columns=WGCNA_GENES
)

pc_meta = pd.DataFrame(pc_meta)

# ------------------------------------------------------------------
# balance Ctrl/KD within site x state x time
# ------------------------------------------------------------------
keep = []

for stratum, d in pc_meta.groupby("stratum"):

    c = d.index[
        d["condition"] == CTRL_LABEL
    ].to_numpy()

    k = d.index[
        d["condition"] == KD_LABEL
    ].to_numpy()

    n = min(len(c), len(k))

    if n < MIN_PCS_PER_COND:
        continue

    keep.extend(
        rng.choice(c, n, replace=False)
    )
    keep.extend(
        rng.choice(k, n, replace=False)
    )

keep = np.asarray(keep)

pc_meta = pc_meta.loc[keep].copy()
pc_expr = pc_expr.loc[keep].copy()

order = (
    pc_meta
    .sort_values(
        ["stratum", "condition", "pseudocell"]
    )
    .index
)

pc_meta = (
    pc_meta.loc[order]
    .reset_index(drop=True)
)

pc_expr = (
    pc_expr.loc[order]
    .reset_index(drop=True)
)

print("\nBalanced pseudocells:")
print(
    pc_meta.groupby(
        ["condition", "stratum"]
    ).size()
)

print("\nTotals:")
print(pc_meta["condition"].value_counts())

# ------------------------------------------------------------------
# export separate expression matrices
#
# rows = pseudocells
# columns = genes
# ------------------------------------------------------------------
ctrl_idx = (
    pc_meta["condition"] == CTRL_LABEL
)

kd_idx = (
    pc_meta["condition"] == KD_LABEL
)

ctrl_expr = pc_expr.loc[ctrl_idx].copy()
kd_expr   = pc_expr.loc[kd_idx].copy()

ctrl_expr.index = (
    pc_meta.loc[ctrl_idx, "pseudocell"].values
)

kd_expr.index = (
    pc_meta.loc[kd_idx, "pseudocell"].values
)

ctrl_expr.to_csv(
    OUT / "modulePreservation_Ctrl_expression.csv"
)

kd_expr.to_csv(
    OUT / "modulePreservation_KD_expression.csv"
)

pc_meta.to_csv(
    OUT / "modulePreservation_pseudocell_metadata.csv",
    index=False
)

print("\nSaved:")
for f in sorted(OUT.glob("*.csv")):
    print(" ", f.name)

In [ ]:
import pandas as pd
import numpy as np
import gseapy as gp

# ----------------------------
# load genes
# ----------------------------
module_df = pd.read_csv(
    ROOT / "06_tables/FIGURE2_clean/WGCNA_Ctrl_mesoderm_only/APLNR_black_all_genes.csv"
)

hub_df = pd.read_csv(
    ROOT / "06_tables/FIGURE2_clean/WGCNA_Ctrl_mesoderm_only/APLNR_black_all_hub_ranks.csv"
)
module_genes = module_df.iloc[:, 0].dropna().astype(str).tolist()

# make sure hub table has gene + kME
hub_df = hub_df.copy()
hub_df["gene"] = hub_df["gene"].astype(str)
gene2kme = dict(zip(hub_df["gene"], hub_df["kME"]))

print(f"{len(module_genes)} module genes")

# ----------------------------
# enrichment
# ----------------------------
enr = gp.enrichr(
    gene_list=module_genes,
    gene_sets=["GO_Biological_Process_2023"],
    organism="human",
    outdir=None
)

res = enr.results.copy()
res = res.sort_values("Adjusted P-value")

display(res.head(20)[["Term", "Adjusted P-value", "Overlap", "Genes"]])


In [ ]:
interesting = res[
    res["Term"].str.contains(
        r"mesoderm|axis|morphogen|migration|motility|"
        r"extracellular|matrix|adhesion|Wnt|BMP|"
        r"epithelial.*mesenchymal|spindle",
        case=False,
        regex=True,
        na=False
    )
].copy()

display(
    interesting[
        ["Term", "Adjusted P-value", "Overlap", "Genes"]
    ].head(40)
)



In [ ]:


from pycirclize import Circos
from pycirclize.parser import Matrix
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd


# ============================================================================
# 1. SELECT ENRICHED TERMS
# ============================================================================

TERM_PATTERNS = [
    "Anterior/Posterior Axis Specification",
    "Mesoderm Morphogenesis",
    "Epithelial To Mesenchymal Transition",
    "Regulation Of Wnt Signaling Pathway",
]

selected_rows = []

for wanted in TERM_PATTERNS:

    # remove "(GO:xxxxxxx)" before matching
    clean_names = (
        res["Term"]
        .str.replace(r"\s+\(GO:\d+\)$", "", regex=True)
        .str.strip()
    )

    hit = res[clean_names.str.lower() == wanted.lower()]

    if len(hit):
        selected_rows.append(hit.iloc[0])
    else:
        print("Not found:", wanted)

terms = pd.DataFrame(selected_rows).drop_duplicates("Term")

display(
    terms[["Term", "Adjusted P-value", "Overlap", "Genes"]]
)


# ============================================================================
# 2. SHORTEN TERM NAMES FOR THE FIGURE
# ============================================================================

def clean_term(x):
    x = str(x).split(" (GO:")[0]

    replacements = {
        "Anterior/Posterior Axis Specification":
            "A/P axis\nspecification",

        "Mesoderm Morphogenesis":
            "Mesoderm\nmorphogenesis",

        "Epithelial To Mesenchymal Transition":
            "EMT",

        "Regulation Of Wnt Signaling Pathway":
            "Regulation of\nWNT signaling",
    }

    return replacements.get(x, x)

terms["short_term"] = terms["Term"].map(clean_term)

# ============================================================================
# 3. HUB / kME TABLE
# ============================================================================

hub_df = hub_df.copy()
hub_df["gene"] = hub_df["gene"].astype(str)

gene2kme = dict(
    zip(
        hub_df["gene"],
        hub_df["kME"]
    )
)


# ============================================================================
# 4. TERM -> GENE LINKS
# ============================================================================

links = []

for _, row in terms.iterrows():

    # Enrichr separates genes with semicolons
    genes = [
        g.strip()
        for g in str(row["Genes"]).split(";")
    ]

    for g in genes:

        # only genes with WGCNA kME available
        if g in gene2kme:
            links.append({
                "term": row["short_term"],
                "gene": g,
            })

links = pd.DataFrame(links).drop_duplicates()

print("Initial genes represented:", links["gene"].nunique())
display(links.head())


# ============================================================================
# 5. KEEP ALL GENES BELONGING TO THE SELECTED GO TERMS
# ============================================================================

keep_genes = sorted(links["gene"].unique())

print("Genes shown:", len(keep_genes))
print(keep_genes)


# ============================================================================
# 6. BUILD FROM-TO TABLE
# ============================================================================

# every membership contributes one ribbon
fromto = pd.DataFrame({
    "from": links["gene"],
    "to": links["term"],
    "value": 1,
})

matrix = Matrix.parse_fromto_table(fromto)


# ============================================================================
# 7. COLORS
#
# Terms = distinct biological colors
# Genes = kME gradient
# ============================================================================

term_colors = {
    "A/P axis\nspecification": "#C66B62",
    "Mesoderm\nmorphogenesis": "#D8B15A",
    "EMT": "#7FA6A2",
    "Regulation of\nWNT signaling": "#9A7B9E",
}

shown_genes = sorted(
    links["gene"].unique()
)

vals = np.array([
    gene2kme[g]
    for g in shown_genes
])

vmin = vals.min()
vmax = vals.max()

gene_cmap = mcolors.LinearSegmentedColormap.from_list(
    "kME",
    ["#E8E1D3", "#B66A83"]   # warm cream -> dusty rose
)

def kme_color(g):
    z = (
        (gene2kme[g] - vmin)
        / (vmax - vmin + 1e-12)
    )
    return mcolors.to_hex(
        gene_cmap(z)
    )

sector_colors = dict(term_colors)

for g in shown_genes:
    sector_colors[g] = kme_color(g)


# ============================================================================
# 8. DRAW CHORD PLOT
# ============================================================================

circos = Circos.chord_diagram(
    matrix,
    start=-265,
    end=95,
    space=2.0,
    cmap=sector_colors,
    label_kws=dict(
        size=28,
        r=102,
        orientation="vertical"
    ),
    link_kws=dict(
        alpha=0.48,
        lw=0.25,
        ec="white"
    ),
)

fig = circos.plotfig()

fig.set_size_inches(11, 11)

# fig.suptitle(
#     "Functional enrichment of the APLNR-containing mesoderm module",
#     fontsize=16,
#     y=0.98
# )
plt.show()


# ============================================================================
# 9. SAVE
# ============================================================================

fig.savefig(
    FIG / "APLNR_mesoderm_module_GO_chord.svg",
    bbox_inches="tight"
)

fig.savefig(
    FIG / "APLNR_mesoderm_module_GO_chord.png",
    dpi=300,
    bbox_inches="tight"
)

In [ ]:
# ============================================================
# GENERATE VERIFIED TOP-25 NODES + TOP-60 TOM EDGES
# This creates new derived visualization files.
# ============================================================

import numpy as np
import pandas as pd
from PyWGCNA.utils import readWGCNA


# Load saved mesoderm-lineage WGCNA
meso = readWGCNA(
    str(
        TAB
        / "WGCNA_Ctrl_mesoderm_only"
        / "Ctrl_mesoderm_only.p"
    )
)

N_NODES = 25
N_EDGES = 60
KME_COL = "kMEblack"


# ------------------------------------------------------------
# 1. TRUE TOP 25 GENES BY MODULE MEMBERSHIP
# ------------------------------------------------------------

nodes = (
    meso.signedKME[KME_COL]
    .dropna()
    .sort_values(ascending=False)
    .head(N_NODES)
    .rename("kME")
    .rename_axis("gene")
    .reset_index()
)

nodes["kME_rank"] = np.arange(
    1,
    len(nodes) + 1
)

assert "APLNR" in set(nodes["gene"])

NODE_FILE = (
    TAB
    / "APLNR_mesoderm_top25_kME_nodes_VERIFIED.csv"
)

nodes.to_csv(
    NODE_FILE,
    index=False
)


# ------------------------------------------------------------
# 2. LABEL FULL TOM MATRIX
# ------------------------------------------------------------

genes = pd.Index(
    meso.datExpr.var_names.astype(str)
)

if isinstance(meso.TOM, pd.DataFrame):

    tom_df = meso.TOM.copy()
    tom_df.index = tom_df.index.astype(str)
    tom_df.columns = tom_df.columns.astype(str)

else:

    tom_array = np.asarray(meso.TOM)

    assert tom_array.shape == (
        len(genes),
        len(genes)
    )

    tom_df = pd.DataFrame(
        tom_array,
        index=genes,
        columns=genes
    )


# ------------------------------------------------------------
# 3. TRUE TOP 60 EDGES AMONG THE TOP 25 GENES
# ------------------------------------------------------------

top_genes = nodes["gene"].tolist()

tom_sub = tom_df.loc[
    top_genes,
    top_genes
]

i, j = np.triu_indices(
    len(top_genes),
    k=1
)

edges = pd.DataFrame({
    "gene1": np.asarray(top_genes)[i],
    "gene2": np.asarray(top_genes)[j],
    "TOM": tom_sub.to_numpy()[i, j],
})

edges = (
    edges.sort_values("TOM", ascending=False)
    .head(N_EDGES)
    .reset_index(drop=True)
)

EDGE_FILE = (
    TAB
    / "APLNR_mesoderm_top25_top60_TOM_edges_VERIFIED.csv"
)

edges.to_csv(
    EDGE_FILE,
    index=False
)


# ------------------------------------------------------------
# 4. CONFIRM SAVED OUTPUT
# ------------------------------------------------------------

print("Created new derived files:")
print("Nodes:", NODE_FILE)
print("Edges:", EDGE_FILE)

print(f"\nNodes saved: {len(nodes)}")
print(f"Edges saved: {len(edges)}")
print(
    "APLNR kME rank:",
    nodes.loc[
        nodes["gene"] == "APLNR",
        "kME_rank"
    ].iloc[0]
)

display(nodes.head())
display(edges.head())

In [ ]:
# ============================================================
# OPTION 1 — INTRAMODULAR CO-EXPRESSION NETWORK
#
# Nodes: top genes by kME
# Node size: kME
# Edges: strongest TOM relationships
# Edge width: TOM
# Layout: force-directed from network structure
# APLNR and its edges are highlighted
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from matplotlib.lines import Line2D
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize


# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------

NODE_FILE = (
    TAB
    / "APLNR_mesoderm_top25_kME_nodes_VERIFIED.csv"
)

EDGE_FILE = (
    TAB
    / "APLNR_mesoderm_top25_top60_TOM_edges_VERIFIED.csv"
)

N_NODES = 25
N_EDGES = 60
RANDOM_SEED = 2026

APLNR_COLOR = "#C9574F"
NODE_CMAP = "GnBu"

OUT_STEM = "APLNR_intramodular_network"


# ------------------------------------------------------------
# 1. LOAD AND VALIDATE
# ------------------------------------------------------------

nodes = pd.read_csv(NODE_FILE)
edges = pd.read_csv(EDGE_FILE)

assert {"gene", "kME"}.issubset(nodes.columns)
assert {"gene1", "gene2", "TOM"}.issubset(edges.columns)

nodes["gene"] = nodes["gene"].astype(str)
edges["gene1"] = edges["gene1"].astype(str)
edges["gene2"] = edges["gene2"].astype(str)

nodes["kME"] = pd.to_numeric(nodes["kME"], errors="coerce")
edges["TOM"] = pd.to_numeric(edges["TOM"], errors="coerce")

nodes = nodes.dropna(subset=["gene", "kME"])
edges = edges.dropna(subset=["gene1", "gene2", "TOM"])


# ------------------------------------------------------------
# 2. SELECT NODES AND EDGES OBJECTIVELY
# ------------------------------------------------------------

# Highest module-membership genes
nodes_plot = (
    nodes.sort_values("kME", ascending=False)
    .drop_duplicates("gene")
    .head(N_NODES)
    .copy()
)

selected_genes = set(nodes_plot["gene"])

# Ensure APLNR remains included
if "APLNR" not in selected_genes:

    aplnr_row = nodes[nodes["gene"] == "APLNR"]

    if aplnr_row.empty:
        raise ValueError("APLNR not found in node table.")

    # Replace the lowest-kME selected node with APLNR
    nodes_plot = pd.concat(
        [
            nodes_plot.iloc[:-1],
            aplnr_row.iloc[[0]],
        ],
        ignore_index=True
    )

    selected_genes = set(nodes_plot["gene"])


# Keep edges whose two endpoints are selected
edges_plot = edges[
    edges["gene1"].isin(selected_genes)
    & edges["gene2"].isin(selected_genes)
].copy()

edges_plot = (
    edges_plot.sort_values("TOM", ascending=False)
    .drop_duplicates(["gene1", "gene2"])
    .head(N_EDGES)
)

print(f"Displayed nodes: {len(nodes_plot)}")
print(f"Displayed edges: {len(edges_plot)}")
print("APLNR included:", "APLNR" in selected_genes)


# ------------------------------------------------------------
# 3. BUILD NETWORK
# ------------------------------------------------------------

G = nx.Graph()

for _, row in nodes_plot.iterrows():

    G.add_node(
        row["gene"],
        kME=float(row["kME"])
    )

for _, row in edges_plot.iterrows():

    G.add_edge(
        row["gene1"],
        row["gene2"],
        TOM=float(row["TOM"])
    )


# Remove nodes that have no displayed edges, except APLNR
isolated = [
    gene
    for gene in nx.isolates(G)
    if gene != "APLNR"
]

G.remove_nodes_from(isolated)

print("Isolated nodes removed:", isolated)


# ------------------------------------------------------------
# 4. FORCE-DIRECTED LAYOUT
# ------------------------------------------------------------

# Higher TOM edges pull nodes together more strongly.
pos = nx.spring_layout(
    G,
    weight="TOM",
    seed=RANDOM_SEED,
    k=1.15 / np.sqrt(max(G.number_of_nodes(), 1)),
    iterations=500,
    scale=1.0
)


# ------------------------------------------------------------
# 5. SCALE NODES BY kME
# ------------------------------------------------------------

kme = np.array([
    G.nodes[g]["kME"]
    for g in G.nodes
])

kme_min = kme.min()
kme_max = kme.max()


def scale_value(value, low, high, out_low, out_high):

    if high == low:
        return (out_low + out_high) / 2

    return (
        out_low
        + (value - low)
        / (high - low)
        * (out_high - out_low)
    )


node_sizes = []

for gene in G.nodes:

    size = scale_value(
        G.nodes[gene]["kME"],
        kme_min,
        kme_max,
        500,
        1450
    )

    if gene == "APLNR":
        size = max(size, 1650)

    node_sizes.append(size)


# Color non-APLNR nodes by kME
cmap = plt.get_cmap(NODE_CMAP)
norm = Normalize(vmin=kme_min, vmax=kme_max)

node_colors = [
    APLNR_COLOR
    if gene == "APLNR"
    else cmap(norm(G.nodes[gene]["kME"]))
    for gene in G.nodes
]


# ------------------------------------------------------------
# 6. SCALE EDGES BY TOM
# ------------------------------------------------------------

tom_values = np.array([
    G.edges[e]["TOM"]
    for e in G.edges
])

tom_min = tom_values.min()
tom_max = tom_values.max()


def tom_width(value):

    return scale_value(
        value,
        tom_min,
        tom_max,
        0.6,
        4.2
    )


apl_edges = []
other_edges = []

apl_widths = []
other_widths = []

for u, v, data in G.edges(data=True):

    if "APLNR" in (u, v):
        apl_edges.append((u, v))
        apl_widths.append(tom_width(data["TOM"]))
    else:
        other_edges.append((u, v))
        other_widths.append(tom_width(data["TOM"]))


# ------------------------------------------------------------
# 7. DRAW
# ------------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(9.5, 8.2)
)

# Background module edges
nx.draw_networkx_edges(
    G,
    pos,
    edgelist=other_edges,
    width=other_widths,
    edge_color="#BEBEBE",
    alpha=0.38,
    ax=ax
)

# APLNR edges
nx.draw_networkx_edges(
    G,
    pos,
    edgelist=apl_edges,
    width=apl_widths,
    edge_color=APLNR_COLOR,
    alpha=0.85,
    ax=ax
)

# Nodes
nx.draw_networkx_nodes(
    G,
    pos,
    node_size=node_sizes,
    node_color=node_colors,
    edgecolors="#303030",
    linewidths=0.8,
    ax=ax
)

# Labels
labels = {
    gene: gene
    for gene in G.nodes
}

nx.draw_networkx_labels(
    G,
    pos,
    labels=labels,
    font_size=10.5,
    font_family="sans-serif",
    font_weight="normal",
    ax=ax
)

# Redraw APLNR label in white
x_apl, y_apl = pos["APLNR"]

ax.text(
    x_apl,
    y_apl,
    "APLNR",
    ha="center",
    va="center",
    fontsize=11.5,
    fontweight="bold",
    color="white",
    zorder=10
)

ax.set_axis_off()


# ------------------------------------------------------------
# 8. LEGENDS
# ------------------------------------------------------------

# kME colorbar
sm = ScalarMappable(
    norm=norm,
    cmap=cmap
)

sm.set_array([])

cbar = fig.colorbar(
    sm,
    ax=ax,
    fraction=0.035,
    pad=0.02
)

cbar.set_label(
    "Module membership (kME)",
    fontsize=11
)

cbar.ax.tick_params(labelsize=9)


# Edge-type legend
edge_legend = [
    Line2D(
        [0],
        [0],
        color=APLNR_COLOR,
        lw=2.5,
        label="APLNR edge"
    ),
    Line2D(
        [0],
        [0],
        color="#BEBEBE",
        lw=2.5,
        label="Other module edge"
    ),
]

ax.legend(
    handles=edge_legend,
    frameon=False,
    fontsize=10,
    loc="upper left"
)


# ------------------------------------------------------------
# 9. SAVE
# ------------------------------------------------------------

fig.tight_layout()

fig.savefig(
    FIG / f"{OUT_STEM}.svg",
    bbox_inches="tight"
)

fig.savefig(
    FIG / f"{OUT_STEM}.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# INSPECT SAVED PyWGCNA OBJECT FOR NETWORK-CONSTRUCTION SETTINGS
# ============================================================

import inspect
import pprint

print("Object class:", type(meso))


# ------------------------------------------------------------
# 1. Search saved object attributes for likely parameter names
# ------------------------------------------------------------

KEYWORDS = [
    "power",
    "soft",
    "threshold",
    "module",
    "tree",
    "cut",
    "deep",
    "merge",
    "diss",
    "min",
    "network",
    "tom",
]

print("\n=== SAVED OBJECT ATTRIBUTES ===")

for key, value in vars(meso).items():
    if any(k in key.lower() for k in KEYWORDS):
        try:
            print(f"{key}: {value}")
        except Exception:
            print(f"{key}: <could not display>")


# ------------------------------------------------------------
# 2. Check AnnData .uns in case parameters were saved there
# ------------------------------------------------------------

print("\n=== datExpr.uns KEYS ===")

if hasattr(meso, "datExpr") and hasattr(meso.datExpr, "uns"):

    print(list(meso.datExpr.uns.keys()))

    for key, value in meso.datExpr.uns.items():
        if any(k in str(key).lower() for k in KEYWORDS):
            print(f"\n{key}:")
            pprint.pp(value)


# ------------------------------------------------------------
# 3. List relevant methods + their signatures/defaults
#
# This does NOT prove which defaults were used, but tells us
# what PyWGCNA's relevant functions expose.
# ------------------------------------------------------------

print("\n=== RELEVANT METHOD SIGNATURES ===")

for name in dir(meso):

    if any(
        k in name.lower()
        for k in [
            "power",
            "module",
            "network",
            "tom",
            "tree",
        ]
    ):

        obj = getattr(meso, name)

        if callable(obj):

            try:
                print(
                    f"\n{name}"
                    f"{inspect.signature(obj)}"
                )
            except Exception:
                pass


# ------------------------------------------------------------
# 4. Explicitly look for common likely names
# ------------------------------------------------------------

LIKELY_NAMES = [
    "power",
    "softPower",
    "soft_power",
    "networkType",
    "network_type",
    "minModuleSize",
    "min_module_size",
    "MEDissThres",
    "mergeCutHeight",
    "merge_cut_height",
    "deepSplit",
    "deep_split",
    "cutHeight",
    "cut_height",
    "TOMType",
    "TOM",
]

print("\n=== EXPLICIT COMMON PARAMETER NAMES ===")

for name in LIKELY_NAMES:

    if hasattr(meso, name):

        value = getattr(meso, name)

        if name == "TOM":
            try:
                print(
                    f"{name}: matrix shape "
                    f"{np.asarray(value).shape}"
                )
            except Exception:
                print(f"{name}: present")

        else:
            print(f"{name}: {value}")

In [ ]:
# ============================================================
# OPTION 2 — RANKED APLNR TOM NEIGHBORS
#
# X-axis: topological overlap with APLNR
# Y-axis: genes
# Selection: highest TOM values involving APLNR
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator


# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------

EDGE_FILE = TAB / "APLNR_WGCNA_mesoderm_network_edges.csv"
NODE_FILE = TAB / "APLNR_WGCNA_mesoderm_network_nodes.csv"
# ============================================================
# TRUE TOP APLNR NEIGHBORS ACROSS ALL 218 MODULE GENES
# ============================================================

TOP_N = 15

# Genes belonging to the APLNR-containing black module
module_colors = (
    meso.datExpr.var["moduleColors"]
    .astype(str)
)

module_genes = (
    module_colors.index[
        module_colors == "black"
    ]
    .astype(str)
    .tolist()
)

assert len(module_genes) == 218
assert "APLNR" in module_genes


# TOM between APLNR and every other gene in the module
apl = (
    tom_df.loc[
        "APLNR",
        module_genes
    ]
    .drop(labels="APLNR")
    .dropna()
    .sort_values(ascending=False)
    .head(TOP_N)
    .rename("TOM")
    .rename_axis("gene")
    .reset_index()
)


# Add module-membership values
kme_table = (
    meso.signedKME[
        ["kMEblack"]
    ]
    .rename_axis("gene")
    .reset_index()
    .rename(
        columns={"kMEblack": "kME"}
    )
)

apl = apl.merge(
    kme_table,
    on="gene",
    how="left"
)

print(apl.to_string(index=False))


# ------------------------------------------------------------
# 3. ORDER FOR HORIZONTAL PLOT
# ------------------------------------------------------------

apl = (
    apl.sort_values("TOM", ascending=True)
    .reset_index(drop=True)
)

y = np.arange(len(apl))

x_start = max(
    0,
    apl["TOM"].min() - 0.03
)


# ------------------------------------------------------------
# 4. DRAW LOLLIPOP PLOT
# ------------------------------------------------------------

fig_height = max(
    4.5,
    0.40 * len(apl) + 1.5
)

fig, ax = plt.subplots(
    figsize=(7.2, fig_height)
)

# Stems
for yi, tom in zip(y, apl["TOM"]):

    ax.hlines(
        y=yi,
        xmin=0,
        xmax=tom,
        color="#C8C8C8",
        linewidth=2.0,
        zorder=1
    )

# Endpoints
ax.scatter(
    apl["TOM"],
    y,
    s=155,
    facecolor="#BE6455",
    #edgecolor="#303030",
    linewidth=0.7,
    zorder=3
)


# ------------------------------------------------------------
# 5. LABELS
# ------------------------------------------------------------

ax.set_yticks(y)

ax.set_yticklabels(
    apl["gene"],
    fontsize=18,
    fontstyle="italic"
)

ax.set_xlabel(
    "Topological overlap with APLNR",
    fontsize=12
)

ax.tick_params(
    axis="x",
    labelsize=18
)

ax.tick_params(
    axis="y",
    length=0
)

ax.xaxis.set_major_locator(
    MaxNLocator(4)
)

ax.grid(
    axis="x",
    color="#E0E0E0",
    linewidth=0.7,
    zorder=0
)


# Optional kME labels to the right
if apl["kME"].notna().any():

    right_pad = 0.055

    for yi, tom, kme in zip(
        y,
        apl["TOM"],
        apl["kME"]
    ):

        if pd.isna(kme):
            continue

        # ax.text(
        #     tom + 0.006,
        #     yi,
        #     f"kME={kme:.2f}",
        #     ha="left",
        #     va="center",
        #     fontsize=8.5,
        #     color="#666666"
        # )

    ax.set_xlim(
        0,
        apl["TOM"].max() + right_pad
    )

else:

    ax.set_xlim(
        0,
        apl["TOM"].max() + 0.02
    )


for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)


# Highest TOM gene appears at top
ax.set_ylim(
    -0.7,
    len(apl) - 0.3
)

fig.tight_layout()


# ------------------------------------------------------------
# 6. SAVE
# ------------------------------------------------------------

fig.savefig(
    FIG / f"{OUT_STEM}.svg",
    bbox_inches="tight"
)

fig.savefig(
    FIG / f"{OUT_STEM}.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# SELF-CONTAINED WGCNA SUPPLEMENT
# A = dendrogram + module colors
# B = unbiased GO enrichment
# C = all-tissue vs mesoderm robustness
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
from matplotlib.ticker import MaxNLocator
from scipy.cluster.hierarchy import dendrogram
from scipy.stats import pearsonr
from PyWGCNA.utils import readWGCNA
import gseapy as gp


# ============================================================
# LOAD SAVED WGCNA OBJECTS
# ============================================================

meso = readWGCNA(
    str(TAB / "WGCNA_Ctrl_mesoderm_only" / "Ctrl_mesoderm_only.p")
)

allt = readWGCNA(
    str(TAB / "WGCNA_allCtrl_allTissues" / "allCtrl_allTissues.p")
)


# ============================================================
# GET COMPLETE MESODERM APLNR MODULE + RUN GO ENRICHMENT
# ============================================================

meso_modules = meso.datExpr.var["moduleColors"].astype(str)

module_genes = (
    meso_modules.index[meso_modules == "black"]
    .astype(str)
    .tolist()
)

print("Mesoderm APLNR module:", len(module_genes), "genes")
print("APLNR in module:", "APLNR" in module_genes)

enr = gp.enrichr(
    gene_list=module_genes,
    gene_sets=["GO_Biological_Process_2023"],
    organism="human",
    outdir=None
)

res = (
    enr.results
    .sort_values("Adjusted P-value")
    .reset_index(drop=True)
)

res.to_csv(
    TAB / "APLNR_mesoderm_module_GO_enrichment_ALL.csv",
    index=False
)


# ============================================================
# A. GENE DENDROGRAM + MODULE COLORS
# ============================================================

Z = np.asarray(meso.geneTree)
genes = pd.Index(meso.datExpr.var_names)
mods = meso.datExpr.var["moduleColors"].astype(str).reindex(genes).to_numpy()

D = dendrogram(Z, no_plot=True)
leaf_order = D["leaves"]

# WGCNA-like hanging leaves
ys_new = []
for ys in D["dcoord"]:
    ys = list(ys)
    parent = ys[1]

    if ys[0] == 0:
        ys[0] = parent - 0.03

    if ys[3] == 0:
        ys[3] = parent - 0.03

    ys_new.append(ys)

fig = plt.figure(figsize=(11, 5))

gs = fig.add_gridspec(
    2, 1,
    height_ratios=[7, 1],
    hspace=0.04
)

ax = fig.add_subplot(gs[0])
axc = fig.add_subplot(gs[1], sharex=ax)

for xs, ys in zip(D["icoord"], ys_new):
    ax.plot(xs, ys, color="black", lw=0.35)

ax.set_ylim(0.56, 1.01)
ax.set_ylabel("Height", fontsize=15)
ax.set_xticks([])
ax.tick_params(labelsize=13)
ax.yaxis.set_major_locator(MaxNLocator(4))

for s in ["top", "right", "bottom"]:
    ax.spines[s].set_visible(False)

ordered = mods[leaf_order]

rgb = np.array([
    to_rgb(c)
    for c in ordered
])[None, :, :]

axc.imshow(
    rgb,
    aspect="auto",
    extent=[0, 10*len(ordered), 0, 1],
    interpolation="nearest"
)

axc.set_yticks([0.5])
axc.set_yticklabels(["Module colors"], fontsize=14)
axc.set_xticks([])
axc.tick_params(axis="y", length=0)

for s in axc.spines.values():
    s.set_visible(False)

# mark APLNR
apl_pos = leaf_order.index(
    genes.get_loc("APLNR")
)

apl_x = 5 + 10*apl_pos

axc.annotate(
    "APLNR",
    xy=(apl_x, 1),
    xytext=(apl_x, 1.8),
    ha="center",
    fontsize=14,
    fontweight="bold",
    arrowprops=dict(
        arrowstyle="-|>",
        color="black",
        lw=1.2
    ),
    annotation_clip=False
)

fig.savefig(
    FIG / "SUPP_WGCNA_A_dendrogram.svg",
    bbox_inches="tight"
)

plt.show()


# ============================================================
# B. UNBIASED GO ENRICHMENT — TOP 20 SIGNIFICANT TERMS
# ============================================================

go = res[
    res["Adjusted P-value"] < 0.05
].copy()

go = (
    go.sort_values("Adjusted P-value")
      .head(20)
)

go["score"] = -np.log10(
    go["Adjusted P-value"]
)

go["n"] = (
    go["Overlap"]
    .str.split("/")
    .str[0]
    .astype(int)
)

go["label"] = (
    go["Term"]
    .str.replace(
        r"\s+\(GO:\d+\)$",
        "",
        regex=True
    )
)

go = go.sort_values("score")

fig, ax = plt.subplots(
    figsize=(9.5, 9)
)

ax.scatter(
    go["score"],
    range(len(go)),
    s=40 + 9*go["n"],
    color="#5C8C88",
    alpha=.88,
    edgecolors="white"
)

ax.set_yticks(
    range(len(go))
)

ax.set_yticklabels(
    go["label"],
    fontsize=20
)

ax.set_xlabel(
    r"$-\log_{10}$(BH-adjusted P)",
    fontsize=20
)

ax.tick_params(
    axis="x",
    labelsize=20
)

ax.xaxis.set_major_locator(
    MaxNLocator(4)
)


for s in ["top", "right"]:
    ax.spines[s].set_visible(False)

fig.savefig(
    FIG / "SUPP_WGCNA_B_GO_enrichment.svg",
    bbox_inches="tight"
)


plt.show()


# ============================================================
# C. ALL-TISSUE vs MESODERM MODULE ROBUSTNESS
# ============================================================

all_modules = (
    allt.datExpr.var["moduleColors"]
    .astype(str)
)

meso_modules = (
    meso.datExpr.var["moduleColors"]
    .astype(str)
)

all_black = set(
    all_modules.index[
        all_modules == "black"
    ]
)

meso_black = set(
    meso_modules.index[
        meso_modules == "black"
    ]
)

shared = sorted(
    all_black & meso_black
)

df = pd.DataFrame({
    "gene": shared,
    "all": allt.signedKME.loc[
        shared, "kMEblack"
    ],
    "meso": meso.signedKME.loc[
        shared, "kMEblack"
    ],
}).dropna()

r, _ = pearsonr(
    df["all"],
    df["meso"]
)

fig, ax = plt.subplots(
    figsize=(6.8, 6.3)
)

ax.scatter(
    df["all"],
    df["meso"],
    s=42,
    color="#BBBBBB",
    alpha=.75,
    edgecolors="white"
)

lo = min(
    df["all"].min(),
    df["meso"].min()
) - .03

hi = max(
    df["all"].max(),
    df["meso"].max()
) + .03

ax.plot(
    [lo, hi],
    [lo, hi],
    "--",
    color="#BBBBBB"
)

ax.set_xlim(lo, hi)
ax.set_ylim(lo, hi)

# highlight APLNR only
q = df[
    df["gene"] == "APLNR"
]

x = q["all"].iloc[0]
y = q["meso"].iloc[0]

ax.scatter(
    x,
    y,
    s=100,
    color="#D88A26",
    edgecolors="black",
    zorder=5
)

ax.annotate(
    "APLNR",
    (x, y),
    xytext=(8, 6),
    textcoords="offset points",
    fontsize=14,
    fontweight="bold"
)

ax.text(
    .05,
    .95,
    f"{len(all_black)} genes (all tissue)\n"
    f"{len(meso_black)} genes (mesoderm)\n"
    f"{len(shared)} shared genes\n"
    f"Pearson r = {r:.2f}",
    transform=ax.transAxes,
    va="top",
    fontsize=13
)

ax.set_xlabel(
    "kME in all-tissue WGCNA",
    fontsize=15
)

ax.set_ylabel(
    "kME in mesoderm-restricted WGCNA",
    fontsize=15
)

ax.tick_params(
    labelsize=13
)

ax.xaxis.set_major_locator(
    MaxNLocator(4)
)

ax.yaxis.set_major_locator(
    MaxNLocator(4)
)

for s in ["top", "right"]:
    ax.spines[s].set_visible(False)
ax.grid(False)
fig.savefig(
    FIG / "SUPP_WGCNA_C_robustness.svg",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# PRERANKED GSEA OF APLNR-MODULE MEMBERSHIP
#
# Question:
# Which biological processes become enriched as genes show
# stronger membership in the APLNR-containing black module?
#
# Requires:
#   meso = loaded PyWGCNA object
#   TAB, FIG
#   gseapy, pandas, numpy, matplotlib
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import gseapy as gp


# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------

MODULE = "black"
KME_COL = f"kME{MODULE}"

GENE_SET_LIBRARIES = [
    "GO_Biological_Process_2023",
    "Reactome_2022",
]

N_PERMUTATIONS = 10_000
MIN_GENE_SET_SIZE = 10
MAX_GENE_SET_SIZE = 500
FDR_CUTOFF = 0.05

N_DISPLAY = 6

# Two pathways are treated as redundant when their leading-edge
# gene sets have Jaccard similarity above this threshold.
LEADING_EDGE_JACCARD_MAX = 0.50

OUT = TAB / "APLNR_module_preranked_GSEA"
OUT.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 1. BUILD RANKING FROM SIGNED MODULE MEMBERSHIP
# ------------------------------------------------------------

assert KME_COL in meso.signedKME.columns, (
    f"{KME_COL} not found. Available columns:\n"
    f"{meso.signedKME.columns.tolist()}"
)

ranking = (
    meso.signedKME[KME_COL]
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
    .rename("score")
    .rename_axis("gene")
    .reset_index()
)

ranking["gene"] = ranking["gene"].astype(str)

# Defensive removal of unusable or duplicated names
ranking = ranking[
    ranking["gene"].notna()
    & ~ranking["gene"].isin(["", "nan", "None"])
    & ~ranking["gene"].str.startswith("ENSG")
].copy()

# If duplicate symbols exist, retain the entry with the highest kME.
ranking = (
    ranking.sort_values("score", ascending=False)
    .drop_duplicates("gene", keep="first")
    .reset_index(drop=True)
)

ranking.to_csv(
    OUT / "APLNR_black_module_kME_ranking.csv",
    index=False
)

print(f"Ranked genes: {len(ranking):,}")
print("\nTop-ranked genes:")
print(ranking.head(20).to_string(index=False))

assert "APLNR" in set(ranking["gene"]), "APLNR missing from ranking"

print(
    "\nAPLNR rank:",
    ranking.index[ranking["gene"] == "APLNR"][0] + 1,
    "of",
    len(ranking)
)


# ------------------------------------------------------------
# 2. RUN PRERANKED GSEA SEPARATELY FOR EACH LIBRARY
# ------------------------------------------------------------

all_results = []

for library in GENE_SET_LIBRARIES:

    print("\n" + "=" * 70)
    print("Running:", library)
    print("=" * 70)

    pre = gp.prerank(
        rnk=ranking[["gene", "score"]],
        gene_sets=library,
        min_size=MIN_GENE_SET_SIZE,
        max_size=MAX_GENE_SET_SIZE,
        permutation_num=N_PERMUTATIONS,
        seed=2026,
        ascending=False,
        threads=4,
        outdir=None,
        verbose=True
    )

    rr = pre.res2d.copy().reset_index(drop=True)
    rr["library"] = library

    all_results.append(rr)


gsea = pd.concat(
    all_results,
    ignore_index=True
)

print("\nGSEA columns:")
print(gsea.columns.tolist())


# ------------------------------------------------------------
# 3. STANDARDIZE COLUMN NAMES
# ------------------------------------------------------------

def find_column(df, candidates):
    """Return the first matching column name, ignoring case/spaces."""
    lookup = {
        str(c).lower().replace(" ", "").replace("_", ""): c
        for c in df.columns
    }

    for candidate in candidates:
        key = candidate.lower().replace(" ", "").replace("_", "")
        if key in lookup:
            return lookup[key]

    raise KeyError(
        f"Could not find any of {candidates}. "
        f"Available columns: {df.columns.tolist()}"
    )


TERM_COL = find_column(
    gsea,
    ["Term", "Name"]
)

NES_COL = find_column(
    gsea,
    ["NES"]
)

FDR_COL = find_column(
    gsea,
    ["FDR q-val", "FDR", "FDR qval"]
)

NOM_P_COL = find_column(
    gsea,
    ["NOM p-val", "NOM pval", "P-value", "pval"]
)

LEADING_COL = find_column(
    gsea,
    ["Lead_genes", "ledge_genes", "Leading edge", "Leading_edge"]
)


gsea["term"] = (
    gsea[TERM_COL]
    .astype(str)
    .str.replace(r"\s+\(GO:\d+\)$", "", regex=True)
)

gsea["NES_value"] = pd.to_numeric(
    gsea[NES_COL],
    errors="coerce"
)

gsea["FDR_value"] = pd.to_numeric(
    gsea[FDR_COL],
    errors="coerce"
)

gsea["NOM_P_value"] = pd.to_numeric(
    gsea[NOM_P_COL],
    errors="coerce"
)


# ------------------------------------------------------------
# 4. PARSE LEADING-EDGE GENES
# ------------------------------------------------------------

def parse_leading_genes(value):

    if pd.isna(value):
        return []

    value = str(value)

    # GSEApy commonly separates genes with semicolons.
    # This also handles comma-separated output defensively.
    if ";" in value:
        genes = value.split(";")
    elif "," in value:
        genes = value.split(",")
    else:
        genes = value.split("/")

    return [
        g.strip()
        for g in genes
        if g.strip()
    ]


gsea["leading_genes_list"] = (
    gsea[LEADING_COL]
    .apply(parse_leading_genes)
)

gsea["n_leading_genes"] = (
    gsea["leading_genes_list"]
    .str.len()
)

gsea["leading_genes"] = (
    gsea["leading_genes_list"]
    .apply(lambda x: "; ".join(x))
)


# Save complete, unfiltered output
save_cols = [
    "library",
    "term",
    "NES_value",
    "FDR_value",
    "NOM_P_value",
    "n_leading_genes",
    "leading_genes",
]

gsea[save_cols].to_csv(
    OUT / "APLNR_black_module_GSEA_ALL_results.csv",
    index=False
)


# ------------------------------------------------------------
# 5. KEEP SIGNIFICANT POSITIVE ENRICHMENTS
# ------------------------------------------------------------

sig = gsea[
    (gsea["FDR_value"] < FDR_CUTOFF)
    & (gsea["NES_value"] > 0)
].copy()

sig = sig.sort_values(
    ["FDR_value", "NES_value"],
    ascending=[True, False]
)

print(f"\nSignificant positive pathways: {len(sig)}")


# ------------------------------------------------------------
# 6. REMOVE REDUNDANT PATHWAYS USING LEADING-EDGE OVERLAP
# ------------------------------------------------------------

def jaccard(a, b):

    a = set(a)
    b = set(b)

    if not a or not b:
        return 0.0

    return len(a & b) / len(a | b)


selected_rows = []

for idx, row in sig.iterrows():

    candidate_genes = row["leading_genes_list"]

    redundant = False

    for kept in selected_rows:

        similarity = jaccard(
            candidate_genes,
            kept["leading_genes_list"]
        )

        if similarity >= LEADING_EDGE_JACCARD_MAX:
            redundant = True
            break

    if not redundant:
        selected_rows.append(row)

    if len(selected_rows) >= N_DISPLAY:
        break


display_df = pd.DataFrame(selected_rows).copy()

if display_df.empty:
    raise ValueError(
        "No significant positive pathways passed the filters."
    )

display_df.to_csv(
    OUT / "APLNR_black_module_GSEA_displayed_terms.csv",
    index=False
)


# ------------------------------------------------------------
# 7. PRINT DISPLAYED PATHWAYS AND LEADING-EDGE GENES
# ------------------------------------------------------------

for _, row in display_df.iterrows():

    print("\n" + row["term"])
    print(f"Library: {row['library']}")
    print(f"NES: {row['NES_value']:.2f}")
    print(f"FDR: {row['FDR_value']:.3g}")
    print(
        "Leading edge:",
        ", ".join(row["leading_genes_list"])
    )


# ------------------------------------------------------------
# 8. PLOT NONREDUNDANT PATHWAYS
# ------------------------------------------------------------

plot_df = (
    display_df
    .sort_values("NES_value", ascending=True)
    .reset_index(drop=True)
)

# Shorten common library suffixes if present
plot_df["display_term"] = (
    plot_df["term"]
    .str.replace("_", " ", regex=False)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

fig_height = max(
    3.8,
    0.72 * len(plot_df) + 1.5
)

fig, ax = plt.subplots(
    figsize=(8.0, fig_height)
)

y = np.arange(len(plot_df))

sizes = (
    55
    + 14 * plot_df["n_leading_genes"].to_numpy()
)

scat = ax.scatter(
    plot_df["NES_value"],
    y,
    s=sizes,
    c=-np.log10(plot_df["FDR_value"]),
    cmap="viridis",
    edgecolors="white",
    linewidths=0.7
)

ax.set_yticks(y)

ax.set_yticklabels(
    plot_df["display_term"],
    fontsize=12
)

ax.set_xlabel(
    "Normalized enrichment score",
    fontsize=13
)

ax.tick_params(
    axis="x",
    labelsize=11
)

ax.xaxis.set_major_locator(
    MaxNLocator(4)
)

for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)

ax.grid(
    axis="x",
    color="#DDDDDD",
    linewidth=0.7,
    alpha=0.7
)

cbar = fig.colorbar(
    scat,
    ax=ax,
    fraction=0.045,
    pad=0.03
)

cbar.set_label(
    r"$-\log_{10}$(FDR)",
    fontsize=11
)

fig.tight_layout()

fig.savefig(
    FIG / "APLNR_black_module_GSEA_nonredundant.svg",
    bbox_inches="tight"
)

fig.savefig(
    FIG / "APLNR_black_module_GSEA_nonredundant.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ------------------------------------------------------------
# 9. OPTIONAL: LEADING-EDGE GENE TABLE FOR THE FIGURE
# ------------------------------------------------------------

leading_edge_table = display_df[
    [
        "library",
        "term",
        "NES_value",
        "FDR_value",
        "n_leading_genes",
        "leading_genes",
    ]
].copy()

display(leading_edge_table)

## Panel A — CS7 embryo (Tyser)

In [ ]:
EMBRYO_ORDER = ["Epiblast","Primitive Streak","Nascent Mesoderm","Emergent Mesoderm",
                "Advanced Mesoderm","Axial Mesoderm","ExE Mesoderm",
                "Hemogenic Endothelial Progenitors","Erythroblasts",
                "Endoderm","Non-Neural Ectoderm"]
present = [c for c in EMBRYO_ORDER if c in set(tyser.obs["cluster_id"].astype(str))]
tyser.obs["tissue_embryo"] = pd.Categorical(tyser.obs["cluster_id"].astype(str),
                                            categories=present, ordered=True)
ty_xy = np.asarray(tyser.obsm["X_umap"])

fig, ax = plt.subplots(figsize=(7, 6))
for c in present:
    m = (tyser.obs["tissue_embryo"].astype(str)==c).to_numpy()
    ax.scatter(ty_xy[m,0], ty_xy[m,1], s=30, color=EMBRYO_COLORS[c],
               linewidths=0, rasterized=True)
if SHOW_TITLES: ax.set_title(f"Human CS7 embryo, n={tyser.n_obs:,}")
ax.set_xticks([]); ax.set_yticks([]); ax.set_frame_on(False)
ax.set_aspect("equal"); ax.grid(False)
side_legend(ax, present, EMBRYO_COLORS, fontsize=EMBRYO_FS, marker=5)
fig.tight_layout()
for ext in SAVE_FORMATS: fig.savefig(FIG/f"A_embryo_umap.{ext}", bbox_inches="tight")
plt.show()

In [ ]:
from matplotlib.colors import PowerNorm
import scipy.sparse as sp
# ---- APLNR + APELA on the Tyser embryo UMAP ----
GENES = ["APLNR", "APELA"]
ty_xy = np.asarray(tyser.obsm["X_umap"])
def get_expr(g):
    if f"{g}_magic" in tyser.obs:                                  # MAGIC in obs
        return tyser.obs[f"{g}_magic"].to_numpy(), "MAGIC"
    syms = [str(v).split(" (")[0] for v in tyser.var_names]        # else from .X
    hit = next((i for i,v in enumerate(syms) if v == g), None)
    if hit is None: raise KeyError(f"{g}: no _magic col and not in var_names")
    x = tyser.X[:, hit]
    x = np.asarray(x.todense()).ravel() if sp.issparse(x) else np.asarray(x).ravel()
    return x, "expr"

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, g in zip(axes, GENES):
    col, src = get_expr(g)
    vmax = max(float(np.nanpercentile(col, 95)), 1e-6)
    norm = PowerNorm(gamma=0.5, vmin=0, vmax=vmax)                 # brighten low end
    o = np.argsort(col)
    sctr = ax.scatter(ty_xy[o,0], ty_xy[o,1], s=30, c=col[o], cmap="Reds",
                      norm=norm, linewidths=0, rasterized=True)
    ax.text(0.02, 0.97, f"{g}  ({src})", transform=ax.transAxes, fontsize=LEGEND_FS,
            va="top", ha="left", color=GREY_TEXT)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_frame_on(False); ax.set_aspect("equal"); ax.grid(False)
    fig.colorbar(sctr, ax=ax, fraction=0.04, pad=0.02, label=g)
for ext in SAVE_FORMATS: fig.savefig(FIG/f"A_embryo_aplnr_apela_umap.{ext}", bbox_inches="tight")
plt.show()

## Panel B — control gastruloids (tissue, then timepoint)

In [ ]:
print(ctrl.obs["tissue"].cat.categories.tolist())
print(ctrl.obs["predicted_leiden_sub_joint"].cat.categories.tolist())
print(dox.obs["tissue"].cat.categories.tolist())
print(dox.obs["predicted_leiden_sub_joint"].cat.categories.tolist())

In [ ]:
xy = np.asarray(ctrl.obsm["X_umap_joint"])
#cats = list(ctrl.obs["tissue"].cat.categories)
cats = list(ctrl.obs["predicted_leiden_sub_joint"].cat.categories)
TIME_COLORS = {"42h": BLUE, "72h": RED}

fig, axes = plt.subplots(1, 2, figsize=(15, 6.5))
for c in cats:
    m = (ctrl.obs["predicted_leiden_sub_joint"].astype(str) == c).to_numpy()
    if m.sum(): axes[0].scatter(xy[m,0], xy[m,1], s=15, color=TISSUE_COLORS[c],
                                linewidths=0, rasterized=True)
side_legend(axes[0], cats, TISSUE_COLORS)
for t in TIMES:
    m = (ctrl.obs["timepoint_true"].astype(str) == t).to_numpy()
    axes[1].scatter(xy[m,0], xy[m,1], s=10, color=TIME_COLORS[t], alpha=0.75,
                    linewidths=0, rasterized=True)
side_legend(axes[1], TIMES, TIME_COLORS)
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([]); ax.set_frame_on(False)
    ax.set_aspect("equal"); ax.grid(False)
fig.tight_layout()
for ext in SAVE_FORMATS: fig.savefig(FIG/f"B_control_umap.{ext}", bbox_inches="tight")
plt.show()

In [ ]:
REP_COL = "sample_id"
REPS = {"Chicago": ["Chicago_Ctrl_48h", "Chicago_Ctrl_72h"],
        "Michigan": ["Michigan_Ctrl_48h", "Michigan_Ctrl_72h"]}

# NOW subsample
rng = np.random.default_rng(42)
idx_chi  = np.where(ctrl.obs[REP_COL].isin(REPS["Chicago"]).to_numpy())[0]
idx_mich = np.where(ctrl.obs[REP_COL].isin(REPS["Michigan"]).to_numpy())[0]

n_each = len(idx_chi)  # Chicago is smaller (4,273) — show same n for Michigan
print(f"Chicago: {len(idx_chi):,}  Michigan: {len(idx_mich):,}  → showing {n_each:,} each")

idx_chi_sub  = idx_chi
idx_mich_sub = rng.choice(idx_mich, n_each, replace=False)
sub_idx = {"Chicago": idx_chi_sub, "Michigan": idx_mich_sub}


xy = np.asarray(ctrl.obsm["X_umap_joint"])
cats = list(ctrl.obs["predicted_leiden_sub_joint"].cat.categories)

fig, axes = plt.subplots(1, 3, figsize=(21, 6.5))

# Panel 1: all cells, coloured by tissue
for c in cats:
    m = (ctrl.obs["predicted_leiden_sub_joint"].astype(str) == c).to_numpy()
    if m.sum(): axes[0].scatter(xy[m,0], xy[m,1], s=15, color=TISSUE_COLORS[c],
                                linewidths=0, rasterized=True)
side_legend(axes[0], cats, TISSUE_COLORS)
axes[0].set_title("All cells", fontsize=12)

# Panels 2 & 3: one per replicate
for ax, (rep_label, rep_ids) in zip(axes[1:], REPS.items()):
    m_rep = np.zeros(ctrl.n_obs, dtype=bool)
    m_rep[sub_idx[rep_label]] = True          # ← use subsampled mask
    for c in cats:
        m = (ctrl.obs["predicted_leiden_sub_joint"].astype(str) == c).to_numpy() & m_rep
        if m.sum(): ax.scatter(xy[m,0], xy[m,1], s=20, color=TISSUE_COLORS[c],
                               linewidths=0, rasterized=True)
    ax.set_title(f"{rep_label} (n={n_each:,})", fontsize=12)

for ax in axes:
    ax.set_xticks([]); ax.set_yticks([]); ax.set_frame_on(False)
    ax.set_aspect("equal"); ax.grid(False)

fig.tight_layout()
for ext in SAVE_FORMATS: fig.savefig(FIG/f"B_umap_by_repeat.{ext}", bbox_inches="tight")
plt.show()

In [ ]:
REP_COL = "sample_id"
REPS = {
    "Chicago":  adata.obs[REP_COL].str.startswith("Chicago").to_numpy(),
    "Michigan": adata.obs[REP_COL].str.startswith("Michigan").to_numpy(),
}

rng = np.random.default_rng(42)
idx_chi  = np.where(REPS["Chicago"])[0]
idx_mich = np.where(REPS["Michigan"])[0]
n_each = len(idx_chi)   # Chicago is smaller
print(f"Chicago: {len(idx_chi):,}  Michigan: {len(idx_mich):,}  → showing {n_each:,} each")

idx_mich_sub = rng.choice(idx_mich, n_each, replace=False)
sub_idx = {"Chicago": idx_chi, "Michigan": idx_mich_sub}

xy_all = np.asarray(adata.obsm["X_umap_joint"])
cats = list(adata.obs["predicted_leiden_sub_joint"].cat.categories)

fig, axes = plt.subplots(1, 3, figsize=(21, 6.5))

# Panel 1: all cells
for c in cats:
    m = (adata.obs["predicted_leiden_sub_joint"].astype(str) == c).to_numpy()
    if m.sum(): axes[0].scatter(xy_all[m,0], xy_all[m,1], s=8, color=TISSUE_COLORS[c],
                                linewidths=0, rasterized=True)
side_legend(axes[0], cats, TISSUE_COLORS)
axes[0].set_title(f"All cells (n={adata.n_obs:,})", fontsize=12)

# Panels 2 & 3: by replicate (DOX + Ctrl, subsampled to equal n)
for ax, rep_label in zip(axes[1:], ["Chicago", "Michigan"]):
    m_rep = np.zeros(adata.n_obs, dtype=bool)
    m_rep[sub_idx[rep_label]] = True
    for c in cats:
        m = (adata.obs["predicted_leiden_sub_joint"].astype(str) == c).to_numpy() & m_rep
        if m.sum(): ax.scatter(xy_all[m,0], xy_all[m,1], s=12, color=TISSUE_COLORS[c],
                               linewidths=0, rasterized=True)
    ax.set_title(f"{rep_label} (n={n_each:,})", fontsize=12)

for ax in axes:
    ax.set_xticks([]); ax.set_yticks([]); ax.set_frame_on(False)
    ax.set_aspect("equal"); ax.grid(False)

fig.tight_layout()
for ext in SAVE_FORMATS: fig.savefig(FIG/f"B_umap_overview_reps.{ext}", bbox_inches="tight")
plt.show()

## Panel C — collaborator-style dot plot (controls)

Stan's marker groups and row order, his n/percent labels. APLNR/APELA appended as their own "apelin" group.

In [ ]:
GENE_W, ROW_H = 0.40, 0.30
SMALLEST_DOT, LARGEST_DOT, DOT_MAX = 14, 190, 1.0

# COLLAB_MARKERS = {
#     "PM":["MSGN1","EVX1","RSPO3","TBX6","FOXC1", "HES7"], "somitomere":["MESP1","MESP2","RIPPLY2"],
#     "somite":["FOXC2","MEOX1", "DLL3"], "PPS":["WNT8A","TBXT"], "APS":["EOMES","MIXL1", "NODAL"],
#     "DE":["FOXA2","HNF1B"], "PGCLC":["SOX17","PRDM1","NANOS3","TFAP2C"],
#     "pluri":["NANOG","SOX2","USP44", "POU5F1"], "AE-LC":["GABRP","WNT6","ISL1"],
#     "apelin":["APLNR","APELA"],
# }
# COLLAB_ROW_ORDER = ["eNasM","lNasM","aPM-LC","LPM-LC","CS-LC","pPS-LC","aPS-LC",
#                     "DE","PGC-LC","pluripotent","AE-LC"]

COLLAB_MARKERS = {
    "pluri":["NANOG","SOX2","USP44", "POU5F1"], "AE-LC":["GABRP","WNT6","ISL1"],
    "PGCLC":["SOX17","PRDM1","TFAP2C"],
    "PPS":["WNT8A","TBXT"], "APS":["EOMES","MIXL1", "NODAL"],
    "PM":["MSGN1","EVX1","RSPO3","TBX6","FOXC1", "HES7", "MESP1","MESP2","RIPPLY2", "FOXC2","DLL3","LFNG"],
    "DE":["FOXA2","HNF1B"],
}

COLLAB_ROW_ORDER = ["pluripotent","CS-LC","PGC-LC","AE-LC","pPS-LC","aPS-LC","eNasM","lNasM","LPM-LC","aPM-LC",
                    "DE"]
DROP_STATES = ["LPM-LC","CS-LC","stressed"]

sym2key = dict(zip(adata.var["symbol"].astype(str), adata.var.index)) \
          if "symbol" in adata.var else {g:g for g in adata.var_names}
def resolve(mapping):
    out={}
    for grp,genes in mapping.items():
        k=[sym2key[g] for g in genes if g in sym2key]
        if k: out[grp]=k
    return out

for t in TIMES:
    # aPM-LC and DE only appear meaningfully at 72h — drop from 42h
    if t == "42h":
        drop = DROP_STATES + ["aPM-LC", "DE"]
    else:
        drop = DROP_STATES

    sub = ctrl[(ctrl.obs["timepoint_true"].astype(str)==t) &
               (~ctrl.obs["predicted_leiden_sub_joint"].astype(str).isin(drop))].copy()
    raw = sub.obs["predicted_leiden_sub_joint"].astype(str)
    counts, total = raw.value_counts(), len(raw)
    rows = [r for r in COLLAB_ROW_ORDER if r in counts.index]
    labels = {r: f"{r} (n={counts[r]},p={counts[r]/total*100:.1f}%)" for r in rows}
    sub.obs["row"] = pd.Categorical(raw.map(labels),
                                    categories=[labels[r] for r in rows], ordered=True)
    marks = resolve(COLLAB_MARKERS)
    n_genes = sum(len(v) for v in marks.values())
    print(f"{t}: {sub.n_obs:,} cells, {len(rows)} rows")

    dp = sc.pl.dotplot(sub, marks, groupby="row", use_raw=False, standard_scale="var",
        dot_max=DOT_MAX, dot_min=0.0, smallest_dot=SMALLEST_DOT, cmap="Reds", #cmap=DOT_CMAP,
        var_group_rotation=0, figsize=(GENE_W*n_genes, ROW_H*len(rows)),
        title=(f"Control {t}" if SHOW_TITLES else None), return_fig=True)
    dp.style(dot_edge_color="black", dot_edge_lw=0.25, largest_dot=LARGEST_DOT)
    dp.legend(width=1.6, colorbar_title="Mean expression\nin group",
              size_title="Fraction of cells\nin group (%)")
    dp.make_figure(); thin_size_legend(dp, keep=3)
    try:
        mp = dp.get_axes()["mainplot_ax"]
        mp.set_xticklabels([g.get_text().split(" (")[0] for g in mp.get_xticklabels()],
                           rotation=70, ha="right")
    except Exception as e: print("  xlabel skip:", e)
    for ext in SAVE_FORMATS: dp.fig.savefig(FIG/f"C_dotplot_control_{t}.{ext}", bbox_inches="tight")
    plt.show()

In [ ]:
GENE_W, ROW_H = 0.75, 0.30
SMALLEST_DOT, LARGEST_DOT, DOT_MAX = 14, 190, 1.0

APELIN_MARKERS = {
    "Apelin": ["APELA", "APLNR"]
}

ROW_ORDER = [
    "pluripotent","CS-LC","PGC-LC","AE-LC",
    "pPS-LC","aPS-LC","eNasM","lNasM",
    "LPM-LC","aPM-LC","DE"
]

DROP_STATES = ["LPM-LC","CS-LC","stressed"]

sym2key = dict(zip(adata.var["symbol"].astype(str), adata.var.index)) \
          if "symbol" in adata.var else {g:g for g in adata.var_names}

def resolve(mapping):
    return {
        grp: [sym2key[g] for g in genes if g in sym2key]
        for grp, genes in mapping.items()
        if any(g in sym2key for g in genes)
    }

marks = resolve(APELIN_MARKERS)
n_genes = 2

for t in TIMES:
    sub = ctrl[
        (ctrl.obs["timepoint_true"].astype(str) == str(t)) &
        (~ctrl.obs["predicted_leiden_sub_joint"].astype(str).isin(DROP_STATES))
    ].copy()

    raw = sub.obs["predicted_leiden_sub_joint"].astype(str)
    counts, total = raw.value_counts(), len(raw)

    rows = [r for r in ROW_ORDER if r in counts.index]

    labels = {
        r: f"{r} (n={counts[r]},p={counts[r]/total*100:.1f}%)"
        for r in rows
    }

    sub.obs["row"] = pd.Categorical(
        raw.map(labels),
        categories=[labels[r] for r in rows],
        ordered=True
    )

    print(f"{t}: {sub.n_obs:,} cells, {len(rows)} rows")

    dp = sc.pl.dotplot(
        sub,
        marks,
        groupby="row",
        use_raw=False,
        standard_scale="var",
        dot_max=DOT_MAX,
        dot_min=0.0,
        smallest_dot=SMALLEST_DOT,
        cmap="Reds",
        var_group_rotation=0,
        figsize=(GENE_W*n_genes + 1.5, ROW_H*len(rows)),
        title=(f"Control {t}" if SHOW_TITLES else None),
        return_fig=True
    )

    dp.style(
        dot_edge_color="black",
        dot_edge_lw=0.25,
        largest_dot=LARGEST_DOT
    )

    dp.legend(
        width=1.6,
        colorbar_title="mean expression\nacross group",
        size_title="Fraction of cells\nexpressing (%)"
    )

    dp.make_figure()
    thin_size_legend(dp, keep=3)

    try:
        mp = dp.get_axes()["mainplot_ax"]
        mp.set_xticklabels(
            ["APELA", "APLNR"],
            rotation=45,
            ha="right"
        )
    except Exception as e:
        print("xlabel skip:", e)

    for ext in SAVE_FORMATS:
        dp.fig.savefig(
            FIG / f"C_APELA_APLNR_control_{t}.{ext}",
            bbox_inches="tight"
        )

    plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.colors import LinearSegmentedColormap
plt.rcParams["axes.grid"] = False
# ---------------------------------------------------------------------------
# Load genome-wide APLNR/APELA correlation results
# ---------------------------------------------------------------------------

CSV = ROOT / "05_notebooks/figure2_clean/APLNR_APELA_genomewide_validation/APLNR_APELA_genomewide_validation_ALL.csv"

OUT = CSV.parent
df = pd.read_csv(CSV)

# ---------------------------------------------------------------------------
# Genes shown in MAIN heatmap
# ---------------------------------------------------------------------------

GENES = [
    "TCF7L2",
    "SOX2",
    "TPM1",
    "LRRN1",
    "ALCAM",
    "CD24",
    "EPCAM",
    "CD9",
    "MIXL1",
    "HAS2",
    "IRX3",
    "MESP1",
    "DLL3",
    "WNT5A",
    "CER1",
    "COL13A1",
    "FERMT2",
    "SERPINE2",
    "LHX1",
    "SALL1",
    "DACT1",
    "WNT8A",
    "SNAI1",
    "MSGN1",
    "BMP2"
    
]

COLS = [
    ("APELA", 42),
    ("APELA", 72),
    ("APLNR", 42),
    ("APLNR", 72),
]

# =====================================================================
# 1. MAIN FIGURE: correlation heatmap
# =====================================================================

heat = (
    df[df["gene"].isin(GENES)]
    .pivot_table(
        index="gene",
        columns=["seed", "timepoint"],
        values="consensus_r",
        aggfunc="first"
    )
    .reindex(index=GENES, columns=COLS)
)

arr = heat.to_numpy(dtype=float)
maxabs = np.nanmax(np.abs(arr))

cmap = LinearSegmentedColormap.from_list(
    "teal_white_orange",
    ["#2F7F7A", "#F7F7F7", "#C96516"],
    N=256
)
cmap.set_bad("#D0D0D0")

fig, ax = plt.subplots(figsize=(6.4, 11.5))

im = ax.imshow(
    np.ma.masked_invalid(arr),
    cmap=cmap,
    vmin=-maxabs,
    vmax=maxabs,
    aspect="equal"
)

ax.set_xticks(range(4))
ax.set_xticklabels(
    ["APELA 42 h", "APELA 72 h", "APLNR 42 h", "APLNR 72 h"],
    fontsize=22, rotation=45, ha="right"
)

ax.set_yticks(range(len(GENES)))
ax.set_yticklabels(GENES, fontsize=22)

# white cell borders
ax.set_xticks(np.arange(-0.5, 4, 1), minor=True)
ax.set_yticks(np.arange(-0.5, len(GENES), 1), minor=True)
ax.grid(which="minor", color="white", linewidth=1)
ax.tick_params(which="minor", bottom=False, left=False)
ax.tick_params(length=0)

for spine in ax.spines.values():
    spine.set_visible(False)


cbar = fig.colorbar(
    im,
    ax=ax,
    fraction=0.09,  # wider/thicker
    pad=0.03,
    shrink=0.3      # shorter
)
cbar.set_label("Consensus correlation", fontsize=15)
cbar.set_ticks([-0.5, 0, 0.5])
cbar.ax.tick_params(labelsize=16)

plt.tight_layout()

fig.savefig(
    OUT / "APLNR_APELA_selected_gene_heatmap.svg",
    bbox_inches="tight"
)
fig.savefig(
    OUT / "APLNR_APELA_selected_gene_heatmap.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# =====================================================================
# 2. SUPPLEMENT: genome-wide distribution + selected genes highlighted
# =====================================================================

# Remove unnamed Ensembl-only rows from background
plot_df = df[
    ~df["gene"].astype(str).str.match(r"^(ENSG|ENGS)", case=False, na=False)
].copy()

# only repeat-concordant genes
plot_df = plot_df[
    plot_df["correlation_same_direction"] == True
].copy()

# signed x coordinate = direction × weaker-repeat magnitude
plot_df["signed_weaker_r"] = (
    np.sign(plot_df["consensus_r"]) *
    plot_df["weaker_abs_correlation"]
)

# For each selected gene, highlight the condition where its
# reproducible correlation is strongest
selected = plot_df[
    plot_df["gene"].isin(GENES)
].copy()

selected = (
    selected.sort_values("weaker_abs_correlation")
    .groupby("gene", as_index=False)
    .tail(1)
)

ypos = {
    ("APELA", 42): 3,
    ("APLNR", 42): 2,
    ("APELA", 72): 1,
    ("APLNR", 72): 0,
}

rng = np.random.default_rng(2026)

fig, ax = plt.subplots(figsize=(7.8, 4.6))

# all genome-wide genes = light gray
for seed, tp in [
    ("APELA", 42),
    ("APLNR", 42),
    ("APELA", 72),
    ("APLNR", 72),
]:
    d = plot_df[
        (plot_df["seed"] == seed) &
        (plot_df["timepoint"] == tp)
    ]

    y = ypos[(seed, tp)] + rng.normal(0, 0.028, len(d))

    ax.scatter(
        d["signed_weaker_r"],
        y,
        s=8,
        color="#D0D0D0",
        alpha=0.35,
        linewidths=0
    )

# selected heatmap genes = larger teal/orange dots
for _, r in selected.iterrows():

    x = r["signed_weaker_r"]
    y = ypos[(r["seed"], int(r["timepoint"]))]

    color = "#C96516" if x > 0 else "#2F7F7A"

    ax.scatter(
        x,
        y,
        s=75,
        color=color,
        linewidths=0,
        zorder=3
    )

ax.axvline(0, color="black", linewidth=1)

ax.set_yticks([3, 2, 1, 0])
ax.set_yticklabels(
    ["APELA 42 h", "APLNR 42 h", "APELA 72 h", "APLNR 72 h"],
    fontsize=16
)

ax.set_xlabel("Signed weaker-repeat correlation", fontsize=14)
ax.tick_params(axis="x", labelsize=12)
ax.tick_params(axis="y", length=0)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)

plt.tight_layout()

fig.savefig(
    OUT / "APLNR_APELA_genomewide_selected_gene_QC.svg",
    bbox_inches="tight"
)
fig.savefig(
    OUT / "APLNR_APELA_genomewide_selected_gene_QC.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
GENE_W, ROW_H = 0.40, 0.30
SMALLEST_DOT, LARGEST_DOT, DOT_MAX = 14, 190, 1.0
plt.rcParams["patch.edgecolor"] = "black"
plt.rcParams["axes.edgecolor"] = "black"
plt.rcParams["text.color"] = "black"
# CORR_MARKERS = {
#     "APELA": ["APELA", "TPM1","LRRN1","ALCAM","TCF7L2"],
#     "Epithelial": ["EPCAM","CD24","CD9"],
#     "Mesoderm": ["MIXL1","HAS2","IRX3","MESP1","DLL3"],
#     "APLNR": ["APLNR", "WNT5A","CER1","COL13A1","FERMT2","LHX1"],
# }

CORR_MARKERS = {
    "Epi/Stem Cell": [
        "APELA", "SOX2", "EPCAM", "ALCAM", "APLN"
    ],
    "Morphogenesis": [
        "APLNR", "HAS2", "RSPO3", "WNT5A", "BMP2", "RND3", "LHX1"
    ],
    "Meso. patterning": [
        "TBXT", "MIXL1", "MESP1", "MESP2", "TBX6", "DLL3", "IRX3"
    ],

}

CORR_ROW_ORDER = [
    "pluripotent","CS-LC","PGC-LC","AE-LC",
    "pPS-LC","aPS-LC","eNasM","lNasM",
    "LPM-LC","aPM-LC","DE"
]

DROP_STATES = ["LPM-LC","CS-LC","stressed"]

sym2key = dict(zip(adata.var["symbol"].astype(str), adata.var.index)) \
          if "symbol" in adata.var else {g:g for g in adata.var_names}

def resolve(mapping):
    return {
        grp: [sym2key[g] for g in genes if g in sym2key]
        for grp, genes in mapping.items()
        if any(g in sym2key for g in genes)
    }

marks = resolve(CORR_MARKERS)
n_genes = sum(len(v) for v in marks.values())

for t in TIMES:
    # drop extra tissues at 42h only
    drop = set(DROP_STATES)
    if str(t) == '42h':
        drop |= {'aPM-LC', 'DE'}

    sub = ctrl[
        (ctrl.obs["timepoint_true"].astype(str) == str(t)) &
        (~ctrl.obs["predicted_leiden_sub_joint"].astype(str).isin(drop))
    ].copy()

    raw = sub.obs["predicted_leiden_sub_joint"].astype(str)
    counts, total = raw.value_counts(), len(raw)

    rows = [r for r in CORR_ROW_ORDER if r in counts.index]
    labels = {
        r: f"{r} (n={counts[r]},p={counts[r]/total*100:.1f}%)"
        for r in rows
    }

    sub.obs["row"] = pd.Categorical(
        raw.map(labels),
        categories=[labels[r] for r in rows],
        ordered=True
    )

    print(f"{t}: {sub.n_obs:,} cells, {len(rows)} rows")

    dp = sc.pl.dotplot(
        sub, marks,
        groupby="row",
        use_raw=False,
        standard_scale="var",
        dot_max=DOT_MAX,
        dot_min=0.0,
        smallest_dot=SMALLEST_DOT,
        cmap="Reds",
        var_group_rotation=0,
        figsize=(GENE_W*n_genes, ROW_H*len(rows)),
        title=(f"Control {t}" if SHOW_TITLES else None),
        return_fig=True
    )

    dp.style(
        dot_edge_color="black",
        dot_edge_lw=0.25,
        largest_dot=LARGEST_DOT
    )

    dp.legend(
        width=1.6,
        colorbar_title="mean expression\nacross states",
        size_title="Fraction of cells\nexpressing (%)"
    )

    dp.make_figure()
    thin_size_legend(dp, keep=3)

    try:
        mp = dp.get_axes()["mainplot_ax"]
        mp.set_xticklabels(
            [x.get_text().split(" (")[0] for x in mp.get_xticklabels()],
            rotation=70,
            ha="right"
        )
    except Exception as e:
        print("xlabel skip:", e)

    for ext in SAVE_FORMATS:
        dp.fig.savefig(
            FIG / f"C_corr_dotplot_control_{t}.{ext}",
            bbox_inches="tight"
        )

    plt.show()

In [ ]:
GENE_W, ROW_H = 0.40, 0.30
SMALLEST_DOT, LARGEST_DOT, DOT_MAX = 14, 190, 1.0

# ------------------------------------------------------------------
# Map directed-differentiation states onto gastruloid cell states
#
# D1 genes = high at D1 / decrease strongly from D1 -> D2
# D2 genes = acquired strongly from D1 -> D2
#
# Selected for developmental interpretability, not simply largest |LFC|.
# ------------------------------------------------------------------
DIRECTED_MARKERS = {
    "Primitive streak / D1": [
        # high in APS/MPS D1 AND enriched in gastruloid aPS/pPS vs NasM
        "NODAL",
        "TBXT",
        "EOMES",
        "MIXL1",
        "SP5",
        "WNT3",
        "NTS",
        "BAMBI",
        "FGF13",
        "EVX1",
        "ISL1",
        "MSX1",
    ],

    "Nascent mesoderm / D2": [
        # up APS/MPS D1->D2 AND enriched in gastruloid NasM vs aPS/pPS
        "DLL3",
        "MESP2",
        "CDH11",
        "FAM89A",
        "PDGFRA",
        "WNT6",
        "WNT3",
        "LRP5",
        "FZD1",
        "FZD2",
        "FZD3",
        "FZD4",
        "FZD5",
        "FZD6",
        "FZD7",
        "WNT8",
        "WNT5A",
        "WNT7",
        "WNT3A",
        "CDH1",
        "TFAP2C",
        "CDH2",
    ],
}


# ------------------------------------------------------------------
# Gastruloid states
# Keep developmental populations together so any correspondence
# with APS/MPS becomes visually obvious.
# ------------------------------------------------------------------
DIRECTED_ROW_ORDER = [
    "pluripotent",
    "pPS-LC",
    "aPS-LC",
    "eNasM",
    "lNasM",
    "aPM-LC",
    "LPM-LC",
    "AE-LC",
    "DE",
    "PGC-LC",
    "CS-LC",
]

DROP_STATES = ["stressed"]


# ------------------------------------------------------------------
# Resolve gene symbols -> adata.var_names
# ------------------------------------------------------------------
sym2key = (
    dict(zip(adata.var["symbol"].astype(str), adata.var.index))
    if "symbol" in adata.var
    else {g: g for g in adata.var_names}
)

def resolve(mapping):
    return {
        grp: [sym2key[g] for g in genes if g in sym2key]
        for grp, genes in mapping.items()
        if any(g in sym2key for g in genes)
    }

marks = resolve(DIRECTED_MARKERS)
n_genes = sum(len(v) for v in marks.values())

print("Genes retained:")
for grp, genes in marks.items():
    print(f"{grp}: {genes}")


# ------------------------------------------------------------------
# Dot plots
# ------------------------------------------------------------------
for t in TIMES:

    sub = ctrl[
        (ctrl.obs["timepoint_true"].astype(str) == str(t)) &
        (~ctrl.obs["predicted_leiden_sub_joint"]
          .astype(str)
          .isin(DROP_STATES))
    ].copy()

    raw = sub.obs["predicted_leiden_sub_joint"].astype(str)

    counts = raw.value_counts()
    total = len(raw)

    rows = [
        r for r in DIRECTED_ROW_ORDER
        if r in counts.index
    ]

    labels = {
        r: f"{r} (n={counts[r]}, p={counts[r]/total*100:.1f}%)"
        for r in rows
    }

    sub.obs["row"] = pd.Categorical(
        raw.map(labels),
        categories=[labels[r] for r in rows],
        ordered=True
    )

    print(f"{t}: {sub.n_obs:,} cells, {len(rows)} rows")

    dp = sc.pl.dotplot(
        sub,
        marks,
        groupby="row",
        use_raw=False,

        # z/range-scale each gene across gastruloid states
        # makes the spatial/state pattern easy to see
        standard_scale="var",

        dot_max=DOT_MAX,
        dot_min=0.0,
        smallest_dot=SMALLEST_DOT,
        cmap="Reds",
        var_group_rotation=0,

        figsize=(
            GENE_W * n_genes,
            ROW_H * len(rows)
        ),

        title=(
            f"Directed differentiation ↔ gastruloid mapping ({t})"
            if SHOW_TITLES else None
        ),

        return_fig=True
    )

    dp.style(
        dot_edge_color="black",
        dot_edge_lw=0.25,
        largest_dot=LARGEST_DOT
    )

    dp.legend(
        width=1.6,
        colorbar_title="Mean expression\nacross states",
        size_title="Fraction of cells\nexpressing (%)"
    )

    dp.make_figure()
    thin_size_legend(dp, keep=3)

    try:
        mp = dp.get_axes()["mainplot_ax"]

        mp.set_xticklabels(
            [
                x.get_text().split(" (")[0]
                for x in mp.get_xticklabels()
            ],
            rotation=70,
            ha="right"
        )

    except Exception as e:
        print("xlabel skip:", e)

    for ext in SAVE_FORMATS:
        dp.fig.savefig(
            FIG / f"directed_diff_to_gastruloid_{t}.{ext}",
            bbox_inches="tight"
        )

    plt.show()

## Panel D — APLNR / APELA in the embryo

Separate colour scale from the gastruloid: cross-dataset magnitude is not comparable (different normalisation, gene panel, depth). Read which tissues carry signal, not how much.

In [ ]:
tsym2key = dict(zip(tyser.var["symbol"].astype(str), tyser.var.index)) \
           if "symbol" in tyser.var else {g:g for g in tyser.var_names}
emb_marks = {"apelin":[tsym2key[g] for g in GENES if g in tsym2key]}

dp = sc.pl.dotplot(tyser, emb_marks, groupby="tissue_embryo",
    categories_order=present, use_raw=False, standard_scale=None,
    dot_max=DOT_MAX, dot_min=0.0, smallest_dot=SMALLEST_DOT, cmap=DOT_CMAP,
    figsize=(0.55*len(GENES)+2, 0.30*len(present)+1),
    title=("CS7 embryo" if SHOW_TITLES else None), return_fig=True)
dp.style(dot_edge_color="black", dot_edge_lw=0.25, largest_dot=LARGEST_DOT)
dp.legend(width=1.6, colorbar_title="Mean expression", size_title="Fraction of cells (%)")
dp.make_figure(); thin_size_legend(dp, keep=3)
try:
    mp = dp.get_axes()["mainplot_ax"]
    mp.set_xticklabels([g.get_text().split(" (")[0] for g in mp.get_xticklabels()], rotation=0)
except Exception as e: print("  xlabel skip:", e)
for ext in SAVE_FORMATS: dp.fig.savefig(FIG/f"D_embryo_apelin_dotplot.{ext}", bbox_inches="tight")
plt.show()

## Panel E — MAGIC-imputed APLNR / APELA on the control UMAP

Per-timepoint colour scales (APLNR is 1.5% detected at 42h, 23% at 72h). Shows where each gene is, not how much between panels.

In [ ]:
missing = [g for g in GENES if f"{g}_magic" not in ctrl.obs]
if missing: raise KeyError(f"MAGIC columns missing: {missing}")

fig, axes = plt.subplots(len(GENES), 2, figsize=(12, 4.8*len(GENES)))
axes = np.atleast_2d(axes)
for r,g in enumerate(GENES):
    col = ctrl.obs[f"{g}_magic"].to_numpy()
    for c,t in enumerate(TIMES):
        ax = axes[r,c]
        m = (ctrl.obs["timepoint_true"].astype(str)==t).to_numpy()
        vals = col[m]; vmax = max(float(np.nanpercentile(vals,99)), 1e-6)
        ax.scatter(xy[~m,0], xy[~m,1], s=10, c=GREY_CTX, linewidths=0, rasterized=True, zorder=1)
        o = np.argsort(vals)
        sctr = ax.scatter(xy[m][o,0], xy[m][o,1], s=10, c=vals[o], cmap="Reds", #cmap=EXPR_CMAP,
                          vmin=0, vmax=vmax, linewidths=0, rasterized=True, zorder=2)
        ax.text(0.02,0.97, f"{g}  {t}", transform=ax.transAxes, fontsize=LEGEND_FS,
                va="top", ha="left", color=GREY_TEXT)
        ax.set_xticks([]); ax.set_yticks([]); ax.set_frame_on(False)
        ax.set_aspect("equal"); ax.grid(False)
        fig.colorbar(sctr, ax=ax, fraction=0.035, pad=0.02)
        raw = ctrl[m, GK[g]].X
        raw = np.asarray(raw.todense()).ravel() if sp.issparse(raw) else np.asarray(raw).ravel()
        print(f"{g} {t}: {(raw>0).mean()*100:.1f}% detected (raw)")
for ext in SAVE_FORMATS: fig.savefig(FIG/f"E_gastruloid_magic_umap.{ext}", bbox_inches="tight")
plt.show()

In [ ]:
# ── APELA / APLN UMAP (MAGIC coloring, raw detection printed) ─────────────
APELIN_GENES = ["APELA", "APLN"]   # adjust if you want APLNR too

# check magic columns exist
missing = [g for g in APELIN_GENES if f"{g}_magic" not in ctrl.obs]
if missing:
    # build them on the fly from X (MAGIC matrix)
    for g in missing:
        idx = ctrl.var_names.get_loc(sym_index(ctrl, g))
        vals = np.asarray(ctrl.X[:, idx].todense()).ravel() \
               if sp.issparse(ctrl.X) else np.asarray(ctrl.X[:, idx]).ravel()
        ctrl.obs[f"{g}_magic"] = vals
    print(f"Built magic cols from ctrl.X for: {missing}")

GK_AP = {g: sym_index(ctrl, g) for g in APELIN_GENES}

# one global vmax across both genes and both timepoints
all_vals = np.concatenate([ctrl.obs[f"{g}_magic"].to_numpy() for g in APELIN_GENES])
vmax_global = max(float(np.nanpercentile(all_vals, 99)), 1e-6)

fig, axes = plt.subplots(len(APELIN_GENES), 2,
                         figsize=(12, 4.8 * len(APELIN_GENES)))
axes = np.atleast_2d(axes)

for r, g in enumerate(APELIN_GENES):
    col = ctrl.obs[f"{g}_magic"].to_numpy()
    for c, t in enumerate(TIMES):
        ax = axes[r, c]
        m    = (ctrl.obs["timepoint_true"].astype(str) == t).to_numpy()
        vals = col[m]

        ax.scatter(xy[~m, 0], xy[~m, 1], s=10, c=GREY_CTX,
                   linewidths=0, rasterized=True, zorder=1)
        o    = np.argsort(vals)
        sctr = ax.scatter(xy[m][o, 0], xy[m][o, 1], s=10,
                          c=vals[o], cmap="Reds",
                          vmin=0, vmax=vmax_global,   # ← same for all panels
                          linewidths=0, rasterized=True, zorder=2)

        ax.text(0.02, 0.97, f"{g}  {t}", transform=ax.transAxes,
                fontsize=LEGEND_FS, va="top", ha="left", color=GREY_TEXT)
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_frame_on(False); ax.set_aspect("equal"); ax.grid(False)
        fig.colorbar(sctr, ax=ax, fraction=0.035, pad=0.02)

        # raw detection
        raw = ctrl[m, GK_AP[g]].layers["counts"]
        raw = np.asarray(raw.todense()).ravel() if sp.issparse(raw) else np.asarray(raw).ravel()
        print(f"{g} {t}: {(raw > 0).mean()*100:.1f}% detected (raw)")

fig.tight_layout()
for ext in SAVE_FORMATS:
    fig.savefig(FIG / f"E_apelin_magic_umap.{ext}", bbox_inches="tight")
plt.show()

In [ ]:
# ---- APLNR + APELA (MAGIC) on the combined UMAP (all cells) ----
GENES = ["APLNR", "APELA"]
missing = [g for g in GENES if f"{g}_magic" not in ctrl.obs]
if missing: raise KeyError(f"MAGIC columns missing: {missing}")
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, g in zip(axes, GENES):
    col = ctrl.obs[f"{g}_magic"].to_numpy()
    vmax = max(float(np.nanpercentile(col, 99)), 1e-6)
    o = np.argsort(col)                                   # weak points first, high on top
    sctr = ax.scatter(xy[o,0], xy[o,1], s=8, c=col[o], cmap="Reds",
                      vmin=0, vmax=vmax, linewidths=0, rasterized=True)
    ax.text(0.02, 0.97, g, transform=ax.transAxes, fontsize=LEGEND_FS, va="top", ha="left", color=GREY_TEXT)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_frame_on(False); ax.set_aspect("equal"); ax.grid(False)
    fig.colorbar(sctr, ax=ax, fraction=0.035, pad=0.02, label=f"{g} (MAGIC)")
    raw = ctrl[:, GK[g]].X
    raw = np.asarray(raw.todense()).ravel() if sp.issparse(raw) else np.asarray(raw).ravel()
    print(f"{g}: {(raw>0).mean()*100:.1f}% detected (raw)")
for ext in SAVE_FORMATS: fig.savefig(FIG/f"E_aplnr_apela_umap.{ext}", bbox_inches="tight")
plt.show()

In [ ]:
# ---- TBXT + SOX2 (raw, no MAGIC) on the control UMAP ----
from matplotlib.colors import PowerNorm

# ctrl.var_names may be bare symbols or "SYMBOL (ENSG...)" -- handle both
_syms = {str(v).split(" (")[0]: v for v in ctrl.var_names}
GENES = ["TBXT", "SOX2"]
GENES = [g if g in _syms else ("T" if g == "TBXT" and "T" in _syms else g) for g in GENES]
missing = [g for g in GENES if g not in _syms]
if missing:
    raise KeyError(f"not in var_names: {missing}. "
                   f"candidates: {[s for s in _syms if s.startswith(('TBX','SOX'))][:20]}")

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, g in zip(axes, GENES):
    x = ctrl[:, _syms[g]].X
    col = np.asarray(x.todense()).ravel() if sp.issparse(x) else np.asarray(x).ravel()
    vmax = max(float(np.nanpercentile(col, 99)), 1e-6)
    norm = PowerNorm(gamma=0.5, vmin=0, vmax=vmax)     # lifts low end; drop for linear
    o = np.argsort(col)
    sctr = ax.scatter(xy[o,0], xy[o,1], s=8, c=col[o], cmap="Reds",
                      norm=norm, linewidths=0, rasterized=True)
    ax.text(0.02, 0.97, g, transform=ax.transAxes, fontsize=LEGEND_FS, va="top", ha="left", color=GREY_TEXT)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_frame_on(False); ax.set_aspect("equal"); ax.grid(False)
    fig.colorbar(sctr, ax=ax, fraction=0.035, pad=0.02, label=f"{g} log1p(CP10K)")
    print(f"{g}: {(col>0).mean()*100:.1f}% detected (raw), max {col.max():.2f}")
for ext in SAVE_FORMATS: fig.savefig(FIG/f"E_tbxt_sox2_umap.{ext}", bbox_inches="tight")
plt.show()

In [ ]:
# ---- TBXT + SOX2 (raw, no MAGIC) on the control UMAP, split by timepoint ----
from matplotlib.colors import PowerNorm

_syms = {str(v).split(" (")[0]: v for v in ctrl.var_names}
GENES = ["TBXT", "SOX2"]
GENES = [g if g in _syms else ("T" if g == "TBXT" and "T" in _syms else g) for g in GENES]
missing = [g for g in GENES if g not in _syms]
if missing:
    raise KeyError(f"not in var_names: {missing}. "
                   f"candidates: {[s for s in _syms if s.startswith(('TBX','SOX'))][:20]}")

TPS = ["42h", "72h"]
tp = ctrl.obs["timepoint_true"].astype(str).to_numpy()
assert xy.shape[0] == ctrl.n_obs, "xy does not match ctrl"
print({t: int((tp == t).sum()) for t in np.unique(tp)})

# pull expression once per gene, and set vmax ACROSS timepoints so the two
# columns are on one scale -- per-panel vmax would make 42h and 72h incomparable
EXPR, VMAX = {}, {}
for g in GENES:
    x = ctrl[:, _syms[g]].X
    col = np.asarray(x.todense()).ravel() if sp.issparse(x) else np.asarray(x).ravel()
    EXPR[g] = col
    VMAX[g] = max(float(np.nanpercentile(col[np.isin(tp, TPS)], 99)), 1e-6)

fig, axes = plt.subplots(len(GENES), len(TPS), figsize=(5.5 * len(TPS), 5 * len(GENES)))
axes = np.atleast_2d(axes)
for i, g in enumerate(GENES):
    norm = PowerNorm(gamma=0.5, vmin=0, vmax=VMAX[g])
    for j, t in enumerate(TPS):
        ax = axes[i, j]
        m = tp == t
        ax.scatter(xy[:, 0], xy[:, 1], s=6, color="#e8e8e6", linewidths=0, rasterized=True)
        o = np.argsort(EXPR[g][m])                      # weak first, high on top
        sctr = ax.scatter(xy[m][o, 0], xy[m][o, 1], s=8, c=EXPR[g][m][o], cmap="Reds",
                          norm=norm, linewidths=0, rasterized=True)
        ax.text(0.02, 0.97, f"{g}  {t}", transform=ax.transAxes, fontsize=LEGEND_FS,
                va="top", ha="left", color=GREY_TEXT)
        ax.set_xticks([]); ax.set_yticks([]); ax.set_frame_on(False)
        ax.set_aspect("equal"); ax.grid(False)
        if j == len(TPS) - 1:
            fig.colorbar(sctr, ax=ax, fraction=0.035, pad=0.02, label=f"{g} log1p(CP10K)")
        print(f"{g} {t}: {(EXPR[g][m]>0).mean()*100:.1f}% detected, n={m.sum():,}")
for ext in SAVE_FORMATS: fig.savefig(FIG/f"E_tbxt_sox2_umap_bytime.{ext}", bbox_inches="tight")
plt.show()

In [ ]:
from matplotlib.colors import PowerNorm      # add at top

# ---- APLNR + APELA (MAGIC) montage: combined | 42h | 72h, shared scale per gene ----
GENES = ["APLNR", "APELA", "APLN"]; TIMES = ["42h", "72h"]
PANELS = ["all"] + TIMES                                       # combined first
missing = [g for g in GENES if f"{g}_magic" not in ctrl.obs]
if missing: raise KeyError(f"MAGIC columns missing: {missing}")
fig, axes = plt.subplots(len(GENES), len(PANELS), figsize=(4.2*len(PANELS), 4.2*len(GENES)))
axes = np.atleast_2d(axes)
for r, g in enumerate(GENES):
    col  = ctrl.obs[f"{g}_magic"].to_numpy()
    vmax = max(float(np.nanpercentile(col, 95)), 1e-6)          # 95th (was 99) → more color range
    norm = PowerNorm(gamma=0.5, vmin=0, vmax=vmax)              # gamma<1 brightens the low end
    # col  = ctrl.obs[f"{g}_magic"].to_numpy()
    # vmax = max(float(np.nanpercentile(col, 99)), 1e-6)         # shared per gene
    for c, p in enumerate(PANELS):
        ax = axes[r, c]
        if p == "all":
            o = np.argsort(col)
            sctr = ax.scatter(xy[o,0], xy[o,1], s=8, c=col[o], cmap="Reds",
                              norm=norm, linewidths=0, rasterized=True)
            lab = f"{g}  all"
        else:
            m = (ctrl.obs["timepoint_true"].astype(str) == p).to_numpy()
            ax.scatter(xy[~m,0], xy[~m,1], s=8, c=GREY_CTX, linewidths=0, rasterized=True, zorder=1)
            o = np.argsort(col[m])
            sctr = ax.scatter(xy[m][o,0], xy[m][o,1], s=8, c=col[m][o], cmap="Reds",
                              norm=norm, linewidths=0, rasterized=True, zorder=2)
            lab = f"{g}  {p}"
        ax.text(0.02, 0.97, lab, transform=ax.transAxes, fontsize=LEGEND_FS,
                va="top", ha="left", color=GREY_TEXT)
        ax.set_xticks([]); ax.set_yticks([]); ax.set_frame_on(False); ax.set_aspect("equal")
    fig.colorbar(sctr, ax=axes[r,:].tolist(), fraction=0.02, pad=0.02, label=f"{g} (MAGIC)")
for ext in SAVE_FORMATS: fig.savefig(FIG/f"E_aplnr_apela_montage.{ext}", bbox_inches="tight")
plt.show()

## Panel G — control tissue composition

In [ ]:
# overwrite ctrl's tissue with the refined labels from adata
ctrl.obs['tissue'] = adata[ctrl.obs.index, :].obs['predicted_leiden_sub_joint']
comp = (ctrl.obs.groupby(["timepoint_true","tissue"], observed=True)
        .size().rename("n").reset_index())
comp["fraction"] = comp["n"]/comp.groupby("timepoint_true")["n"].transform("sum")
comp.to_csv(TAB/"G_composition.csv", index=False)

fig, ax = plt.subplots(figsize=(5.5, 5.5))
bottom = np.zeros(len(TIMES))
for c in cats:
    vals = [float(comp[(comp.timepoint_true==t)&(comp.tissue==c)]["fraction"].sum()) for t in TIMES]
    ax.bar(np.arange(len(TIMES)), vals, bottom=bottom, color=TISSUE_COLORS[c],
           width=0.6, edgecolor="white", linewidth=0.8)
    bottom += np.array(vals)
ax.set_xticks(np.arange(len(TIMES))); ax.set_xticklabels(TIMES, fontsize=LEGEND_FS)
ax.set_ylabel("fraction of cells", fontsize=13); ax.set_ylim(0,1); ax.grid(False)
side_legend(ax, cats, TISSUE_COLORS)
ax.spines[["top","right"]].set_visible(False)
fig.tight_layout()
for ext in SAVE_FORMATS: fig.savefig(FIG/f"G_composition.{ext}", bbox_inches="tight")
plt.show()
print("\nPanels:")
for p in sorted(FIG.glob("*.svg")): print("  ", p.name)

In [ ]:
# ── APELA transcript contribution by cell state ───────────────────────────
# Uses: ctrl (filtered control adata), TISSUE_COLORS, TISSUE_ORDER, TIMES,
#       sym_index, side_legend, LEGEND_FS, TAB, FIG, SAVE_FORMATS
# "To quantify the cellular origin of APELA ligand, raw UMI counts for APELA were summed per 
# tissue type and expressed as a fraction of total APELA counts detected at each timepoint."
# 1. Get APELA raw counts per cell
#    ctrl.X is MAGIC-imputed — use the raw counts layer instead

TISSUE_ORDER = [
    'pluripotent',
    'AE-LC',
    'pPS-LC',
    'aPS-LC',
    'eNasM',
    'lNasM',
    'DE',
    'PGC-LC',
    'aPM-LC',
    'CS-LC',
    'LPM-LC',
    'stressed',
]
APELA_LAYER = "counts"          # adjust if your raw layer has a different name
apela_key   = sym_index(ctrl, "APELA")
raw_mat     = ctrl.layers[APELA_LAYER]
ctrl.obs['tissue'] = adata.obs.loc[ctrl.obs.index, 'predicted_leiden_sub_joint'].astype(str)

apela_vals  = (np.asarray(raw_mat[:, ctrl.var_names.get_loc(apela_key)].todense()).flatten()
               if sp.issparse(raw_mat)
               else np.asarray(raw_mat[:, ctrl.var_names.get_loc(apela_key)]).flatten())

df_ap = ctrl.obs[["timepoint_true", "tissue"]].copy()
df_ap["apela"] = apela_vals

# 2. Fraction of total APELA transcripts per timepoint × tissue
comp_ap = (df_ap.groupby(["timepoint_true", "tissue"], observed=False)["apela"]
              .sum().rename("sum_apela").reset_index())
comp_ap["fraction"] = (comp_ap["sum_apela"]
                       / comp_ap.groupby("timepoint_true")["sum_apela"].transform("sum"))
comp_ap.to_csv(TAB / "G_apela_contribution.csv", index=False)


# 3. Plot — same style as composition bar
cats = [c for c in TISSUE_ORDER if c in df_ap["tissue"].unique()]

fig, ax = plt.subplots(figsize=(5.5, 5.5))
bottom = np.zeros(len(TIMES))

for c in cats:
    vals = [float(comp_ap[(comp_ap.timepoint_true == t) &
                           (comp_ap.tissue == c)]["fraction"].sum())
            for t in TIMES]
    ax.bar(np.arange(len(TIMES)), vals, bottom=bottom,
           color=TISSUE_COLORS[c], width=0.6, edgecolor="white", linewidth=0.8)
    # label segments ≥ 5%
    # for i, (v, b) in enumerate(zip(vals, bottom)):
    #     if v >= 0.05:
    #         ax.text(i, b + v / 2, f"{100*v:.0f}%",
    #                 ha="center", va="center",
    #                 fontsize=9, color="white", fontweight="bold")
    bottom += np.array(vals)

ax.set_xticks(np.arange(len(TIMES)))
ax.set_xticklabels(TIMES, fontsize=LEGEND_FS)
ax.set_ylabel("fraction of APELA transcripts\n(raw counts)", fontsize=13)
ax.set_ylim(0, 1)
ax.grid(False)
ax.spines[["top", "right"]].set_visible(False)
side_legend(ax, cats, TISSUE_COLORS)

fig.tight_layout()
for ext in SAVE_FORMATS:
    fig.savefig(FIG / f"G_apela_contribution.{ext}", bbox_inches="tight")
plt.show()

In [ ]:
# ── APELA vs APLN absolute CPM — gastruloid (42h, 72h) + CS7 + CS8 ────────
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import matplotlib.patches as mpatches, matplotlib.ticker as mticker
import scipy.sparse as sp

# ── colors & sizing ────────────────────────────────────────────────────────
APELA_COL = "#000000"
APLN_COL  = "#A5A4A4"
BAR_W     = 0.35
FMT       = mticker.FuncFormatter(
    lambda v, _: f"{v:.1e}".replace("e+0", "×10^").replace("e+", "×10^"))

# ── gastruloid data ────────────────────────────────────────────────────────
LIGANDS = {"APELA": sym_index(ctrl, "APELA"), "APLN": sym_index(ctrl, "APLN")}
DROP    = {"DE", "CS-LC", "LPM-LC", "aPM-LC", "stressed", "other"}
raw_mat = ctrl.layers["counts"]
tot     = np.asarray(raw_mat.sum(axis=1)).flatten()
for sym, key in LIGANDS.items():
    col  = raw_mat[:, ctrl.var_names.get_loc(key)]
    vals = np.asarray(col.todense()).flatten() if sp.issparse(raw_mat) else np.asarray(col).flatten()
    ctrl.obs[f"_{sym}_cpm"] = 1e6 * vals / np.where(tot > 0, tot, 1)

cats = [c for c in TISSUE_ORDER if c in ctrl.obs["tissue"].cat.categories and c not in DROP]
rows = []
for t in TIMES:
    for tissue in cats:
        sub = ctrl.obs[(ctrl.obs["timepoint_true"].astype(str)==str(t)) & (ctrl.obs["tissue"]==tissue)]
        rows.append({"tp": t, "tissue": tissue,
                     "APELA": sub["_APELA_cpm"].sum(), "APLN": sub["_APLN_cpm"].sum()})
sf = pd.DataFrame(rows)

# ── CS7 embryo data ────────────────────────────────────────────────────────
apela_k = tsym2key_full.get("APELA")
apln_k  = tsym2key_full.get("APLN")
def get_cpm(adata, key):
    X = adata.X
    col = X[:, adata.var_names.get_loc(key)]
    vals = np.asarray(col.todense()).flatten() if sp.issparse(X) else np.asarray(col).flatten()
    return 1e6 * vals / np.where(np.asarray(X.sum(axis=1)).flatten() > 0,
                                  np.asarray(X.sum(axis=1)).flatten(), 1)

tyser_full.obs["_APELA_cpm"] = get_cpm(tyser_full, apela_k)
tyser_full.obs["_APLN_cpm"]  = get_cpm(tyser_full, apln_k)
rows_ty = []
for ct in present_full:
    sub = tyser_full.obs[tyser_full.obs["tissue_embryo"] == ct]
    rows_ty.append({"tissue": ct, "APELA": sub["_APELA_cpm"].sum(), "APLN": sub["_APLN_cpm"].sum()})
sf_ty = pd.DataFrame(rows_ty)

CS7_LABELS = {"Epiblast":"Epi","Primitive Streak":"PS","Nascent Mesoderm":"N.Meso",
              "Emergent Mesoderm":"Em.Meso","Advanced Mesoderm":"Adv.Meso",
              "Axial Mesoderm":"Ax.Meso","ExE Mesoderm":"ExE.Meso",
              "Endoderm":"Endo","Non-Neural Ectoderm":"N.N.Ecto"}

# ── CS8 spatial data ───────────────────────────────────────────────────────
CS8_ORDER = ["Epi/Ecto","Gast/PS","Meso","AM","AM.EXE.Meso","Noto","Endo","HEP","Ery"]
cs8 = pd.DataFrame({
    "cluster": ["AM","AM.EXE.Meso","Endo","Epi/Ecto","Ery","Gast/PS",
                "HEP","Meso","Noto","Visceral.Endo","YS.EXE.Meso-A","YS.EXE.Meso-B","YS.Endo"],
    "APELA":   [21687.0,38634.9,284916.5,128758.7,2802.5,47526.2,2832.6,64270.4,5377.4,49294.5,48477.6,18143.1,18364.9],
    "APLN":    [1026.9,9520.6,6065.9,7865.3,19082.6,5298.8,11990.8,11255.8,0.0,7528.6,33643.4,30074.8,21032.1],
}).set_index("cluster")
present_cs8 = [c for c in CS8_ORDER if c in cs8.index]

# ── 4-panel figure ─────────────────────────────────────────────────────────
panels = [
    ("42h",     cats,           sf[sf["tp"]=="42h"].set_index("tissue").reindex(cats),   "Total CPM",        None),
    ("72h",     cats,           sf[sf["tp"]=="72h"].set_index("tissue").reindex(cats),   None,               None),
    ("CS7",     present_full,   sf_ty.set_index("tissue").reindex(present_full),          "Total CPM",        CS7_LABELS),
    ("CS8",     present_cs8,    cs8.reindex(present_cs8),                                 "Total CPM (spots)", None),
]
n_panels  = len(panels)
widths    = [len(p[1]) for p in panels]
fig, axes = plt.subplots(1, n_panels,
                         figsize=(sum(widths) * 0.55 + 1.0, 4.0),
                         gridspec_kw={"width_ratios": widths})

for ax, (title, labels, data, ylabel, label_map) in zip(axes, panels):
    x = np.arange(len(labels))
    ax.bar(x - BAR_W/2, data["APELA"], BAR_W, color=APELA_COL, zorder=3)
    ax.bar(x + BAR_W/2, data["APLN"],  BAR_W, color=APLN_COL,  zorder=3)
    ax.set_xticks(x)
    xlbls = [label_map.get(c, c) if label_map else c for c in labels]
    ax.set_xticklabels(xlbls, rotation=45, ha="right", fontsize=LEGEND_FS)
    if SHOW_TITLES: ax.set_title(title, loc="left", fontsize=LEGEND_FS + 2)
    if ylabel: ax.set_ylabel(ylabel, fontsize=LEGEND_FS + 1)
    ax.yaxis.set_major_formatter(FMT)
    ax.spines[["top","right"]].set_visible(False)
    ax.yaxis.grid(False); ax.xaxis.grid(False)
    ax.margins(x=0.06)

legend_handles = [mpatches.Patch(facecolor=APELA_COL, label="APELA"),
                  mpatches.Patch(facecolor=APLN_COL,  label="APLN")]
axes[-1].legend(handles=legend_handles, frameon=False, fontsize=LEGEND_FS,
                loc="upper right", handlelength=1.2)

fig.tight_layout()
SAVE_FORMATS_SVG = list(SAVE_FORMATS) + ["svg"] if "svg" not in SAVE_FORMATS else SAVE_FORMATS
for ext in SAVE_FORMATS_SVG:
    fig.savefig(FIG / f"G_apela_apln_all4.{ext}", bbox_inches="tight")
plt.show()

In [ ]:
# ── APELA vs APLN ligand share (% of combined CPM pool) ───────────────────
# Uses: ctrl, TISSUE_COLORS, TISSUE_ORDER, TIMES, sym_index,
#       side_legend, LEGEND_FS, FIG, SAVE_FORMATS

import numpy as np, pandas as pd, matplotlib.pyplot as plt
import matplotlib.patches as mpatches, scipy.sparse as sp

LIGANDS = {"APELA": sym_index(ctrl, "APELA"), "APLN": sym_index(ctrl, "APLN")}

raw_mat = ctrl.layers["counts"]
tot     = np.asarray(raw_mat.sum(axis=1)).flatten()

for sym, key in LIGANDS.items():
    col  = raw_mat[:, ctrl.var_names.get_loc(key)]
    vals = np.asarray(col.todense()).flatten() if sp.issparse(raw_mat) else np.asarray(col).flatten()
    ctrl.obs[f"_{sym}_cpm"] = 1e6 * vals / np.where(tot > 0, tot, 1)

DROP_SHARE = {"DE", "CS-LC", "LPM-LC", "aPM-LC", "stressed", "other"}   # remove these
cats = [c for c in TISSUE_ORDER
        if c in ctrl.obs["tissue"].cat.categories and c not in DROP_SHARE]

rows = []
for t in TIMES:
    for tissue in cats:
        sub = ctrl.obs[
            (ctrl.obs["timepoint_true"].astype(str) == str(t)) &
            (ctrl.obs["tissue"] == tissue)
        ]
        a = sub["_APELA_cpm"].sum()
        n = sub["_APLN_cpm"].sum()
        total = a + n
        rows.append({
            "tp": t, "tissue": tissue,
            "apela_share": 100 * a / total if total > 0 else 0,
            "apln_share":  100 * n / total if total > 0 else 0,
        })
df_share = pd.DataFrame(rows)

APELA_COL = "#E07B39"   # your warm orange
APLN_COL  = "#CCCCCC"   # light grey
BAR_W = 0.55

fig, axes = plt.subplots(1, len(TIMES), figsize=(5 * len(TIMES), 5.5), sharey=True)
if len(TIMES) == 1:
    axes = [axes]

for ax, t in zip(axes, TIMES):
    sub = df_share[df_share["tp"] == t]
    tissues_here = [c for c in cats if c in sub["tissue"].values]
    x = np.arange(len(tissues_here))

    apela_vals = [float(sub[sub["tissue"] == c]["apela_share"].values[0]) for c in tissues_here]
    apln_vals  = [float(sub[sub["tissue"] == c]["apln_share"].values[0])  for c in tissues_here]

    ax.bar(x, apela_vals, BAR_W, color=APELA_COL,
           zorder=3, edgecolor="white", linewidth=0.5)
    ax.bar(x, apln_vals,  BAR_W, bottom=apela_vals, color=APLN_COL,
           zorder=3, edgecolor="white", linewidth=0.5)

    ax.set_xticks(x)
    ax.set_xticklabels(tissues_here, rotation=45, ha="right", fontsize=LEGEND_FS)
    ax.set_title(str(t), fontsize=LEGEND_FS + 2, loc="left")
    ax.set_ylim(0, 100)
    ax.spines[["top", "right"]].set_visible(False)
    ax.margins(x=0.08)

axes[0].set_ylabel("% of APELA + APLN CPM pool", fontsize=LEGEND_FS + 1)

legend_handles = [
    mpatches.Patch(facecolor=APELA_COL, label="APELA"),
    mpatches.Patch(facecolor=APLN_COL,  label="APLN"),
]
axes[-1].legend(handles=legend_handles, frameon=False,
                fontsize=LEGEND_FS, loc="lower right", handlelength=1.2)

fig.tight_layout()
for ext in SAVE_FORMATS:
    fig.savefig(FIG / f"G_apela_apln_ligand_share.{ext}", bbox_inches="tight")
plt.show()

In [ ]:
"""
UNBIASED hierarchical clustering of the ANNOTATED TISSUES (not genes, not the KD).
Shows how the cell states relate in the data: pseudobulk each tissue, cluster on
HVG expression, draw a dendrogram + tissue x tissue correlation heatmap.

This is a structure/relatedness supplement -- it should recover the expected biology
(streak states together, NasM together, DE/AE apart, pluripotent as an outgroup).
It uses NO KD contrast; by default it uses CONTROL cells only so the picture is the
native developmental structure, not something the perturbation moved. Set
CONTROL_ONLY = False to use all cells.

Run in the pseudobulk env:  python tissue_dendrogram.py
Needs: numpy, pandas, h5py, scipy, matplotlib   (no scanpy).
"""
import json
import numpy as np, pandas as pd, h5py, scipy.sparse as sp
from pathlib import Path
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import squareform
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex

MASTER   = ROOT / "00_DATA_FROZEN" / "03_analysis_ready" / "adata_master_normalized_magic.h5ad"
TAB      = ROOT / "06_tables" / "FIGURE_KD_clean"; TAB.mkdir(parents=True, exist_ok=True)
FIG      = ROOT / "07_figures" / "FIGURE2_clean"; FIG.mkdir(parents=True, exist_ok=True)

TISSUE_COL   = "predicted_leiden_sub_joint"
CONTROL_ONLY = True     # native structure, not KD-moved. False = all cells.
MERGE_NASM   = False    # keep eNasM / lNasM separate so you can SEE they cluster together
N_HVG        = 2000     # top variable genes across the tissue pseudobulks
MIN_CELLS    = 30       # drop a tissue with fewer cells than this (too noisy to place)
METHOD       = "ward"   # ward on correlation-distance is the usual choice for this plot
DROP         = ["stressed"]   # exclude junk/QC clusters from the structure plot


def _rc(it):
    if isinstance(it, h5py.Group):
        ks = set(it.keys())
        if {"categories", "codes"} <= ks:
            return pd.Categorical.from_codes(it["codes"][:], categories=np.asarray(_rc(it["categories"])))
        if "values" in ks:
            v = it["values"][:]
            return (np.array([x.decode() if isinstance(x, bytes) else x for x in v], dtype=object)
                    if v.dtype == object or v.dtype.kind in "SU" else v)
    a = it[:]
    return a.astype(str) if a.dtype.kind == "S" else a


# ---------- load counts + obs ----------
with h5py.File(MASTER, "r") as h:
    v = h["var"]; ik = v.attrs.get("_index", "_index"); ik = ik.decode() if isinstance(ik, bytes) else ik
    vit = v[ik]; vraw = vit["values"][:] if isinstance(vit, h5py.Group) else vit[:]
    var = np.array([(x.decode() if isinstance(x, bytes) else str(x)).split(" (")[0] for x in vraw])
    X = h["layers"]["counts"]
    M = sp.csr_matrix((X["data"][:], X["indices"][:], X["indptr"][:]),
                      shape=tuple(int(s) for s in X.attrs["shape"]))
    og = h["obs"]
    obs = pd.DataFrame({c: np.asarray(_rc(og[c])).astype(str)
                        for c in ["condition", TISSUE_COL]})
obs["cond"] = obs["condition"].map({"Ctrl": "Ctrl", "DOX": "KD"})
if MERGE_NASM:
    obs[TISSUE_COL] = obs[TISSUE_COL].replace({"eNasM": "NasM", "lNasM": "NasM"})

keep = ~obs[TISSUE_COL].isin(DROP)
if CONTROL_ONLY:
    keep &= (obs["cond"] == "Ctrl")
obs = obs[keep].reset_index(drop=True)
M = M[keep.to_numpy()]

# ---------- per-cell log-norm (CP10k, log1p) then pseudobulk mean per tissue ----------
lib = np.asarray(M.sum(1)).ravel(); lib[lib == 0] = 1
Ln = M.multiply(1.0 / lib[:, None]).tocsr(); Ln.data = np.log1p(Ln.data * 1e4)

tissues, profiles, ncells = [], [], []
for t, idx in obs.groupby(TISSUE_COL).indices.items():
    if len(idx) < MIN_CELLS:
        print(f"drop {t}: only {len(idx)} cells")
        continue
    tissues.append(t); ncells.append(len(idx))
    profiles.append(np.asarray(Ln[idx].mean(0)).ravel())
P = np.vstack(profiles)                                    # tissues x genes, mean log-norm
print(f"{len(tissues)} tissues x {P.shape[1]} genes | cells: {dict(zip(tissues, ncells))}")

# ---------- pick HVGs across the tissue profiles, then z-score genes ----------
hvg = np.argsort(P.var(0))[::-1][:N_HVG]
Z = P[:, hvg]
Z = (Z - Z.mean(0)) / (Z.std(0) + 1e-9)                    # z per gene so all genes count equally

# ---------- correlation-distance + hierarchical clustering ----------
C = np.corrcoef(Z)                                         # tissue x tissue correlation
D = 1.0 - C
np.fill_diagonal(D, 0.0)
Lk = linkage(squareform(D, checks=False), method=METHOD)

# ---------- colors ----------
tcol = TISSUE_COLORS   # from palettes.py (was a silently-failing json load)
def col(t): return tcol.get(t, "#666666")

# ---------- plot: dendrogram (left) + ordered correlation heatmap (right) ----------
fig, (axd, axh) = plt.subplots(1, 2, figsize=(11, 5.2),
                               gridspec_kw={"width_ratios": [1, 1.25], "wspace": 0.05})

dn = dendrogram(Lk, labels=tissues, ax=axd, orientation="left",
                color_threshold=0, above_threshold_color="#444444")
axd.set_title("Tissue relatedness (Ward, 1-corr)", fontsize=10)
axd.set_xlabel("distance"); axd.spines[["top", "right"]].set_visible(False)
for lbl in axd.get_yticklabels():
    lbl.set_color(col(lbl.get_text())); lbl.set_fontweight("bold")

order = dn["leaves"][::-1]                                 # match heatmap to dendrogram order
Co = C[np.ix_(order, order)]
labs = [tissues[i] for i in order]
im = axh.imshow(Co, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
axh.set_xticks(range(len(labs))); axh.set_xticklabels(labs, rotation=90, fontsize=8)
axh.set_yticks(range(len(labs))); axh.set_yticklabels(labs, fontsize=8)
for i, t in enumerate(labs):
    axh.get_xticklabels()[i].set_color(col(t))
    axh.get_yticklabels()[i].set_color(col(t))
axh.set_title("Pairwise correlation", fontsize=10)
fig.colorbar(im, ax=axh, fraction=0.046, pad=0.04, label="Pearson r")

sub = "control cells only" if CONTROL_ONLY else "all cells"
fig.suptitle(f"Unbiased hierarchical clustering of annotated tissues ({sub}, top {N_HVG} HVG)",
             fontsize=11, y=1.02)
fig.tight_layout()
out = FIG / f"tissue_dendrogram_{'ctrl' if CONTROL_ONLY else 'all'}.png"
fig.savefig(out, dpi=200, bbox_inches="tight"); print("saved", out)

# ---------- save the numbers too ----------
pd.DataFrame(C, index=tissues, columns=tissues).to_csv(TAB / "tissue_correlation_matrix.csv")
pd.DataFrame({"leaf_order_top_to_bottom": labs}).to_csv(TAB / "tissue_dendrogram_order.csv", index=False)
print("saved tissue_correlation_matrix.csv, tissue_dendrogram_order.csv")



In [ ]:
"""
Marker heatmap, columns ORDERED by tissue (not clustered) in a developmental order.
Rows = top one-vs-rest markers per tissue (each gene assigned to its most-enriched
tissue, so no gene appears twice). This is the clean "block-diagonal" supplement for
CONTINUOUS states, where unsupervised cell clustering can't give crisp blocks.

Control cells only. Columns downsampled to equal width per tissue and ordered by ORDER;
within a tissue, cells are sorted by that tissue's own marker score (gives a gradient).
Values are row z-scored log-norm expression.

Run in the pseudobulk env:  python marker_heatmap_ordered.py
Needs: numpy, pandas, h5py, scipy, matplotlib   (no seaborn, no scanpy).
"""
import json
import numpy as np, pandas as pd, h5py, scipy.sparse as sp
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.colors import ListedColormap, BoundaryNorm

MASTER   = ROOT / "00_DATA_FROZEN" / "03_analysis_ready" / "adata_master_normalized_magic.h5ad"
FIG      = ROOT / "07_figures" / "FIGURE2_clean"; FIG.mkdir(parents=True, exist_ok=True)
TAB      = ROOT / "06_tables" / "FIGURE_KD_clean"; TAB.mkdir(parents=True, exist_ok=True)

TISSUE_COL   = "predicted_leiden_sub_joint"

# ---- EDIT THIS to your exact label meanings / preferred early->late order ----
# stem/epiblast -> posterior then anterior streak -> nascent mesoderm early->late
# -> paraxial / lateral mesoderm -> endoderm / ectoderm / germline branches.
ORDER = ["pluripotent", "CS-LC", "pPS-LC", "aPS-LC", "eNasM", "lNasM",
         "aPM-LC", "LPM-LC", "DE", "AE-LC", "PGC-LC"]
DROP         = ["stressed"]
N_MARKERS    = 8        # top markers per tissue (rows)
N_PER_TISSUE = 60       # cells per tissue (columns), equal width
MIN_PCT      = 0.10     # a marker must be expressed in >=10% of its tissue's cells
SEED         = 0
ZCLIP        = 2.5
SHOW_GENES   = True     # label rows with gene names (readable at ~8/tissue)


def _rc(it):
    if isinstance(it, h5py.Group):
        ks = set(it.keys())
        if {"categories", "codes"} <= ks:
            return pd.Categorical.from_codes(it["codes"][:], categories=np.asarray(_rc(it["categories"])))
        if "values" in ks:
            v = it["values"][:]
            return (np.array([x.decode() if isinstance(x, bytes) else x for x in v], dtype=object)
                    if v.dtype == object or v.dtype.kind in "SU" else v)
    a = it[:]
    return a.astype(str) if a.dtype.kind == "S" else a


# ---------- load counts + obs (control only) ----------
with h5py.File(MASTER, "r") as h:
    v = h["var"]; ik = v.attrs.get("_index", "_index"); ik = ik.decode() if isinstance(ik, bytes) else ik
    vit = v[ik]; vraw = vit["values"][:] if isinstance(vit, h5py.Group) else vit[:]
    var = np.array([(x.decode() if isinstance(x, bytes) else str(x)).split(" (")[0] for x in vraw])
    X = h["layers"]["counts"]
    M = sp.csr_matrix((X["data"][:], X["indices"][:], X["indptr"][:]),
                      shape=tuple(int(s) for s in X.attrs["shape"]))
    og = h["obs"]
    obs = pd.DataFrame({c: np.asarray(_rc(og[c])).astype(str)
                        for c in ["condition", TISSUE_COL]})
obs["cond"] = obs["condition"].map({"Ctrl": "Ctrl", "DOX": "KD"})
keep = (obs["cond"] == "Ctrl") & (~obs[TISSUE_COL].isin(DROP))
obs = obs[keep].reset_index(drop=True); M = M[keep.to_numpy()]

# keep only tissues that exist, in ORDER
present = [t for t in ORDER if (obs[TISSUE_COL] == t).any()]
missing_from_order = [t for t in obs[TISSUE_COL].unique() if t not in ORDER and t not in DROP]
if missing_from_order:
    print("NOT in ORDER (append or ignore):", missing_from_order)
tissues = present
print("tissue order:", tissues)

# ---------- log-norm ----------
lib = np.asarray(M.sum(1)).ravel(); lib[lib == 0] = 1
Ln = M.multiply(1.0 / lib[:, None]).tocsr(); Ln.data = np.log1p(Ln.data * 1e4)
lab = obs[TISSUE_COL].to_numpy()

# ---------- one-vs-rest markers (mean log-norm in tissue minus mean in the rest) ----------
Tmean = np.vstack([np.asarray(Ln[lab == t].mean(0)).ravel() for t in tissues])   # tissues x genes
Tpct  = np.vstack([np.asarray((M[lab == t] > 0).mean(0)).ravel() for t in tissues])
grand = Tmean.mean(0)
chosen, chosen_rows = set(), []
for ti, t in enumerate(tissues):
    others = Tmean[np.arange(len(tissues)) != ti].mean(0)
    score = Tmean[ti] - others
    score[Tpct[ti] < MIN_PCT] = -np.inf
    # each gene to the tissue where it is most enriched: skip if already taken
    order_g = np.argsort(score)[::-1]
    got = []
    for gi in order_g:
        if gi in chosen or not np.isfinite(score[gi]):
            continue
        # enforce "most enriched here": this tissue must be the argmax of (Tmean - grand)
        if np.argmax(Tmean[:, gi] - grand[gi]) != ti:
            continue
        chosen.add(gi); got.append(gi)
        if len(got) >= N_MARKERS:
            break
    for gi in got:
        chosen_rows.append((t, gi, var[gi]))
row_tissue = [r[0] for r in chosen_rows]
row_gidx   = [r[1] for r in chosen_rows]
row_gene   = [r[2] for r in chosen_rows]
print(f"{len(row_gidx)} marker rows ({N_MARKERS}/tissue target)")

# ---------- downsample columns per tissue, order by ORDER, sort within tissue ----------
rng = np.random.default_rng(SEED)
col_idx, col_tissue = [], []
gidx_by_tissue = {t: [row_gidx[i] for i in range(len(row_gidx)) if row_tissue[i] == t] for t in tissues}
for t in tissues:
    idx = np.where(lab == t)[0]
    if len(idx) > N_PER_TISSUE:
        idx = rng.choice(idx, N_PER_TISSUE, replace=False)
    gcols = gidx_by_tissue[t]
    key = np.asarray(Ln[idx][:, gcols].mean(1)).ravel() if gcols else np.zeros(len(idx))
    idx = idx[np.argsort(key)]                         # gradient within the block
    col_idx.append(idx); col_tissue += [t] * len(idx)
col_idx = np.concatenate(col_idx)

# ---------- build z-scored matrix (genes x cells) ----------
E = np.asarray(Ln[col_idx][:, row_gidx].todense()).T   # rows(genes) x cols(cells)
Z = (E - E.mean(1, keepdims=True)) / (E.std(1, keepdims=True) + 1e-9)
Z = np.clip(Z, -ZCLIP, ZCLIP)

# ---------- colors ----------
tcol = TISSUE_COLORS   # from palettes.py (was a silently-failing json load)
cmap_fb = plt.get_cmap("tab20")
cmap_t = {t: tcol.get(t, cmap_fb(i % 20)) for i, t in enumerate(tissues)}

# ---------- plot ----------
fig = plt.figure(figsize=(12, 9))
gs = GridSpec(2, 2, height_ratios=[0.04, 1], width_ratios=[1, 0.02],
              hspace=0.02, wspace=0.02)
ax_strip = fig.add_subplot(gs[0, 0])
ax_hm    = fig.add_subplot(gs[1, 0], sharex=ax_strip)
ax_cb    = fig.add_subplot(gs[1, 1])

# top color strip
tid = np.array([tissues.index(t) for t in col_tissue])
strip_cmap = ListedColormap([cmap_t[t] for t in tissues])
ax_strip.imshow(tid[None, :], aspect="auto", cmap=strip_cmap,
                norm=BoundaryNorm(np.arange(len(tissues) + 1) - 0.5, len(tissues)))
ax_strip.set_xticks([]); ax_strip.set_yticks([])
# tissue labels centered over each block
b = 0
for t in tissues:
    n = col_tissue.count(t)
    ax_strip.text(b + n / 2, -0.9, t, ha="center", va="bottom", fontsize=8,
                  color=cmap_t[t], fontweight="bold", rotation=0)
    b += n

# heatmap
im = ax_hm.imshow(Z, aspect="auto", cmap="RdBu_r", vmin=-ZCLIP, vmax=ZCLIP,
                  interpolation="nearest")
# block boundaries
b = 0
for t in tissues[:-1]:
    b += col_tissue.count(t)
    ax_hm.axvline(b - 0.5, color="white", lw=1.2)
r = 0
for t in tissues[:-1]:
    r += row_tissue.count(t)
    ax_hm.axhline(r - 0.5, color="white", lw=0.8)

ax_hm.set_xticks([])
if SHOW_GENES:
    ax_hm.set_yticks(range(len(row_gene)))
    ax_hm.set_yticklabels(row_gene, fontsize=6)
    for lbl in ax_hm.get_yticklabels():
        lbl.set_color("black")
else:
    ax_hm.set_yticks([])
ax_hm.set_xlabel(f"control cells, grouped by tissue ({N_PER_TISSUE}/tissue)")
ax_hm.set_ylabel(f"top {N_MARKERS} markers / tissue (row z-score)")
fig.colorbar(im, cax=ax_cb, label="z-score")
fig.suptitle("Marker heatmap, columns ordered by tissue (control)", y=0.95, fontsize=12)

out = FIG / "marker_heatmap_ordered_ctrl.png"
fig.savefig(out, dpi=200, bbox_inches="tight"); print("saved", out)
pd.DataFrame({"tissue": row_tissue, "gene": row_gene}).to_csv(
    TAB / "marker_heatmap_genes.csv", index=False)
print("saved marker_heatmap_genes.csv")





In [ ]:
"""
Marker heatmap, columns ORDERED by tissue (not clustered) in a developmental order.
Rows = top one-vs-rest markers per tissue (each gene assigned to its most-enriched
tissue, so no gene appears twice). This is the clean "block-diagonal" supplement for
CONTINUOUS states, where unsupervised cell clustering can't give crisp blocks.

Control cells only. Columns are downsampled to equal width per tissue and ordered
by developmental state; cells are randomly ordered within each tissue.
Values are row z-scored log-normalized expression.

Run in the pseudobulk env:  python marker_heatmap_ordered.py
Needs: numpy, pandas, h5py, scipy, matplotlib   (no seaborn, no scanpy).
"""
import json
import numpy as np, pandas as pd, h5py, scipy.sparse as sp
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch


MASTER   = ROOT / "00_DATA_FROZEN" / "03_analysis_ready" / "adata_master_normalized_magic.h5ad"
FIG      = ROOT / "07_figures" / "FIGURE2_clean"; FIG.mkdir(parents=True, exist_ok=True)
TAB      = ROOT / "06_tables" / "FIGURE_KD_clean"; TAB.mkdir(parents=True, exist_ok=True)

TISSUE_COL   = "predicted_leiden_sub_joint"

# ---- EDIT THIS to your exact label meanings / preferred early->late order ----
# stem/epiblast -> posterior then anterior streak -> nascent mesoderm early->late
# -> paraxial / lateral mesoderm -> endoderm / ectoderm / germline branches.
ORDER = ["pluripotent", "CS-LC", "pPS-LC", "aPS-LC", "eNasM", "lNasM",
         "aPM-LC", "LPM-LC", "DE", "AE-LC", "PGC-LC"]
DROP         = ["stressed"]
N_MARKERS    = 20        # top markers per tissue (rows)
N_PER_TISSUE = 60       # cells per tissue (columns), equal width
MIN_PCT      = 0.10     # a marker must be expressed in >=10% of its tissue's cells
SEED         = 0
ZCLIP        = 2.5
SHOW_GENES   = True     # label rows with gene names (readable at ~8/tissue)


def _rc(it):
    if isinstance(it, h5py.Group):
        ks = set(it.keys())
        if {"categories", "codes"} <= ks:
            return pd.Categorical.from_codes(it["codes"][:], categories=np.asarray(_rc(it["categories"])))
        if "values" in ks:
            v = it["values"][:]
            return (np.array([x.decode() if isinstance(x, bytes) else x for x in v], dtype=object)
                    if v.dtype == object or v.dtype.kind in "SU" else v)
    a = it[:]
    return a.astype(str) if a.dtype.kind == "S" else a


# ---------- load counts + obs (control only) ----------
with h5py.File(MASTER, "r") as h:
    v = h["var"]; ik = v.attrs.get("_index", "_index"); ik = ik.decode() if isinstance(ik, bytes) else ik
    vit = v[ik]; vraw = vit["values"][:] if isinstance(vit, h5py.Group) else vit[:]
    var = np.array([(x.decode() if isinstance(x, bytes) else str(x)).split(" (")[0] for x in vraw])
    # Don't use unresolved Ensembl IDs as displayed markers
    valid_gene_name = ~np.char.startswith(var.astype(str), "ENSG")
    X = h["layers"]["counts"]
    M = sp.csr_matrix((X["data"][:], X["indices"][:], X["indptr"][:]),
                      shape=tuple(int(s) for s in X.attrs["shape"]))
    og = h["obs"]
    obs = pd.DataFrame({c: np.asarray(_rc(og[c])).astype(str)
                        for c in ["condition", TISSUE_COL]})
obs["cond"] = obs["condition"].map({"Ctrl": "Ctrl", "DOX": "KD"})
keep = (obs["cond"] == "Ctrl") & (~obs[TISSUE_COL].isin(DROP))
obs = obs[keep].reset_index(drop=True); M = M[keep.to_numpy()]

# keep only tissues that exist, in ORDER
present = [t for t in ORDER if (obs[TISSUE_COL] == t).any()]
missing_from_order = [t for t in obs[TISSUE_COL].unique() if t not in ORDER and t not in DROP]
if missing_from_order:
    print("NOT in ORDER (append or ignore):", missing_from_order)
tissues = present
print("tissue order:", tissues)

# ---------- log-norm ----------
lib = np.asarray(M.sum(1)).ravel(); lib[lib == 0] = 1
Ln = M.multiply(1.0 / lib[:, None]).tocsr(); Ln.data = np.log1p(Ln.data * 1e4)
lab = obs[TISSUE_COL].to_numpy()

# ---------- one-vs-rest markers (mean log-norm in tissue minus mean in the rest) ----------
# true cell-weighted one-vs-rest enrichment
scores = []
pcts = []

for t in tissues:
    in_t = lab == t
    scores.append(
        np.asarray(Ln[in_t].mean(0)).ravel()
        - np.asarray(Ln[~in_t].mean(0)).ravel()
    )
    pcts.append(
        np.asarray((M[in_t] > 0).mean(0)).ravel()
    )

scores = np.vstack(scores)
pcts = np.vstack(pcts)

# tissue where each gene is most enriched
best_tissue = np.argmax(scores, axis=0)

chosen, chosen_rows = set(), []

for ti, t in enumerate(tissues):
    score = scores[ti].copy()
    score[pcts[ti] < MIN_PCT] = -np.inf
    score[score <= 0] = -np.inf
    score[~valid_gene_name] = -np.inf

    got = []

    for gi in np.argsort(score)[::-1]:
        if gi in chosen or not np.isfinite(score[gi]):
            continue

        if best_tissue[gi] != ti:
            continue

        chosen.add(gi)
        got.append(gi)

        if len(got) >= N_MARKERS:
            break

    for gi in got:
        chosen_rows.append((t, gi, var[gi]))
row_tissue = [r[0] for r in chosen_rows]
row_gidx   = [r[1] for r in chosen_rows]
row_gene   = [r[2] for r in chosen_rows]
print(f"{len(row_gidx)} marker rows ({N_MARKERS}/tissue target)")

# ---------- downsample columns per tissue, order by ORDER, sort within tissue ----------
rng = np.random.default_rng(SEED)
col_idx, col_tissue = [], []
gidx_by_tissue = {t: [row_gidx[i] for i in range(len(row_gidx)) if row_tissue[i] == t] for t in tissues}
for t in tissues:
    idx = np.where(lab == t)[0]
    if len(idx) > N_PER_TISSUE:
        idx = rng.choice(idx, N_PER_TISSUE, replace=False)
    idx = rng.permutation(idx)
    col_idx.append(idx); col_tissue += [t] * len(idx)
col_idx = np.concatenate(col_idx)

# ---------- build z-scored matrix (genes x cells) ----------
E = np.asarray(Ln[col_idx][:, row_gidx].todense()).T   # rows(genes) x cols(cells)
Z = (E - E.mean(1, keepdims=True)) / (E.std(1, keepdims=True) + 1e-9)
Z = np.clip(Z, -ZCLIP, ZCLIP)

# ---------- colors ----------
tcol = TISSUE_COLORS   # from palettes.py (was a silently-failing json load)
cmap_fb = plt.get_cmap("tab20")
cmap_t = {t: tcol.get(t, cmap_fb(i % 20)) for i, t in enumerate(tissues)}

# ---------- plot ----------
fig = plt.figure(figsize=(12, 12))
gs = GridSpec(
    2, 1,
    height_ratios=[0.025, 1],
    hspace=0.02
)

ax_strip = fig.add_subplot(gs[0, 0])
ax_hm    = fig.add_subplot(gs[1, 0], sharex=ax_strip)

# top color strip
tid = np.array([tissues.index(t) for t in col_tissue])
strip_cmap = ListedColormap([cmap_t[t] for t in tissues])

ax_strip.imshow(
    tid[None, :],
    aspect="auto",
    cmap=strip_cmap,
    norm=BoundaryNorm(
        np.arange(len(tissues) + 1) - 0.5,
        len(tissues)
    ),
    interpolation="nearest"
)

ax_strip.set_xticks([])
ax_strip.set_yticks([])
# remmove grid lines

for spine in ax_strip.spines.values():
    spine.set_visible(False)

# white borders between tissue blocks in top strip
b = 0
for t in tissues[:-1]:
    b += col_tissue.count(t)
    ax_strip.axvline(b - 0.5, color="white", lw=1.5)


# ---------- HEATMAP ----------

im = ax_hm.imshow(
    Z,
    aspect="auto",
    cmap="RdBu_r",
    vmin=-ZCLIP,
    vmax=ZCLIP,
    interpolation="nearest"
)

for spine in ax_hm.spines.values():
    spine.set_visible(False)
    
# vertical borders between tissue blocks
b = 0
for t in tissues[:-1]:
    b += col_tissue.count(t)
    ax_hm.axvline(b - 0.5, color="white", lw=1.2)


# horizontal borders between marker groups
r = 0
for t in tissues[:-1]:
    r += row_tissue.count(t)
    ax_hm.axhline(r - 0.5, color="white", lw=0.8)

ax_hm.set_xticks([])
if SHOW_GENES:
    ax_hm.set_yticks(range(len(row_gene)))
    ax_hm.set_yticklabels(row_gene, fontsize=9)
    ax_hm.tick_params(axis="y", which="both", left=False, right=False, length=0)
    ax_hm.grid(False)
    for lbl in ax_hm.get_yticklabels():
        lbl.set_color("black")
else:
    ax_hm.set_yticks([])
ax_hm.set_xlabel(f"control cells, grouped by tissue ({N_PER_TISSUE}/tissue)")
ax_hm.set_ylabel(f"top {N_MARKERS} markers / tissue (row z-score)")
handles = [
    Patch(facecolor=cmap_t[t], edgecolor="none", label=t)
    for t in tissues
]

ax_hm.legend(
    handles=handles,
    title="Cell state",
    loc="upper left",
    bbox_to_anchor=(1.06, 1.0),
    frameon=False,
    fontsize=7,
    title_fontsize=8,
    handlelength=1.2,
    handleheight=0.8,
    labelspacing=0.4
)
# short z-score colorbar
cax = ax_hm.inset_axes([1.02, 0.08, 0.018, 0.18])

cb = fig.colorbar(im, cax=cax)
cb.set_label("z-score", fontsize=8)
cb.ax.tick_params(labelsize=7)
fig.suptitle("Marker heatmap, columns ordered by tissue (control)", y=0.95, fontsize=12)

out = FIG / "marker_heatmap_ordered_ctrl.svg"
fig.savefig(out, bbox_inches="tight"); print("saved", out)
pd.DataFrame({"tissue": row_tissue, "gene": row_gene}).to_csv(
    TAB / "marker_heatmap_genes.csv", index=False)
print("saved marker_heatmap_genes.csv")





In [ ]:
# ================================================================
# APLNR / APELA COEXPRESSION
# TIME-RESOLVED, TISSUE-BLIND, STAN-FILTERED CONTROLS
# ================================================================
#
# Biological question:
#   At each time point, what genes reproducibly co-vary with
#   APLNR or APELA across the ENTIRE control gastruloid landscape?
#
# Analysis:
#   - Reload MASTER
#   - Apply Stan barcode filter
#   - Ctrl only
#   - 42 h and 72 h analyzed separately
#   - ALL tissues/states retained
#   - Chicago and Michigan analyzed independently
#   - raw/non-imputed counts preferred
#   - log1p(CP10K) normalization
#   - sequencing depth regressed out
#   - tissue/state NOT regressed out
#   - same-direction correlation required across repeats
#   - genes ranked by weaker |r| across Chicago/Michigan
#
# Assumes these path variables already exist in notebook:
#     MASTER
#     STAN_CSV
#
# ================================================================

from pathlib import Path
import re
import warnings

import anndata as ad
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.sparse as sp


# ================================================================
# 1. CONFIGURATION
# ================================================================

REPEAT_COLUMN = "site"
CONDITION_COLUMN = "condition"
TIME_COLUMN = "timepoint_true"

CONTROL_LABEL = "Ctrl"

REPEATS = [
    "Chicago",
    "Michigan",
]

SEEDS = [
    "APLNR",
    "APELA",
]

# Biological timepoints we want to report
TIMEPOINTS = [
    42,
    72,
]

MIN_CELLS = 30

# Candidate gene must be detected in >=5% of cells
# within that timepoint/repeat.
MIN_GENE_DETECTION = 0.05

# If seed is detected but sparse, warn rather than silently fail.
MIN_SEED_DETECTION_WARNING = 0.03

# Number shown in diagnostic dumbbell plots.
N_POSITIVE_TO_SHOW = 20
N_NEGATIVE_TO_SHOW = 10

GENE_CHUNK_SIZE = 400

OUT = Path("APLNR_APELA_global_coexpression")
OUT.mkdir(
    parents=True,
    exist_ok=True,
)

COUNT_LAYER_CANDIDATES = [
    "counts",
    "raw_counts",
    "count",
]

LOG_LAYER_CANDIDATES = [
    "log1p",
    "lognorm",
    "log_normalized",
    "normalized",
    "norm",
]

# Exclude these from the NAMED diagnostic plot only.
# They remain in all genome-wide exported CSVs.
DISPLAY_EXCLUDE_PATTERN = re.compile(
    r"^(MT-|RPL|RPS|MALAT1$|NEAT1$)",
    flags=re.IGNORECASE,
)


# ================================================================
# 2. LOAD MASTER
# ================================================================

adata_coex = ad.read_h5ad(
    MASTER
)

print(
    f"MASTER loaded: {adata_coex.n_obs:,} cells"
)


# ================================================================
# 3. APPLY STAN BARCODE FILTER + CTRL FILTER
# ================================================================

def bc16(x):
    """
    Extract the first DNA barcode-like sequence of length >=14.
    Matches your existing Stan-filter logic.
    """
    hits = re.findall(
        r"[ACGT]{14,}",
        str(x),
    )

    return (
        hits[0]
        if hits
        else None
    )


stan = pd.read_csv(
    STAN_CSV,
    index_col=0,
)

STAN_BC = {
    bc16(x)
    for x in stan.index
}

# Remove None if present.
STAN_BC.discard(
    None
)

adata_coex.obs["_bc"] = [
    bc16(x)
    for x in adata_coex.obs_names
]


stan_mask = (
    adata_coex.obs["_bc"]
    .isin(STAN_BC)
)

ctrl_mask = (
    adata_coex.obs[
        CONDITION_COLUMN
    ]
    .astype(str)
    .eq(CONTROL_LABEL)
)

adata_coex = adata_coex[
    stan_mask
    & ctrl_mask
].copy()


print()
print(
    f"Stan-filtered control cells: "
    f"{adata_coex.n_obs:,}"
)


# ================================================================
# 4. PARSE / CORRECT TIMEPOINT
# ================================================================
#
# Your original 42 h dataset was accidentally labeled 48 h.
#
# This code automatically handles that:
#
#   if 48 exists and 42 does NOT:
#       48 -> 42
#
# Otherwise it leaves 42 as 42.
#
# ================================================================

if TIME_COLUMN not in adata_coex.obs.columns:

    raise KeyError(
        f"{TIME_COLUMN!r} not found in adata.obs.\n"
        f"Available columns include:\n"
        f"{list(adata_coex.obs.columns)}"
    )


def parse_numeric_time(value):

    if pd.isna(value):
        return np.nan

    match = re.search(
        r"(\d+(?:\.\d+)?)",
        str(value),
    )

    if match is None:
        return np.nan

    return int(
        round(
            float(
                match.group(1)
            )
        )
    )


raw_time = (
    adata_coex.obs[
        TIME_COLUMN
    ]
    .map(parse_numeric_time)
    .astype(float)
)


observed_times = set(
    raw_time
    .dropna()
    .astype(int)
    .unique()
)


# Automatically fix accidental 48 h label.
if (
    48 in observed_times
    and 42 not in observed_times
):

    print()
    print(
        "Detected 48 h but no 42 h in metadata."
    )
    print(
        "Treating 48 h as experimental 42 h."
    )

    analysis_time = (
        raw_time
        .replace(
            {
                48: 42,
            }
        )
    )

else:

    analysis_time = (
        raw_time.copy()
    )


adata_coex.obs[
    "_analysis_time"
] = analysis_time.values


print()
print(
    "Raw time labels:"
)

print(
    adata_coex.obs[
        TIME_COLUMN
    ]
    .value_counts(
        dropna=False
    )
)


print()
print(
    "Analysis timepoints:"
)

print(
    adata_coex.obs[
        "_analysis_time"
    ]
    .value_counts(
        dropna=False
    )
    .sort_index()
)


print()
print(
    "Stan-filtered Ctrl cells by site/time:"
)

print(
    adata_coex.obs
    .groupby(
        [
            REPEAT_COLUMN,
            "_analysis_time",
        ],
        observed=True,
    )
    .size()
    .to_string()
)


# ================================================================
# 5. CHOOSE NON-IMPUTED EXPRESSION SOURCE
# ================================================================

def choose_expression_source(
    a,
):

    # Preferred = raw counts.
    for layer in COUNT_LAYER_CANDIDATES:

        if layer in a.layers:

            return {
                "kind": "counts",
                "name": (
                    f"adata_coex.layers['{layer}']"
                ),
                "matrix": a.layers[layer],
                "var": a.var,
            }


    # Second choice = explicit log-normalized layer.
    for layer in LOG_LAYER_CANDIDATES:

        if layer in a.layers:

            return {
                "kind": "log",
                "name": (
                    f"adata_coex.layers['{layer}']"
                ),
                "matrix": a.layers[layer],
                "var": a.var,
            }


    # Third choice = raw.
    if a.raw is not None:

        warnings.warn(
            "No counts/named normalized layer found. "
            "Using adata_coex.raw.X. "
            "Confirm this is NON-IMPUTED."
        )

        return {
            "kind": "log",
            "name": "adata_coex.raw.X",
            "matrix": a.raw.X,
            "var": a.raw.var,
        }


    raise RuntimeError(
        "No safe non-imputed expression matrix found."
    )


expression_source = (
    choose_expression_source(
        adata_coex
    )
)


print()
print(
    "Expression source:",
    expression_source[
        "name"
    ],
)

print(
    "Expression mode:",
    expression_source[
        "kind"
    ],
)


# ================================================================
# 6. BUILD GENE SYMBOL LOOKUP
# ================================================================

source_var = (
    expression_source[
        "var"
    ]
)


if "symbol" in source_var.columns:

    source_symbols = (
        source_var[
            "symbol"
        ]
        .astype(str)
        .to_numpy()
    )

else:

    source_symbols = (
        source_var.index
        .astype(str)
        .to_numpy()
    )


# Keep first occurrence of duplicated symbols.
symbol_to_index = {}


for index, symbol in enumerate(
    source_symbols
):

    if (
        symbol
        and symbol != "nan"
        and symbol not in symbol_to_index
    ):

        symbol_to_index[
            symbol
        ] = index


all_symbols = np.asarray(
    list(
        symbol_to_index.keys()
    ),
    dtype=object,
)


all_gene_indices = np.asarray(
    [
        symbol_to_index[
            symbol
        ]
        for symbol
        in all_symbols
    ],
    dtype=int,
)


for seed in SEEDS:

    if seed not in symbol_to_index:

        raise KeyError(
            f"{seed} not found in "
            f"{expression_source['name']}"
        )


# ================================================================
# 7. LIBRARY-SIZE / DEPTH VECTOR
# ================================================================

def row_sums(
    matrix,
):

    if sp.issparse(
        matrix
    ):

        return np.asarray(
            matrix.sum(
                axis=1
            )
        ).ravel()

    return (
        np.asarray(
            matrix
        )
        .sum(
            axis=1
        )
    )


depth_vector = None
depth_source = None


# Prefer actual counts.
for layer in COUNT_LAYER_CANDIDATES:

    if layer in adata_coex.layers:

        depth_vector = (
            row_sums(
                adata_coex.layers[
                    layer
                ]
            )
        )

        depth_source = (
            f"adata_coex.layers['{layer}'] "
            "library size"
        )

        break


# Metadata fallback.
if depth_vector is None:

    for obs_column in [
        "total_counts",
        "n_counts",
        "nCount_RNA",
    ]:

        if (
            obs_column
            in adata_coex.obs.columns
        ):

            values = (
                pd.to_numeric(
                    adata_coex.obs[
                        obs_column
                    ],
                    errors="coerce",
                )
            )

            depth_vector = (
                values
                .fillna(
                    values.median()
                )
                .to_numpy(
                    dtype=float
                )
            )

            depth_source = (
                obs_column
            )

            break


if depth_vector is None:

    raise RuntimeError(
        "Could not identify sequencing depth."
    )


print(
    "Depth covariate:",
    depth_source,
)


# ================================================================
# 8. EXTRACT + NORMALIZE EXPRESSION
# ================================================================

def extract_expression(
    cell_indices,
    gene_indices,
):

    matrix = (
        expression_source[
            "matrix"
        ]
    )


    values = matrix[
        cell_indices,
        :
    ][
        :,
        gene_indices
    ]


    if sp.issparse(
        values
    ):

        values = (
            values.toarray()
        )

    else:

        values = (
            np.asarray(
                values
            )
        )


    values = values.astype(
        np.float64,
        copy=False,
    )


    # Counts -> log1p(CP10K)
    if (
        expression_source[
            "kind"
        ]
        == "counts"
    ):

        library_size = (
            depth_vector[
                cell_indices
            ]
            .astype(float)
        )


        library_size = (
            np.where(
                library_size > 0,
                library_size,
                1.0,
            )
        )


        values = np.log1p(
            values
            / library_size[
                :,
                None
            ]
            * 1e4
        )


    return values


# ================================================================
# 9. DEPTH-ONLY RESIDUALIZATION
# ================================================================
#
# We intentionally DO NOT include:
#   - tissue/state
#   - time
#
# because:
#   - time is analyzed separately
#   - tissue/domain structure is what we WANT to discover
#
# ================================================================

def make_depth_basis(
    cell_indices,
):

    log_depth = np.log1p(
        depth_vector[
            cell_indices
        ].astype(float)
    )


    if np.std(
        log_depth
    ) > 0:

        log_depth = (
            log_depth
            - np.mean(
                log_depth
            )
        ) / np.std(
            log_depth
        )

    else:

        log_depth = (
            np.zeros_like(
                log_depth
            )
        )


    design = np.column_stack(
        [
            np.ones(
                len(
                    cell_indices
                ),
                dtype=float,
            ),
            log_depth,
        ]
    )


    q, _ = np.linalg.qr(
        design,
        mode="reduced",
    )


    return q


def residualize(
    values,
    q,
):

    return (
        values
        - q
        @ (
            q.T
            @ values
        )
    )


# ================================================================
# 10. ONE SEED × TIMEPOINT × REPEAT
# ================================================================

def correlate_seed_time_repeat(
    seed,
    timepoint,
    repeat,
):

    mask = (
        adata_coex.obs[
            REPEAT_COLUMN
        ]
        .astype(str)
        .eq(
            repeat
        )
        &
        adata_coex.obs[
            "_analysis_time"
        ]
        .eq(
            timepoint
        )
    )


    cell_indices = np.where(
        mask.to_numpy()
    )[0]


    if len(
        cell_indices
    ) < MIN_CELLS:

        raise RuntimeError(
            f"{seed}, {timepoint} h, {repeat}: "
            f"only {len(cell_indices)} eligible "
            "Stan-filtered Ctrl cells."
        )


    q = (
        make_depth_basis(
            cell_indices
        )
    )


    seed_index = (
        symbol_to_index[
            seed
        ]
    )


    seed_values = (
        extract_expression(
            cell_indices,
            np.asarray(
                [
                    seed_index
                ],
                dtype=int,
            ),
        )[
            :,
            0
        ]
    )


    seed_detection = float(
        np.mean(
            seed_values > 0
        )
    )


    print(
        f"    {seed} detection: "
        f"{seed_detection:.1%}"
    )


    if seed_detection == 0:

        raise RuntimeError(
            f"{seed}, {timepoint} h, {repeat}: "
            "seed is not detected in any cells."
        )


    if (
        seed_detection
        < MIN_SEED_DETECTION_WARNING
    ):

        warnings.warn(
            f"{seed}, {timepoint} h, {repeat}: "
            f"seed detected in only "
            f"{seed_detection:.1%} of cells. "
            "Interpret correlations cautiously."
        )


    # Residualize seed against depth.
    seed_residual = (
        residualize(
            seed_values[
                :,
                None
            ],
            q,
        )[
            :,
            0
        ]
    )


    seed_residual = (
        seed_residual
        - seed_residual.mean()
    )


    seed_norm = np.sqrt(
        np.sum(
            seed_residual ** 2
        )
    )


    if seed_norm == 0:

        raise RuntimeError(
            f"{seed}, {timepoint} h, {repeat}: "
            "no variation remains after "
            "depth adjustment."
        )


    output_blocks = []


    # ------------------------------------------------------------
    # Process genes in chunks
    # ------------------------------------------------------------

    for start in range(
        0,
        len(
            all_symbols
        ),
        GENE_CHUNK_SIZE,
    ):

        stop = min(
            start
            + GENE_CHUNK_SIZE,
            len(
                all_symbols
            ),
        )


        chunk_symbols = (
            all_symbols[
                start:stop
            ]
        )


        chunk_indices = (
            all_gene_indices[
                start:stop
            ]
        )


        values = (
            extract_expression(
                cell_indices,
                chunk_indices,
            )
        )


        detection = np.mean(
            values > 0,
            axis=0,
        )


        residual = residualize(
            values,
            q,
        )


        residual = (
            residual
            - residual.mean(
                axis=0,
                keepdims=True,
            )
        )


        gene_norm = np.sqrt(
            np.sum(
                residual ** 2,
                axis=0,
            )
        )


        denominator = (
            seed_norm
            * gene_norm
        )


        correlation = np.full(
            len(
                chunk_symbols
            ),
            np.nan,
            dtype=float,
        )


        valid = (
            denominator > 0
        )


        correlation[
            valid
        ] = (
            seed_residual
            @ residual[
                :,
                valid
            ]
        ) / denominator[
            valid
        ]


        output_blocks.append(
            pd.DataFrame(
                {
                    "gene":
                        chunk_symbols,

                    "correlation":
                        correlation,

                    "detection_fraction":
                        detection,
                }
            )
        )


    result = pd.concat(
        output_blocks,
        ignore_index=True,
    )


    # Only consider genes detected in >=5% of cells.
    result = result[
        result[
            "detection_fraction"
        ]
        >= MIN_GENE_DETECTION
    ].copy()


    # Remove seed itself.
    result = result[
        result[
            "gene"
        ]
        != seed
    ].copy()


    result[
        "seed"
    ] = seed

    result[
        "timepoint"
    ] = timepoint

    result[
        "repeat"
    ] = repeat

    result[
        "n_cells"
    ] = len(
        cell_indices
    )

    result[
        "seed_detection_fraction"
    ] = seed_detection


    return result


# ================================================================
# 11. RUN:
#
# APLNR 42
# APLNR 72
# APELA 42
# APELA 72
#
# ================================================================

all_results = {}


for seed in SEEDS:

    for timepoint in TIMEPOINTS:

        print()
        print(
            "=" * 72
        )

        print(
            f"{seed} — {timepoint} h"
        )

        print(
            "=" * 72
        )


        repeat_results = {}


        # --------------------------------------------------------
        # Run each biological dataset separately
        # --------------------------------------------------------

        for repeat in REPEATS:

            print(
                f"\n{repeat}:"
            )


            result = (
                correlate_seed_time_repeat(
                    seed=seed,
                    timepoint=timepoint,
                    repeat=repeat,
                )
            )


            repeat_results[
                repeat
            ] = result


            print(
                f"    cells: "
                f"{int(result['n_cells'].iloc[0]):,}"
            )

            print(
                f"    eligible genes: "
                f"{len(result):,}"
            )


        # --------------------------------------------------------
        # Join Chicago + Michigan by gene
        # --------------------------------------------------------

        combined = (
            repeat_results[
                REPEATS[0]
            ][
                [
                    "gene",
                    "correlation",
                    "detection_fraction",
                ]
            ]
            .rename(
                columns={
                    "correlation":
                        REPEATS[0],

                    "detection_fraction":
                        f"detection_{REPEATS[0]}",
                }
            )
            .merge(
                repeat_results[
                    REPEATS[1]
                ][
                    [
                        "gene",
                        "correlation",
                        "detection_fraction",
                    ]
                ]
                .rename(
                    columns={
                        "correlation":
                            REPEATS[1],

                        "detection_fraction":
                            f"detection_{REPEATS[1]}",
                    }
                ),

                on="gene",
                how="inner",
            )
        )


        # Average correlation for descriptive location.
        combined[
            "mean_correlation"
        ] = (
            combined[
                REPEATS
            ]
            .mean(
                axis=1
            )
        )


        # Must have same sign in both datasets.
        combined[
            "same_direction"
        ] = (
            np.sign(
                combined[
                    REPEATS[0]
                ]
            )
            ==
            np.sign(
                combined[
                    REPEATS[1]
                ]
            )
        )


        # Conservative ranking:
        # rank by whichever dataset has WEAKER |r|.
        combined[
            "weaker_abs_correlation"
        ] = (
            combined[
                REPEATS
            ]
            .abs()
            .min(
                axis=1
            )
        )


        combined[
            "seed"
        ] = seed

        combined[
            "timepoint"
        ] = timepoint

        combined[
            "analysis"
        ] = (
            "Stan-filtered Ctrl; "
            "all tissues; "
            "depth-adjusted"
        )


        combined = (
            combined
            .sort_values(
                [
                    "same_direction",
                    "weaker_abs_correlation",
                ],
                ascending=[
                    False,
                    False,
                ],
            )
        )


        all_results[
            (
                seed,
                timepoint,
            )
        ] = combined


        # ========================================================
        # 12. SAVE FULL GENOME-WIDE TABLE
        # ========================================================

        full_output = (
            OUT
            / (
                f"{seed.lower()}_"
                f"{timepoint}h_"
                "global_depth_adjusted_correlations.csv"
            )
        )


        combined.to_csv(
            full_output,
            index=False,
        )


        # ========================================================
        # 13. SAVE CONCORDANT-ONLY TABLE
        # ========================================================

        concordant = combined[
            combined[
                "same_direction"
            ]
        ].copy()


        concordant_output = (
            OUT
            / (
                f"{seed.lower()}_"
                f"{timepoint}h_"
                "concordant_correlations.csv"
            )
        )


        concordant.to_csv(
            concordant_output,
            index=False,
        )


        print()
        print(
            f"Concordant genes: "
            f"{len(concordant):,}"
        )


        # ========================================================
        # 14. PICK DIAGNOSTIC NAMED HITS
        # ========================================================

        display_pool = (
            concordant.copy()
        )


        # Don't label Ensembl-like unnamed features.
        display_pool = display_pool[
            ~display_pool[
                "gene"
            ]
            .astype(str)
            .str.match(
                r"^(ENSG|ENGS)\w*$",
                case=False,
                na=False,
            )
        ].copy()


        # Don't label ribosomal/mitochondrial/MALAT/etc
        # in the visual diagnostic.
        display_pool = display_pool[
            ~display_pool[
                "gene"
            ]
            .astype(str)
            .str.match(
                DISPLAY_EXCLUDE_PATTERN
            )
        ].copy()


        positive = (
            display_pool[
                (
                    display_pool[
                        REPEATS[0]
                    ] > 0
                )
                &
                (
                    display_pool[
                        REPEATS[1]
                    ] > 0
                )
            ]
            .nlargest(
                N_POSITIVE_TO_SHOW,
                "weaker_abs_correlation",
            )
            .sort_values(
                "mean_correlation",
                ascending=True,
            )
        )


        negative = (
            display_pool[
                (
                    display_pool[
                        REPEATS[0]
                    ] < 0
                )
                &
                (
                    display_pool[
                        REPEATS[1]
                    ] < 0
                )
            ]
            .nlargest(
                N_NEGATIVE_TO_SHOW,
                "weaker_abs_correlation",
            )
            .sort_values(
                "mean_correlation",
                ascending=True,
            )
        )


        named = pd.concat(
            [
                negative,
                positive,
            ],
            ignore_index=True,
        )


        named_output = (
            OUT
            / (
                f"{seed.lower()}_"
                f"{timepoint}h_"
                "top_reproducible_correlates.csv"
            )
        )


        named.to_csv(
            named_output,
            index=False,
        )


        # ========================================================
        # 15. DIAGNOSTIC DUMBBELL PLOT
        # ========================================================

        row_height = 0.36


        fig_height = max(
            6.0,
            1.8
            + row_height
            * len(
                named
            ),
        )


        fig, ax = plt.subplots(
            figsize=(
                6.8,
                fig_height,
            )
        )


        y = np.arange(
            len(
                named
            )
        )


        for y_pos, row in zip(
            y,
            named.itertuples(
                index=False
            ),
        ):

            chicago = getattr(
                row,
                "Chicago",
            )

            michigan = getattr(
                row,
                "Michigan",
            )


            ax.plot(
                [
                    chicago,
                    michigan,
                ],
                [
                    y_pos,
                    y_pos,
                ],
                linewidth=1.5,
                alpha=0.55,
                zorder=1,
            )


            ax.scatter(
                chicago,
                y_pos,
                s=65,
                zorder=3,
            )


            ax.scatter(
                michigan,
                y_pos,
                s=65,
                facecolors="white",
                linewidths=1.3,
                zorder=3,
            )


        ax.axvline(
            0,
            linewidth=1,
        )


        ax.set_yticks(
            y
        )


        ax.set_yticklabels(
            named[
                "gene"
            ],
            fontsize=10,
        )


        ax.set_xlabel(
            f"Depth-adjusted correlation with {seed}"
        )


        ax.set_title(
            f"{seed} — control {timepoint} h\n"
            "all tissues included"
        )


        for spine in [
            "top",
            "right",
            "left",
        ]:

            ax.spines[
                spine
            ].set_visible(
                False
            )


        ax.tick_params(
            axis="y",
            left=False,
        )


        fig.tight_layout()


        png_output = (
            OUT
            / (
                f"{seed.lower()}_"
                f"{timepoint}h_"
                "reproducible_correlates.png"
            )
        )


        svg_output = (
            OUT
            / (
                f"{seed.lower()}_"
                f"{timepoint}h_"
                "reproducible_correlates.svg"
            )
        )


        fig.savefig(
            png_output,
            dpi=300,
            bbox_inches="tight",
        )


        fig.savefig(
            svg_output,
            bbox_inches="tight",
        )


        plt.show()


        # ========================================================
        # 16. PRINT TOP HITS
        # ========================================================

        print()
        print(
            f"TOP POSITIVE {seed} "
            f"CORRELATES — {timepoint} h"
        )


        print(
            positive[
                [
                    "gene",
                    "Chicago",
                    "Michigan",
                    "mean_correlation",
                    "weaker_abs_correlation",
                ]
            ]
            .sort_values(
                "weaker_abs_correlation",
                ascending=False,
            )
            .round(
                3
            )
            .to_string(
                index=False
            )
        )


        print()
        print(
            f"TOP NEGATIVE {seed} "
            f"CORRELATES — {timepoint} h"
        )


        print(
            negative[
                [
                    "gene",
                    "Chicago",
                    "Michigan",
                    "mean_correlation",
                    "weaker_abs_correlation",
                ]
            ]
            .sort_values(
                "weaker_abs_correlation",
                ascending=False,
            )
            .round(
                3
            )
            .to_string(
                index=False
            )
        )


# ================================================================
# 17. CREATE CROSS-TIME SUMMARY TABLE
# ================================================================

summary_tables = []


for (
    seed,
    timepoint,
), table in all_results.items():

    temp = table[
        table[
            "same_direction"
        ]
    ][
        [
            "gene",
            "mean_correlation",
            "weaker_abs_correlation",
        ]
    ].copy()


    temp = temp.rename(
        columns={
            "mean_correlation":
                f"{seed}_{timepoint}h_mean_r",

            "weaker_abs_correlation":
                f"{seed}_{timepoint}h_weaker_abs_r",
        }
    )


    temp = temp.set_index(
        "gene"
    )


    summary_tables.append(
        temp
    )


summary = pd.concat(
    summary_tables,
    axis=1,
    join="outer",
)


summary_output = (
    OUT
    / "APLNR_APELA_42h_72h_concordant_summary.csv"
)


summary.to_csv(
    summary_output
)


# ================================================================
# 18. FINAL OUTPUT SUMMARY
# ================================================================

print()
print(
    "=" * 72
)

print(
    "DONE"
)

print(
    "=" * 72
)

print(
    "Input object:"
)

print(
    f"  {adata_coex.n_obs:,} "
    "Stan-filtered control cells"
)

print()
print(
    "Output folder:"
)

print(
    OUT.resolve()
)

print()
print(
    "Main files to inspect:"
)

for seed in SEEDS:

    for timepoint in TIMEPOINTS:

        print(
            " ",
            OUT
            / (
                f"{seed.lower()}_"
                f"{timepoint}h_"
                "top_reproducible_correlates.csv"
            )
        )

print()
print(
    "Cross-time summary:"
)

print(
    summary_output
)

In [ ]:
# ============================================================================
# GENOME-WIDE VALIDATION OF APLNR / APELA COEXPRESSION
# ============================================================================
#
# RUN AFTER the main genome-wide coexpression block.
#
# Requires these objects/functions from that block:
#   adata_coex
#   all_results
#   symbol_to_index
#   extract_expression
#   make_depth_basis
#   residualize
#   REPEATS
#   SEEDS
#   TIMEPOINTS
#   REPEAT_COLUMN
#
#
# WHAT THIS DOES
# --------------
# For every gene in the original correlation analysis:
#
#   1. Seed OFF = zero raw counts for APLNR/APELA
#
#   2. Among seed-positive cells:
#          Q1 = lowest depth-adjusted seed expression
#          Q2
#          Q3
#          Q4 = highest depth-adjusted seed expression
#
#   3. For every target gene, calculate in each seed bin:
#
#          A) fraction of cells expressing target
#          B) target abundance among target-positive cells
#
#   4. Summarize:
#
#          delta_detection =
#              target detection in Q4 - target detection in OFF
#
#          delta_abundance =
#              depth-adjusted target abundance in Q4
#              - depth-adjusted target abundance in OFF
#
#   5. Chicago and Michigan are calculated INDEPENDENTLY.
#
#   6. Results are merged with the original APLNR/APELA correlations.
#
#
# OUTPUTS — ONLY TWO CSVs
# -----------------------
# 1. APLNR_APELA_genomewide_validation_ALL.csv
#       all genes + all metrics
#
# 2. APLNR_APELA_genomewide_validation_TOP.csv
#       top 25 positive + top 25 negative correlations
#       for each seed/timepoint
#
# ============================================================================


from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp


# ============================================================================
# 1. SETTINGS
# ============================================================================

VALID_OUT = Path(
    "APLNR_APELA_genomewide_validation"
)

VALID_OUT.mkdir(
    parents=True,
    exist_ok=True
)


BIN_ORDER = [
    "Off",
    "Q1",
    "Q2",
    "Q3",
    "Q4",
]


# Need at least this many target-positive cells in a bin
# before interpreting abundance among positive cells.
#
# We keep this low because APLNR+ Chicago 72 h is sparse.
MIN_TARGET_POS_FOR_ABUNDANCE = 3


GENE_CHUNK_SIZE_VALIDATION = 300


# ============================================================================
# 2. CHECK THAT MAIN ANALYSIS HAS BEEN RUN
# ============================================================================

required_objects = [
    "adata_coex",
    "all_results",
    "symbol_to_index",
    "extract_expression",
    "make_depth_basis",
    "residualize",
    "REPEATS",
    "SEEDS",
    "TIMEPOINTS",
    "REPEAT_COLUMN",
]


missing_objects = [
    x
    for x in required_objects
    if x not in globals()
]


if missing_objects:

    raise RuntimeError(
        "Run the main APLNR/APELA coexpression block first.\n"
        f"Missing: {missing_objects}"
    )


# ============================================================================
# 3. FIND RAW COUNT LAYER
# ============================================================================

count_layer = None


for candidate in [
    "counts",
    "raw_counts",
    "count",
]:

    if candidate in adata_coex.layers:

        count_layer = candidate
        break


if count_layer is None:

    raise RuntimeError(
        "A raw-count layer is required for zero/nonzero detection."
    )


print(
    "Raw detection source:",
    f"adata_coex.layers['{count_layer}']"
)


# ============================================================================
# 4. RAW COUNT EXTRACTION
# ============================================================================

def extract_raw_counts(
    cell_indices,
    gene_indices,
):

    x = (
        adata_coex.layers[
            count_layer
        ][
            cell_indices,
            :
        ][
            :,
            gene_indices
        ]
    )


    if sp.issparse(x):

        x = x.toarray()

    else:

        x = np.asarray(x)


    return x


# ============================================================================
# 5. CREATE SEED-EXPRESSION BINS
# ============================================================================

def make_seed_bins(
    seed,
    timepoint,
    repeat,
):

    # ------------------------------------------------------------------------
    # Cells for one biological dataset and one time point
    # ------------------------------------------------------------------------

    mask = (
        adata_coex.obs[
            REPEAT_COLUMN
        ]
        .astype(str)
        .eq(repeat)
        &
        adata_coex.obs[
            "_analysis_time"
        ]
        .eq(timepoint)
    )


    cell_indices = np.where(
        mask.to_numpy()
    )[0]


    if len(cell_indices) == 0:

        raise RuntimeError(
            f"No cells found for {repeat}, {timepoint} h."
        )


    # ------------------------------------------------------------------------
    # Seed counts
    # ------------------------------------------------------------------------

    seed_index = (
        symbol_to_index[
            seed
        ]
    )


    seed_counts = (
        extract_raw_counts(
            cell_indices,
            np.asarray(
                [seed_index],
                dtype=int,
            ),
        )[
            :,
            0
        ]
    )


    seed_positive = (
        seed_counts > 0
    )


    n_seed_positive = int(
        seed_positive.sum()
    )


    if n_seed_positive < 4:

        raise RuntimeError(
            f"{seed}, {repeat}, {timepoint} h: "
            f"only {n_seed_positive} seed-positive cells."
        )


    # ------------------------------------------------------------------------
    # Normalized seed expression
    # ------------------------------------------------------------------------

    seed_norm = (
        extract_expression(
            cell_indices,
            np.asarray(
                [seed_index],
                dtype=int,
            ),
        )[
            :,
            0
        ]
    )


    # ------------------------------------------------------------------------
    # Remove sequencing-depth effect
    # ------------------------------------------------------------------------

    q_basis = (
        make_depth_basis(
            cell_indices
        )
    )


    seed_residual = (
        residualize(
            seed_norm[
                :,
                None
            ],
            q_basis,
        )[
            :,
            0
        ]
    )


    # ------------------------------------------------------------------------
    # OFF + Q1-Q4
    # ------------------------------------------------------------------------

    bins = np.full(
        len(cell_indices),
        "Off",
        dtype=object,
    )


    pos_idx = np.where(
        seed_positive
    )[0]


    # Rank depth-adjusted seed expression.
    #
    # This means "high" is defined RELATIVE to the distribution
    # within that biological dataset, not by an arbitrary expression cutoff.
    ranks = (
        pd.Series(
            seed_residual[
                pos_idx
            ]
        )
        .rank(
            method="first"
        )
    )


    quartiles = (
        pd.qcut(
            ranks,
            q=4,
            labels=[
                "Q1",
                "Q2",
                "Q3",
                "Q4",
            ],
        )
        .astype(str)
        .to_numpy()
    )


    bins[
        pos_idx
    ] = quartiles


    bin_sizes = {
        b:
            int(
                np.sum(
                    bins == b
                )
            )
        for b in BIN_ORDER
    }


    print(
        f"{seed} {timepoint} h {repeat}: "
        f"{len(cell_indices):,} cells | "
        f"{n_seed_positive:,} {seed}+ cells | "
        +
        ", ".join(
            [
                f"{b}={bin_sizes[b]}"
                for b in BIN_ORDER
            ]
        )
    )


    return (
        cell_indices,
        bins,
        q_basis,
        bin_sizes,
        n_seed_positive,
    )


# ============================================================================
# 6. VALIDATE ALL GENES IN ONE SEED × TIME × REPEAT
# ============================================================================

def validate_all_genes_one_repeat(
    seed,
    timepoint,
    repeat,
    genes,
):

    (
        cell_indices,
        seed_bins,
        q_basis,
        bin_sizes,
        n_seed_positive,
    ) = make_seed_bins(
        seed=seed,
        timepoint=timepoint,
        repeat=repeat,
    )


    result_blocks = []


    # ------------------------------------------------------------------------
    # Gene chunks
    # ------------------------------------------------------------------------

    for start in range(
        0,
        len(genes),
        GENE_CHUNK_SIZE_VALIDATION,
    ):

        stop = min(
            start
            + GENE_CHUNK_SIZE_VALIDATION,
            len(genes),
        )


        chunk_genes = (
            genes[
                start:stop
            ]
        )


        chunk_indices = np.asarray(
            [
                symbol_to_index[g]
                for g in chunk_genes
            ],
            dtype=int,
        )


        # --------------------------------------------------------------------
        # Raw counts = detection
        # --------------------------------------------------------------------

        raw = (
            extract_raw_counts(
                cell_indices,
                chunk_indices,
            )
        )


        detected = (
            raw > 0
        )


        # --------------------------------------------------------------------
        # Normalized expression
        # --------------------------------------------------------------------

        norm_expr = (
            extract_expression(
                cell_indices,
                chunk_indices,
            )
        )


        # --------------------------------------------------------------------
        # Depth-adjusted target expression
        #
        # This makes the abundance component parallel to the original
        # correlation analysis.
        # --------------------------------------------------------------------

        target_resid = (
            residualize(
                norm_expr,
                q_basis,
            )
        )


        block = pd.DataFrame(
            {
                "gene":
                    chunk_genes,
            }
        )


        # --------------------------------------------------------------------
        # Calculate metrics for every seed bin
        # --------------------------------------------------------------------

        for bin_name in BIN_ORDER:

            b = (
                seed_bins
                == bin_name
            )


            n_bin = int(
                b.sum()
            )


            # --------------------------------------------------------------
            # Number of target-positive cells
            # --------------------------------------------------------------

            n_target_positive = (
                detected[
                    b,
                    :
                ]
                .sum(
                    axis=0
                )
                .astype(int)
            )


            # --------------------------------------------------------------
            # Fraction target+
            # --------------------------------------------------------------

            detection_fraction = (
                n_target_positive
                / n_bin
            )


            # --------------------------------------------------------------
            # Mean normalized target expression across ALL cells
            # --------------------------------------------------------------

            mean_norm_all = (
                norm_expr[
                    b,
                    :
                ]
                .mean(
                    axis=0
                )
            )


            # --------------------------------------------------------------
            # Expression among target-positive cells only
            # --------------------------------------------------------------

            mean_norm_positive = np.full(
                len(
                    chunk_genes
                ),
                np.nan,
                dtype=float,
            )


            mean_resid_positive = np.full(
                len(
                    chunk_genes
                ),
                np.nan,
                dtype=float,
            )


            for j in range(
                len(
                    chunk_genes
                )
            ):

                positive_in_bin = (
                    detected[
                        b,
                        j
                    ]
                )


                n_positive = int(
                    positive_in_bin.sum()
                )


                if (
                    n_positive
                    >= MIN_TARGET_POS_FOR_ABUNDANCE
                ):

                    mean_norm_positive[
                        j
                    ] = (
                        norm_expr[
                            b,
                            j
                        ][
                            positive_in_bin
                        ]
                        .mean()
                    )


                    mean_resid_positive[
                        j
                    ] = (
                        target_resid[
                            b,
                            j
                        ][
                            positive_in_bin
                        ]
                        .mean()
                    )


            # --------------------------------------------------------------
            # Save bin metrics
            # --------------------------------------------------------------

            block[
                f"{bin_name}_n_cells"
            ] = n_bin


            block[
                f"{bin_name}_n_target_positive"
            ] = n_target_positive


            block[
                f"{bin_name}_fraction_positive"
            ] = detection_fraction


            block[
                f"{bin_name}_mean_expression_all"
            ] = mean_norm_all


            block[
                f"{bin_name}_mean_expression_positive"
            ] = mean_norm_positive


            block[
                f"{bin_name}_mean_depth_adjusted_expression_positive"
            ] = mean_resid_positive


        # --------------------------------------------------------------------
        # Q4 - OFF effects
        # --------------------------------------------------------------------

        block[
            "delta_detection_Q4_minus_Off"
        ] = (
            block[
                "Q4_fraction_positive"
            ]
            -
            block[
                "Off_fraction_positive"
            ]
        )


        block[
            "delta_expression_positive_Q4_minus_Off"
        ] = (
            block[
                "Q4_mean_expression_positive"
            ]
            -
            block[
                "Off_mean_expression_positive"
            ]
        )


        block[
            "delta_depth_adjusted_expression_positive_Q4_minus_Off"
        ] = (
            block[
                "Q4_mean_depth_adjusted_expression_positive"
            ]
            -
            block[
                "Off_mean_depth_adjusted_expression_positive"
            ]
        )


        block[
            "seed"
        ] = seed


        block[
            "timepoint"
        ] = timepoint


        block[
            "repeat"
        ] = repeat


        block[
            "n_cells"
        ] = len(
            cell_indices
        )


        block[
            "n_seed_positive"
        ] = n_seed_positive


        block[
            "seed_positive_fraction"
        ] = (
            n_seed_positive
            /
            len(
                cell_indices
            )
        )


        result_blocks.append(
            block
        )


    return pd.concat(
        result_blocks,
        ignore_index=True,
    )


# ============================================================================
# 7. RUN EVERY SEED × TIMEPOINT
# ============================================================================

all_combined_validation = []


for seed in SEEDS:

    for timepoint in TIMEPOINTS:

        print()
        print(
            "=" * 78
        )

        print(
            f"{seed} — {timepoint} h"
        )

        print(
            "=" * 78
        )


        # --------------------------------------------------------------------
        # Use EXACTLY the genes from the original correlation analysis.
        # --------------------------------------------------------------------

        correlation_table = (
            all_results[
                (
                    seed,
                    timepoint,
                )
            ]
            .copy()
        )


        genes = (
            correlation_table[
                "gene"
            ]
            .astype(str)
            .tolist()
        )


        print(
            f"Genes to validate: {len(genes):,}"
        )


        repeat_tables = {}


        # --------------------------------------------------------------------
        # Chicago + Michigan independently
        # --------------------------------------------------------------------

        for repeat in REPEATS:

            d = (
                validate_all_genes_one_repeat(
                    seed=seed,
                    timepoint=timepoint,
                    repeat=repeat,
                    genes=genes,
                )
            )


            repeat_tables[
                repeat
            ] = d


        # ====================================================================
        # 8. COLLAPSE EACH REPEAT TO IMPORTANT VALIDATION METRICS
        # ====================================================================

        keep_metrics = [
            "gene",

            "n_cells",
            "n_seed_positive",
            "seed_positive_fraction",

            "Off_fraction_positive",
            "Q1_fraction_positive",
            "Q2_fraction_positive",
            "Q3_fraction_positive",
            "Q4_fraction_positive",

            "Off_n_target_positive",
            "Q4_n_target_positive",

            "Off_mean_expression_positive",
            "Q4_mean_expression_positive",

            "Off_mean_depth_adjusted_expression_positive",
            "Q4_mean_depth_adjusted_expression_positive",

            "delta_detection_Q4_minus_Off",

            "delta_expression_positive_Q4_minus_Off",

            "delta_depth_adjusted_expression_positive_Q4_minus_Off",
        ]


        chicago = (
            repeat_tables[
                "Chicago"
            ][
                keep_metrics
            ]
            .copy()
        )


        michigan = (
            repeat_tables[
                "Michigan"
            ][
                keep_metrics
            ]
            .copy()
        )


        chicago = chicago.rename(
            columns={
                c:
                    (
                        c
                        if c == "gene"
                        else f"Chicago_{c}"
                    )
                for c in chicago.columns
            }
        )


        michigan = michigan.rename(
            columns={
                c:
                    (
                        c
                        if c == "gene"
                        else f"Michigan_{c}"
                    )
                for c in michigan.columns
            }
        )


        validation = (
            chicago
            .merge(
                michigan,
                on="gene",
                how="inner",
            )
        )


        # ====================================================================
        # 9. ADD ORIGINAL CORRELATION RESULTS
        # ====================================================================

        corr = (
            correlation_table[
                [
                    "gene",
                    "Chicago",
                    "Michigan",
                    "mean_correlation",
                    "weaker_abs_correlation",
                    "same_direction",
                ]
            ]
            .copy()
            .rename(
                columns={
                    "Chicago":
                        "Chicago_r",

                    "Michigan":
                        "Michigan_r",

                    "same_direction":
                        "correlation_same_direction",
                }
            )
        )


        validation = (
            corr.merge(
                validation,
                on="gene",
                how="inner",
            )
        )


        validation[
            "seed"
        ] = seed


        validation[
            "timepoint"
        ] = timepoint


        # ====================================================================
        # 10. EQUAL-WEIGHT CONSENSUS CORRELATION
        #
        # Fisher-z mean:
        # Chicago and Michigan contribute equally regardless of cell count.
        # ====================================================================

        r_chi = (
            validation[
                "Chicago_r"
            ]
            .clip(
                -0.999999,
                0.999999,
            )
        )


        r_mi = (
            validation[
                "Michigan_r"
            ]
            .clip(
                -0.999999,
                0.999999,
            )
        )


        validation[
            "consensus_r"
        ] = np.tanh(
            (
                np.arctanh(
                    r_chi
                )
                +
                np.arctanh(
                    r_mi
                )
            )
            / 2
        )


        # ====================================================================
        # 11. DETECTION EFFECT REPRODUCIBILITY
        # ====================================================================

        chi_det = (
            validation[
                "Chicago_delta_detection_Q4_minus_Off"
            ]
        )


        mi_det = (
            validation[
                "Michigan_delta_detection_Q4_minus_Off"
            ]
        )


        validation[
            "detection_effect_same_direction"
        ] = (
            np.sign(
                chi_det
            )
            ==
            np.sign(
                mi_det
            )
        )


        validation[
            "mean_delta_detection"
        ] = (
            chi_det
            +
            mi_det
        ) / 2


        # ====================================================================
        # 12. POSITIVE-CELL ABUNDANCE EFFECT REPRODUCIBILITY
        #
        # Use DEPTH-ADJUSTED abundance for the classification.
        # ====================================================================

        chi_abund = (
            validation[
                "Chicago_delta_depth_adjusted_expression_positive_Q4_minus_Off"
            ]
        )


        mi_abund = (
            validation[
                "Michigan_delta_depth_adjusted_expression_positive_Q4_minus_Off"
            ]
        )


        abundance_finite = (
            chi_abund.notna()
            &
            mi_abund.notna()
        )


        validation[
            "abundance_effect_same_direction"
        ] = (
            abundance_finite
            &
            (
                np.sign(
                    chi_abund
                )
                ==
                np.sign(
                    mi_abund
                )
            )
        )


        validation[
            "mean_delta_abundance"
        ] = (
            chi_abund
            +
            mi_abund
        ) / 2


        # ====================================================================
        # 13. DOES EACH COMPONENT AGREE WITH THE ORIGINAL CORRELATION?
        # ====================================================================

        validation[
            "detection_matches_correlation_both_repeats"
        ] = (
            (
                np.sign(
                    validation[
                        "Chicago_r"
                    ]
                )
                ==
                np.sign(
                    chi_det
                )
            )
            &
            (
                np.sign(
                    validation[
                        "Michigan_r"
                    ]
                )
                ==
                np.sign(
                    mi_det
                )
            )
        )


        validation[
            "abundance_matches_correlation_both_repeats"
        ] = (
            abundance_finite
            &
            (
                np.sign(
                    validation[
                        "Chicago_r"
                    ]
                )
                ==
                np.sign(
                    chi_abund
                )
            )
            &
            (
                np.sign(
                    validation[
                        "Michigan_r"
                    ]
                )
                ==
                np.sign(
                    mi_abund
                )
            )
        )


        # ====================================================================
        # 14. SIMPLE INTERPRETIVE CLASSIFICATION
        #
        # This is descriptive, NOT a statistical significance test.
        # ====================================================================

        def classify_support(
            row,
        ):

            if not row[
                "correlation_same_direction"
            ]:

                return (
                    "correlation not reproduced"
                )


            detection_support = bool(
                row[
                    "detection_matches_correlation_both_repeats"
                ]
            )


            abundance_support = bool(
                row[
                    "abundance_matches_correlation_both_repeats"
                ]
            )


            if (
                detection_support
                and abundance_support
            ):

                return (
                    "detection + abundance"
                )


            if detection_support:

                return (
                    "mostly detection"
                )


            if abundance_support:

                return (
                    "mostly abundance"
                )


            return (
                "mixed / weak decomposition"
            )


        validation[
            "support_type"
        ] = (
            validation.apply(
                classify_support,
                axis=1,
            )
        )


        all_combined_validation.append(
            validation
        )


        # ====================================================================
        # 15. PRINT QUICK SUMMARY
        # ====================================================================

        reproduced = validation[
            validation[
                "correlation_same_direction"
            ]
        ]


        print()
        print(
            f"Reproducible correlations: "
            f"{len(reproduced):,}"
        )


        print()
        print(
            "Support types:"
        )


        print(
            reproduced[
                "support_type"
            ]
            .value_counts()
            .to_string()
        )


        # --------------------------------------------------------------------
        # Top positive
        # --------------------------------------------------------------------

        top_pos = (
            reproduced[
                reproduced[
                    "consensus_r"
                ] > 0
            ]
            .nlargest(
                10,
                "weaker_abs_correlation",
            )
        )


        print()
        print(
            f"TOP POSITIVE {seed} — {timepoint} h"
        )


        print(
            top_pos[
                [
                    "gene",
                    "Chicago_r",
                    "Michigan_r",
                    "consensus_r",
                    "mean_delta_detection",
                    "mean_delta_abundance",
                    "support_type",
                ]
            ]
            .round(
                3
            )
            .to_string(
                index=False
            )
        )


        # --------------------------------------------------------------------
        # Top negative
        # --------------------------------------------------------------------

        top_neg = (
            reproduced[
                reproduced[
                    "consensus_r"
                ] < 0
            ]
            .nlargest(
                10,
                "weaker_abs_correlation",
            )
        )


        print()
        print(
            f"TOP NEGATIVE {seed} — {timepoint} h"
        )


        print(
            top_neg[
                [
                    "gene",
                    "Chicago_r",
                    "Michigan_r",
                    "consensus_r",
                    "mean_delta_detection",
                    "mean_delta_abundance",
                    "support_type",
                ]
            ]
            .round(
                3
            )
            .to_string(
                index=False
            )
        )


# ============================================================================
# 16. SAVE ONE MASTER TABLE
# ============================================================================

validation_all = (
    pd.concat(
        all_combined_validation,
        ignore_index=True,
    )
)


master_output = (
    VALID_OUT
    / "APLNR_APELA_genomewide_validation_ALL.csv"
)


validation_all.to_csv(
    master_output,
    index=False,
)


# ============================================================================
# 17. SAVE ONE SMALLER "TOP HITS" TABLE
#
# top 25 positive + top 25 negative
# for each seed/timepoint
# ============================================================================

top_blocks = []


for seed in SEEDS:

    for timepoint in TIMEPOINTS:

        d = validation_all[
            (
                validation_all[
                    "seed"
                ]
                == seed
            )
            &
            (
                validation_all[
                    "timepoint"
                ]
                == timepoint
            )
            &
            (
                validation_all[
                    "correlation_same_direction"
                ]
            )
        ].copy()


        positive = (
            d[
                d[
                    "consensus_r"
                ]
                > 0
            ]
            .nlargest(
                25,
                "weaker_abs_correlation",
            )
        )


        positive[
            "correlation_direction"
        ] = "positive"


        negative = (
            d[
                d[
                    "consensus_r"
                ]
                < 0
            ]
            .nlargest(
                25,
                "weaker_abs_correlation",
            )
        )


        negative[
            "correlation_direction"
        ] = "negative"


        top_blocks.extend(
            [
                positive,
                negative,
            ]
        )


top_validation = pd.concat(
    top_blocks,
    ignore_index=True,
)


top_output = (
    VALID_OUT
    / "APLNR_APELA_genomewide_validation_TOP.csv"
)


top_validation.to_csv(
    top_output,
    index=False,
)


# ============================================================================
# 18. DONE
# ============================================================================

print()
print(
    "=" * 78
)

print(
    "GENOME-WIDE VALIDATION COMPLETE"
)

print(
    "=" * 78
)

print()
print(
    "Saved ONLY these two files:"
)

print(
    master_output.resolve()
)

print(
    top_output.resolve()
)

## Methods

Cells restricted to the set used by Stan et al. (per-sample library-size gating). Expression normalised to 1e4 and log1p. Identities transferred from the IH gastruloid reference by scANVI; eNasM and lNasM merged (TBX6/MESP2 do not separate them at 42-72h); aPM-LC/LPM-LC/CS-LC grouped as "other" (predominantly 96h states, beyond the sampling window). Embryo identities are Tyser et al.'s own annotations. APLNR/APELA imputation: MAGIC (knn=5, t=1) fit per site, used for visualisation only (panel E); dot plots show unimputed values. n=2 differentiations (Michigan, Chicago); site confounded with batch. Timepoints 42h and 72h.